# RSNA Knee — Complete Offline Inference v2

**One notebook, existing pretrained weights, image-only inference, one final `submission.csv`.**

This replaces the seven-stage RSNA966 research workflow with an executable pretrained inference graph. It does not need that kit, its labels, a feature factory, or newly trained head bundles. All code is embedded below; the competition and published pretrained model assets still need to be attached.

**Target: 0.966. This notebook has no measured Kaggle score.** It is derived from the available Speedy Raptors / DINO / A5 / Rad / CoAt / Global96 source family, not a recovered copy of your reported 0.956 anchor. Keep that scored notebook unchanged.

The only default ranking experiment is **probability pooling of the existing Residual-CoAt and D4 family means**, followed by the same Global96 and outer fusion. Acquisition-repair, coherent-only, extra-gap, and sampling-mass experiments are all disabled. A full same-run rank-family reference is also saved. No model is trained or silently substituted.


## 1. Kaggle setup — before Run All

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 ×2** — the inherited pinned runtimes require two actual T4 GPUs |
| Internet | **Off** |
| Visibility | **Private** |
| Language | Python; use Kaggle's GPU image with its installed PyTorch/CUDA stack |
| Output to submit | `/kaggle/working/submission.csv` |
| Internal deadline | 8.5 hours including a 600-second finalization reserve; runtime not benchmarked here |
| New training prerequisites | **None** |

The 18 frozen native input references are embedded in this notebook's metadata. Use the companion `kernel-metadata.json` to populate the named sources through the Kaggle CLI, or retain the inputs of the original published source before importing this notebook. **A downloaded notebook cannot attach resources to an account by itself. Check the Inputs panel.** Named CLI sources may resolve to newer versions; preserve the frozen native input versions if a checksum fails rather than disabling the check.

### Required existing sources

- `rsna-knee-abnormality-detection`
- `metaresearch/dinov2/pytorch/small/1`
- `antoinegg1/rsna-knee-e11-diverse-heads-v20`
- `antoinegg1/rsna-knee-e9-radimagenet-heads-v15`
- `dreaddevelopment/raptor-knee-maxspan`
- `dreaddevelopment/raptor-knee-native384`
- `dreaddevelopment/raptor-knee-native384dense`
- `marwanmath/resnet-50-radimagenet-marwan`
- `mattiaangeli/knee-mri-fold-weights`
- `mattiaangeli/opencv-python-headless-4120088-x86`
- `mattiaangeli/rsna-knee-coat-resgated-ep10-top3`
- `mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2`
- `mattiaangeli/rsna-knee-coatnet-global96-top3`
- `pilkwang/rsna-knee-llm-labels`
- `pilkwang/rsna-knee-weights`
- `prvsiyan/rsna-knee-v52-radimagenet-heads-20260812`
- `sofiaanjenje/rsna-knee-e11-train`
- `sofiaanjenje/rsna-knee-e13-train`

The two `sofiaanjenje` notebook outputs are already-published model artifacts—not new jobs from the previous kit. No API credentials or private reports are embedded. If a published input is inaccessible to your account, execution stops with the missing asset; weights are not downloadable from this notebook when internet is disabled.


In [ ]:
# Choose ONE recipe before executing any model. No selection from test labels/LB.
SINGLE_CONFIG = {
    "recipe": "probability_pool",    # or "rank_reference"; neither has a measured score here
    "decoded_cache_mib": 128,        # retained raw/rescaled DINO pixels only
}
if SINGLE_CONFIG["recipe"] not in {"probability_pool", "rank_reference"}:
    raise ValueError("Unsupported recipe")
if type(SINGLE_CONFIG["decoded_cache_mib"]) is not int or not 0 <= SINGLE_CONFIG["decoded_cache_mib"] <= 512:
    raise ValueError("decoded_cache_mib must be an integer in [0,512]")


In [ ]:
# Archive earlier named submission files before the inherited clean-start logic.
import time as _single_time, os as _single_os
from pathlib import Path as _SinglePath
_single_work = _SinglePath('/kaggle/working'); _single_work.mkdir(parents=True, exist_ok=True)
_previous = [p for p in _single_work.glob('submission*.csv') if p.is_file()]
_previous += [p for p in _single_work.glob('submission*.receipt.json') if p.is_file()]
if _previous:
    _archive = _single_work / 'previous_runs' / str(_single_time.time_ns())
    _archive.mkdir(parents=True)
    for _p in _previous: _single_os.replace(_p, _archive / _p.name)
    print('Preserved earlier output files in', _archive)


In [ ]:
# Frozen source model controls. Extra acquisition/window experiments are disabled.
import os, time, json, hashlib
from pathlib import Path
MASTER_STARTED_AT = time.time()
MASTER = {
    'sampling_mass_enabled': False,    # disabled in this build
    'gap_view_alpha': 0.0,           # disabled in this build
    'gap_max_windows': 0,             # disabled in this build
    'gap_max_fraction': 0.0,           # disabled in this build
    'gap_attempt_multiplier': 3,        # bounded geometry/decoder attempts
    'gap_spacing_ratio': 1.50,          # neighbor spacing consistency gate
    'acquisition_alpha': 0.0,          # disabled in this build
    'acquisition_max_fraction': 0.0,   # disabled in this build
    'acquisition_max_studies': 0,     # disabled in this build
    'acquisition_max_slots': 2,         # preserve the other original context slots
    'allow_secondary_acquisition': False, # only missing slots / annotation disagreements
    'exclude_duplicate_context': True,
    'minimum_alternate_slices': 8,
    'minimum_relative_slices': 0.60,
    'global96_enabled': True,          # requires the extra Global96 asset
    'global96_family_share': 0.25,      # experiment; NOT a measured optimal weight
    'time_budget_hours': 8.5,
    'publication_reserve_seconds': 600,
    'decoder_probe_series': 64,
    'series_coherence_alpha': 0.0,     # disabled in this build
    'series_min_windows': 12,          # bypass when too little same-acquisition context remains
    'series_min_fraction': 0.50,
    'raptor_micro_images': 8,          # memory failures retry 4/2/1 without dropping windows
}
MASTER_SOURCE_HASHES = {'rsna-knee-fast-2xt4-inference.ipynb': 'df759fce119004e575202a4010d87bd9b2dba44de47f4a070cedb7ea012a5fef', 'bend-the-knee-to-speedy-raptors.ipynb': '366d6cbb21e7497c5ae5154a04edcaa84c9d3f9247918a51bef4566619030cf6', 'rsna-knee-0942-restructured.ipynb': '8e5b9d517b347fa0808b0f27bd12c10ad477e3ad92803069fcdcf16863e9f306', 'rsna_knee_master_plan/MASTER_PLAN.md': 'ae5b4e4dc566136c5a36f6ad62c4f1bc9858d21cc9367bcbf9f2dc966ad1e3ae'}
MASTER_SOURCE_HASHES.update({'rsna-knee-mast-master.ipynb': 'dccf19520cab2ed01aa7d0e67e5ce9f3f5506633281bf26261e63dbad901ef6c', 'bend-the-knee-to-speedy-raptors-the-original.ipynb': '59ffd587e3ac7fbcb48050a1d4a3dc7e6a2a40acef5fad65a6f046dee4ee7418'})
MASTER_FATAL_EVENTS = {
    'dino_member_nonfinite_neutral', 'dino_member_public_nonfinite_neutral',
    'a5_nonfinite_neutral', 'a5_study_failed', 'rad_nonfinite_zeroed',
    'rad_head_nonfinite_neutral', 'rad_invalid_values_repaired',
    'raptor_study_failed', 'raptor_neutral_fill',
    'coat_resgated_child_failed', 'coat_d4_child_failed',
    'coat_resgated_output_rejected', 'coat_d4_output_rejected',
    'coat_family_unavailable', 'coat_family_partial', 'coat_resgated_fallback_studies',
    'submission_neutral_fill',
}
if not 0 <= MASTER['acquisition_alpha'] <= .30:
    raise ValueError('acquisition_alpha must be between 0 and .30')
if not 0 <= MASTER['acquisition_max_fraction'] <= .30:
    raise ValueError('Use a bounded additional-acquisition cohort: fraction <= .30')
if not isinstance(MASTER['acquisition_max_studies'], int) or MASTER['acquisition_max_studies'] < 0:
    raise ValueError('acquisition_max_studies must be a nonnegative integer')
for _key, _value in {
    'HF_HUB_OFFLINE':'1', 'TRANSFORMERS_OFFLINE':'1', 'HF_HUB_DISABLE_TELEMETRY':'1',
    'PIP_NO_INDEX':'1', 'PIP_DISABLE_PIP_VERSION_CHECK':'1',
    'OMP_NUM_THREADS':'4', 'OPENBLAS_NUM_THREADS':'4', 'MKL_NUM_THREADS':'4',
    'RSNA_CACHE_RAM_GIB':'0.5', 'RSNA_PIXEL_SCRATCH':'/kaggle/temp/rsna_mast_pixels',
    'RSNA_PARALLEL_COAT_READERS':'0', # each child can use both GPUs; avoid overlapping residency
}.items():
    if _key in {'HF_HUB_OFFLINE','TRANSFORMERS_OFFLINE','HF_HUB_DISABLE_TELEMETRY',
                 'PIP_NO_INDEX','PIP_DISABLE_PIP_VERSION_CHECK','RSNA_PARALLEL_COAT_READERS'}:
        os.environ[_key] = _value
    else:
        os.environ.setdefault(_key, _value)
# Never spoof GPU identity to bypass a checkpoint/runtime contract.
if os.environ.get('RSNA_LOCAL_ALLOW_V100'):
    raise RuntimeError('Remove RSNA_LOCAL_ALLOW_V100; use an actual T4 x2 Kaggle runtime')
_work = Path('/kaggle/working')
_work.mkdir(parents=True, exist_ok=True)
for _stale in ('submission.csv', 'submission.receipt.json', 'submission_anchor.csv',
               'submission_anchor.receipt.json', 'mast_complete.json', 'mast_v2_complete.json', 'mast_v3_complete.json', 'mast_v4_complete.json', 'mast_v5_complete.json'):
    (_work/_stale).unlink(missing_ok=True)
print('Pretrained inference settings:', json.dumps(MASTER, sort_keys=True))
print('No training or leaderboard score is claimed. Required model failures are fatal.')

MASTER_SOURCE_HASHES['rsna-knee-mast-v2-global96.ipynb'] = '39724bc5645422997a854296c8f95eb8982a2f94b0d54a7110b0728d314a01ea'

MASTER_SOURCE_HASHES['rsna-knee-mast-v3-series-coherent.ipynb'] = '7ca902a11a87747692b64e47b3da3ce66c82903417b0356c11417dd72820b044'

MASTER_SOURCE_HASHES['rsna-knee-mast-v4-gap-coverage.ipynb'] = '2292167696c679876bc0adff0f591384fba7efe20dcf8363c5ad7974a9552dae'


In [ ]:
# Embedded support code: no external code dataset is required.
import sys as _single_sys, importlib.util as _single_importlib
from pathlib import Path as _SinglePath
_SINGLE_SUPPORT_SOURCE = '"""Offline inference support. Contains no trained weights or invented predictions.\n\nThe family pooling experiment preserves the source model outputs; its AUC gain\nis unmeasured. Caches contain only actual decoded arrays from this execution.\n"""\nfrom __future__ import annotations\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nimport re\nimport threading\nfrom collections import OrderedDict\nfrom concurrent.futures import Future\nfrom pathlib import Path\nfrom typing import Callable, Iterable\nimport numpy as np\nimport pandas as pd\n\nUID = \'StudyInstanceUID\'\nLABELS = [\'ACL\',\'MCL\',\'Medial Meniscus\',\'Lateral Meniscus\',\'Medial OA\',\n          \'Lateral OA\',\'PF OA\',\'Effusion\',\'Synovitis\',"Baker\'s",\'Contusion\',\'Fracture\']\nOUTER = np.array([.75,.60,.80,1.,.60,.75,.60,.60,.60,.60,.60,.75],np.float64)\n\ndef sha256(path: str | Path) -> str:\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda:f.read(8<<20),b\'\'):h.update(block)\n    return h.hexdigest()\n\ndef checked_ids(ids: Iterable[str]) -> list[str]:\n    ids=list(ids)\n    if not ids or any(not isinstance(i,str) or not i.strip() for i in ids):\n        raise ValueError(\'Study IDs must be nonempty strings\')\n    if len(ids)!=len(set(ids)):raise ValueError(\'Duplicate StudyInstanceUID\')\n    return ids\n\ndef probabilities(x, name=\'predictions\') -> np.ndarray:\n    a=np.asarray(x)\n    if a.dtype.kind!=\'f\' or not a.size or not np.isfinite(a).all():\n        raise ValueError(name+\': expected nonempty finite floating-point array\')\n    if np.any((a<0)|(a>1)):raise ValueError(name+\': outside [0,1]\')\n    return a\n\ndef align(frame: pd.DataFrame, ids: Iterable[str]) -> pd.DataFrame:\n    ids=checked_ids(ids)\n    if list(frame.columns)!=[UID,*LABELS]:raise ValueError(\'Target schema/order mismatch\')\n    found=checked_ids(frame[UID].tolist())\n    if set(found)!=set(ids):raise ValueError(\'Missing/extra study IDs\')\n    result=frame.set_index(UID).loc[ids].reset_index()\n    probabilities(result[LABELS].to_numpy(), \'aligned frame\')\n    return result\n\ndef read_csv(path: str | Path) -> pd.DataFrame:\n    with Path(path).open(encoding=\'utf-8-sig\',newline=\'\') as f:\n        header=next(csv.reader(f),[])\n    if not header or len(header)!=len(set(header)):raise ValueError(\'Empty/duplicate CSV header\')\n    return pd.read_csv(path,dtype={UID:str},float_precision=\'round_trip\')\n\ndef rankpct(a) -> np.ndarray:\n    a=probabilities(a)\n    if a.ndim!=2 or a.shape[1]!=12:raise ValueError(\'Expected N x 12\')\n    return pd.DataFrame(a).rank(method=\'average\',pct=True).to_numpy(np.float64)\n\ndef pooled_family(residual,d4) -> np.ndarray:\n    a=probabilities(residual,\'residual\');b=probabilities(d4,\'D4\')\n    if a.shape!=b.shape:raise ValueError(\'Family shape mismatch\')\n    # Cast before addition: preserve the saved float32 means exactly.\n    return rankpct((a.astype(np.float64)+b.astype(np.float64))/2.0)\n\ndef fuse(transformer, raptor, family, global96, share=.25) -> np.ndarray:\n    arrays=[probabilities(a) for a in (transformer,raptor,family)]\n    if len({a.shape for a in arrays})!=1 or arrays[0].ndim!=2 or arrays[0].shape[1]!=12:\n        raise ValueError(\'Fusion expects same-cohort N x 12 matrices\')\n    if isinstance(share,bool) or not np.isfinite(share) or not 0<=share<=.5:\n        raise ValueError(\'Invalid Global96 family share\')\n    f=arrays[2].astype(np.float64)\n    if share:\n        g=probabilities(global96,\'Global96\')\n        if g.shape!=f.shape:raise ValueError(\'Global96 shape mismatch\')\n        f=rankpct((1.-share)*f+share*rankpct(g))\n    hybrid=(1.-.4)*rankpct(arrays[1])+.4*f\n    # IMPORTANT: do not rank hybrid at this junction.\n    return rankpct((1.-OUTER[None,:])*rankpct(arrays[0])+OUTER[None,:]*hybrid)\n\ndef load_probability_mean(raw_path, companion_csv, ids, expected_members):\n    """Read exact source archives, checking membership, cohort and mean consistency.\n\n    Rank CSVs alone are never treated as raw probabilities. The residual release\n    may use another checkpoint-reduction convention for its ranked CSV, so its\n    CSV is an ID/schema check rather than a claimed rank(mean) equality check.\n    """\n    ids=checked_ids(ids); raw_path=Path(raw_path);companion_csv=Path(companion_csv)\n    if type(expected_members)!=int or expected_members<1:raise ValueError(\'Invalid member count\')\n    before={\'archive_sha256\':sha256(raw_path),\'companion_sha256\':sha256(companion_csv)}\n    align(read_csv(companion_csv),ids)\n    with np.load(raw_path,allow_pickle=False) as z:\n        if \'study_uids\' not in z.files:raise ValueError(\'Probability archive lacks explicit study_uids\')\n        source_ids=checked_ids(z[\'study_uids\'].astype(str).tolist())\n        if set(source_ids)!=set(ids):raise ValueError(\'Probability archive belongs to another cohort\')\n        if \'labels\' in z.files and z[\'labels\'].astype(str).tolist()!=LABELS:\n            raise ValueError(\'Probability archive label-order mismatch\')\n        if \'raw_probabilities\' not in z.files:raise ValueError(\'Missing member probability array\')\n        raw=probabilities(z[\'raw_probabilities\'],\'raw members\')\n        if raw.shape!=(expected_members,len(ids),12):raise ValueError(\'Wrong raw member axis/count\')\n        mean64=raw.astype(np.float64).mean(0)\n        if \'probability_mean\' in z.files:\n            saved=probabilities(z[\'probability_mean\'],\'saved mean\')\n            if saved.shape!=(len(ids),12):raise ValueError(\'Saved mean shape mismatch\')\n            native=raw.mean(0)\n            eps=4*np.finfo(raw.dtype).eps\n            if not (np.allclose(saved,mean64,rtol=0,atol=eps) or\n                    np.allclose(saved,native,rtol=0,atol=eps)):\n                raise ValueError(\'Saved mean disagrees with member probabilities\')\n            values=saved.astype(np.float64)\n            reduction=\'saved_probability_mean_promoted_to_float64\'\n        else:\n            values=mean64;reduction=\'float64_member_mean\'\n    order={u:i for i,u in enumerate(source_ids)}\n    values=values[[order[u] for u in ids]]\n    if sha256(raw_path)!=before[\'archive_sha256\'] or sha256(companion_csv)!=before[\'companion_sha256\']:\n        raise RuntimeError(\'Prediction artifact changed during read\')\n    return values,{**before,\'members\':expected_members,\'studies\':len(ids),\'reduction\':reduction}\n\nclass BoundedArrayCache:\n    """Thread-safe, single-flight LRU; max_bytes bounds retained arrays only.\n\n    Returned arrays are immutable. No failures/partial arrays are cached.\n    In-flight decoders and callers can transiently use additional memory.\n    """\n    def __init__(self,max_bytes: int=128<<20):\n        if type(max_bytes)!=int or max_bytes<0:raise ValueError(\'Invalid cache budget\')\n        self.max_bytes=max_bytes;self._bytes=0;self._items=OrderedDict()\n        self._pending={};self._lock=threading.RLock()\n        self.hits=0;self.misses=0;self.waits=0;self.evictions=0\n    def get(self,key,loader: Callable[[],np.ndarray]):\n        with self._lock:\n            if key in self._items:\n                self.hits+=1;self._items.move_to_end(key);return self._items[key]\n            if key in self._pending:\n                future=self._pending[key];owner=False;self.waits+=1\n            else:\n                future=Future();self._pending[key]=future;owner=True;self.misses+=1\n        if not owner:return future.result()\n        try:\n            # Own the cache storage so the caller cannot mutate the loader\'s base.\n            value=np.array(loader(),copy=True,order=\'C\')\n            if value.dtype!=np.float32 or value.ndim!=2 or not value.size or not np.isfinite(value).all():\n                raise ValueError(\'Decoded cache value must be finite nonempty float32 2-D pixels\')\n            value.setflags(write=False)\n            with self._lock:\n                if value.nbytes<=self.max_bytes:\n                    while self._items and self._bytes+value.nbytes>self.max_bytes:\n                        _,old=self._items.popitem(last=False);self._bytes-=old.nbytes;self.evictions+=1\n                    self._items[key]=value;self._bytes+=value.nbytes\n                future.set_result(value);self._pending.pop(key,None)\n            return value\n        except BaseException as exc:\n            with self._lock:\n                future.set_exception(exc);self._pending.pop(key,None)\n            raise\n    def stats(self):\n        with self._lock:return {\'retained_bytes\':self._bytes,\'max_bytes\':self.max_bytes,\n            \'entries\':len(self._items),\'hits\':self.hits,\'misses\':self.misses,\n            \'waits\':self.waits,\'evictions\':self.evictions,\'pending\':len(self._pending)}\n    def clear(self):\n        with self._lock:\n            if self._pending:raise RuntimeError(\'Cannot clear cache with in-flight decoders\')\n            self._items.clear();self._bytes=0\n\ndef pixel_key(path):\n    p=Path(path).resolve();s=p.stat()\n    return (str(p),s.st_size,s.st_mtime_ns,s.st_ino)\n\nclass AssetResolver:\n    """Index known artifact directories once, never the competition image tree."""\n    def __init__(self,roots,max_depth=6):\n        self.roots=[Path(r) for r in roots];self.max_depth=max_depth\n        self._files=None;self._hashes={};self._hits={};self._lock=threading.RLock()\n    def catalogue(self):\n        with self._lock:\n            if self._files is None:\n                files={}\n                for root in self.roots:\n                    if not root.is_dir():continue\n                    for directory,dirs,names in os.walk(root):\n                        depth=len(Path(directory).relative_to(root).parts)\n                        dirs[:]=sorted(d for d in dirs if d not in {\n                            \'train_series\',\'test_series\',\'train_images\',\'test_images\',\'.git\',\'__pycache__\'})\n                        if depth>=self.max_depth:dirs[:]=[]\n                        for name in sorted(names):\n                            p=(Path(directory)/name).resolve()\n                            if p.is_file():files[str(p)]=p\n                self._files=tuple(files[k] for k in sorted(files))\n            return self._files\n    def digest(self,path):\n        key=pixel_key(path)\n        with self._lock:\n            if key not in self._hashes:self._hashes[key]=sha256(path)\n            return self._hashes[key]\n    def find(self,pattern,digest=None):\n        import fnmatch\n        with self._lock:\n            key=(pattern,digest)\n            if key in self._hits:return self._hits[key]\n            hits=[p for p in self.catalogue() if fnmatch.fnmatchcase(p.name,pattern)]\n            if digest:hits=[p for p in hits if self.digest(p)==digest]\n            if not hits:raise FileNotFoundError(f\'Required pretrained asset missing/changed: {pattern}; expected SHA256={digest}\')\n            if len(hits)>1:\n                hashes={self.digest(p) for p in hits}\n                if len(hashes)!=1:\n                    raise RuntimeError(f\'Ambiguous different assets for {pattern}: \'+str([str(p) for p in hits]))\n            # Identical pinned wheel/model copies are valid; choose deterministically.\n            selected=sorted(hits,key=lambda p:(len(p.parts),str(p)))[0]\n            self._hits[key]=selected;return selected\n\ndef check_test_tables(root):\n    root=Path(root)\n    test=read_csv(root/\'test.csv\')\n    if UID not in test:raise ValueError(\'test.csv lacks StudyInstanceUID\')\n    ids=checked_ids(test[UID].tolist())\n    series=pd.read_csv(root/\'test_series.csv\',dtype={UID:str,\'SeriesInstanceUID\':str})\n    required={UID,\'SeriesInstanceUID\',\'Anatomical_Plane\',\'Fat_Suppression\',\'Fluid_Sensitive\'}\n    if not required.issubset(series):raise ValueError(\'Missing test_series fields: \'+str(sorted(required-set(series))))\n    if series[[UID,\'SeriesInstanceUID\']].isna().any().any():raise ValueError(\'Missing series IDs\')\n    if series.duplicated([UID,\'SeriesInstanceUID\']).any():raise ValueError(\'Duplicate series rows\')\n    if set(series[UID])!=set(ids):raise ValueError(\'Test study/series coverage mismatch\')\n    for value in [*ids,*series[\'SeriesInstanceUID\'].tolist()]:\n        if \'/\' in value or \'\\\\\' in value or value in {\'.\',\'..\'}:raise ValueError(\'Invalid path component\')\n    sample=root/\'sample_submission.csv\'\n    if sample.is_file():\n        if list(read_csv(sample).columns)!=[UID,*LABELS]:raise ValueError(\'Submission target schema changed\')\n        # The sample may contain preview IDs; never use it as the scoring cohort.\n    return {\'studies\':len(ids),\'series\':len(series),\'labels\':LABELS,\n        \'test_csv_sha256\':sha256(root/\'test.csv\'),\'series_csv_sha256\':sha256(root/\'test_series.csv\'),\n        \'fluid_fs_equal_rows\':int(series[\'Fluid_Sensitive\'].eq(series[\'Fat_Suppression\']).sum())}\n\ndef environment_report():\n    import importlib.metadata as m,sys\n    names=[\'numpy\',\'pandas\',\'torch\',\'torchvision\',\'pydicom\',\'timm\',\'transformers\',\n           \'opencv-python-headless\',\'pylibjpeg\',\'pylibjpeg-libjpeg\',\'pylibjpeg-openjpeg\']\n    packages={}\n    for name in names:\n        try:packages[name]=m.version(name)\n        except m.PackageNotFoundError:packages[name]=None\n    return {\'python\':sys.version,\'packages\':packages,\'internet_required\':False}\n'
_single_module_dir=_SinglePath('/kaggle/working/_single_runtime')
_single_module_dir.mkdir(parents=True,exist_ok=True)
_single_module_path=_single_module_dir/'single_support.py'
_single_module_path.write_text(_SINGLE_SUPPORT_SOURCE)
_single_spec=_single_importlib.spec_from_file_location('single_support',_single_module_path)
s2=_single_importlib.module_from_spec(_single_spec)
_single_sys.modules['single_support']=s2
_single_spec.loader.exec_module(s2)
_SINGLE_DECODE_CACHE=s2.BoundedArrayCache(SINGLE_CONFIG['decoded_cache_mib']<<20)


## 2. Checked numerical helpers and inherited model contracts

In [ ]:
"""Pure-CPU selection, fusion and publication helpers for RSNA Knee MAST.

MAST = metadata-guided acquisition sampling at test time.
No weights are fitted on test data; alpha is an explicitly unvalidated prior.
"""
from __future__ import annotations
import hashlib
import io
import json
import math
import os
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd

MA_LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
             'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
MA_OUTER = np.asarray([.75, .60, .80, 1., .60, .75, .60, .60, .60, .60, .60, .75])

def ma_flag(value: Any) -> bool | None:
    """An unknown annotation is neither a positive nor a negative."""
    if isinstance(value, str):
        text = value.strip().lower()
        if text in ('true', 'yes', 'y', 't'): return True
        if text in ('false', 'no', 'n', 'f'): return False
    try:
        val = float(value)
    except (TypeError, ValueError):
        return None
    return bool(val) if np.isfinite(val) and val in (0., 1.) else None

def ma_rank(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x)
    if x.ndim != 2 or len(x) == 0 or not np.isfinite(x).all():
        raise ValueError('Ranking requires a finite, nonempty study x finding matrix')
    return pd.DataFrame(x).rank(method='average', pct=True).to_numpy(np.float64)

def ma_check_prob(x: np.ndarray, name: str) -> np.ndarray:
    x = np.asarray(x)
    if not np.issubdtype(x.dtype, np.number) or not np.isfinite(x).all():
        raise ValueError(name + ': nonfinite or nonnumeric prediction')
    if x.size and (x.min() < 0 or x.max() > 1):
        raise ValueError(name + ': value outside [0,1]')
    return x

def ma_frame(frame: pd.DataFrame, ids: list[str]) -> pd.DataFrame:
    ids = list(map(str, ids))
    if not ids or len(set(ids)) != len(ids): raise ValueError('Empty/duplicate expected UID')
    if frame.columns.tolist() != ['StudyInstanceUID', *MA_LABELS]:
        raise ValueError('Incorrect label names/order')
    frame = frame.copy()
    frame['StudyInstanceUID'] = frame['StudyInstanceUID'].astype(str)
    if frame.StudyInstanceUID.duplicated().any() or set(frame.StudyInstanceUID) != set(ids):
        raise ValueError('Missing, extra or duplicate prediction UID')
    frame = frame.set_index('StudyInstanceUID').loc[ids].reset_index()
    ma_check_prob(frame[MA_LABELS].to_numpy(), 'submission')
    return frame

def ma_csv_values(x: np.ndarray) -> np.ndarray:
    """Reproduce the parent's default pandas CSV round trip between stages."""
    return pd.read_csv(io.StringIO(pd.DataFrame(x, columns=MA_LABELS).to_csv(index=False))).to_numpy(np.float64)

def _ma_number(value: Any, fallback: float = 0.) -> float:
    try: ans = float(value)
    except (ValueError, TypeError): return fallback
    return ans if np.isfinite(ans) else fallback

def ma_select_alternates(headers: pd.DataFrame, metadata: pd.DataFrame,
                         anchor: dict, slots: list, ids: list[str], *,
                         max_fraction: float = .30, max_studies: int = 384,
                         max_slots: int = 2, allow_secondary: bool = True,
                         min_slices: int = 8, min_relative_slices: float = .60,
                         exclude_duplicate_context: bool = False) -> tuple[dict, list]:
    """Return study->slot->new record, without ever modifying the anchor.

    Preference: recover an empty slot; repair a CSV/header disagreement; then
    expose one second eligible acquisition. Counts/resolution are quality
    proxies, NOT validated measures of diagnostic adequacy. All ranking is
    metadata-only and deterministic. Unknown required flags cannot qualify.
    """
    if not 0 <= max_fraction <= 1 or max_studies < 0 or max_slots < 1:
        raise ValueError('Invalid acquisition budget')
    if min_slices < 1 or not 0 <= min_relative_slices <= 1:
        raise ValueError('Invalid acquisition eligibility threshold')
    ids = list(map(str, ids))
    if len(ids) != len(set(ids)): raise ValueError('Duplicate study UID')
    required = {'StudyInstanceUID', 'SeriesInstanceUID', 'Anatomical_Plane',
                'Fat_Suppression', 'Fluid_Sensitive'}
    if not required <= set(metadata): raise ValueError('Missing supplied series annotations')
    md = metadata.copy()
    for c in ('StudyInstanceUID', 'SeriesInstanceUID'): md[c] = md[c].astype(str)
    if md.duplicated(['StudyInstanceUID', 'SeriesInstanceUID']).any():
        raise ValueError('Duplicate study/series metadata key')
    hd = headers.copy()
    for c in ('StudyInstanceUID', 'SeriesInstanceUID'): hd[c] = hd[c].astype(str)
    if hd.duplicated(['StudyInstanceUID', 'SeriesInstanceUID']).any():
        raise ValueError('Duplicate study/series header key')
    lookup = {(r.StudyInstanceUID, r.SeriesInstanceUID): r
              for r in md.itertuples(index=False)}
    groups = {str(st): g for st, g in hd.groupby('StudyInstanceUID', sort=False)}
    proposals = []
    audit = []
    for uid in ids:
        if uid not in groups: continue
        chosen = anchor.get(uid, {})
        context_series = {str(rec['SeriesInstanceUID']) for rec in chosen.values() if rec is not None}
        rows = groups[uid]
        candidates_for_study = []
        for slot_name, plane, fluid, fs in slots:
            old = chosen.get(slot_name)
            old_uid = None if old is None else str(old['SeriesInstanceUID'])
            eligible = []
            for _, row in rows.iterrows():
                sid = str(row.SeriesInstanceUID)
                if exclude_duplicate_context and sid != old_uid and sid in context_series:
                    continue  # already represented elsewhere; not additional acquisition evidence
                rec = lookup.get((uid, sid))
                if rec is None or str(rec.Anatomical_Plane) != plane: continue
                if ma_flag(rec.Fat_Suppression) is not bool(fs): continue
                if fluid is not None and ma_flag(rec.Fluid_Sensitive) is not bool(fluid): continue
                ns = int(_ma_number(row.get('n_slices')))
                if ns < min_slices: continue
                # Same longest-acquisition preference as the anchor; deterministic ties.
                px = _ma_number(row.get('px'), 1.e6)
                if px <= 0: px = 1.e6
                eligible.append((row.copy(deep=True), ns, px, sid))
            eligible.sort(key=lambda t: (-t[1], t[2], t[3]))
            if not eligible: continue
            same = any(t[3] == old_uid for t in eligible)
            alternative = next((t for t in eligible if t[3] != old_uid), None)
            if alternative is None: continue
            if old is None:
                reason, priority = 'recover_missing_slot', 3
            elif not same:
                reason, priority = 'csv_header_disagreement', 2
            elif allow_secondary:
                reason, priority = 'second_eligible_acquisition', 1
            else: continue
            row, ns, px, sid = alternative
            old_ns = int(_ma_number(old.get('n_slices'))) if old is not None else 0
            if reason == 'second_eligible_acquisition' and ns < min_relative_slices * max(1, old_ns):
                continue
            info = dict(study_uid=uid, slot=slot_name, old_series_uid=old_uid,
                        new_series_uid=sid, reason=reason, priority=priority,
                        old_slices=old_ns, new_slices=ns,
                        selected=False, exclusion=None)
            candidates_for_study.append((priority, slot_name, row, info))
        if candidates_for_study:
            candidates_for_study.sort(key=lambda t: (-t[0], t[1], str(t[2].SeriesInstanceUID)))
            # Do not reuse the same alternate acquisition in two slots.
            accepted, used = [], set()
            for item in candidates_for_study:
                if len(accepted) >= max_slots or str(item[2].SeriesInstanceUID) in used:
                    item[3]['exclusion'] = 'per_study_slot_budget_or_duplicate'
                else:
                    accepted.append(item); used.add(str(item[2].SeriesInstanceUID))
                audit.append(item[3])
            priority = sum(t[0] for t in accepted)
            tie = hashlib.sha256(('MAST-v1|' + uid).encode()).hexdigest()
            proposals.append((-priority, tie, uid, accepted))
    limit = min(max_studies, int(math.floor(len(ids) * max_fraction + 1.e-12)))
    # On tiny example cohorts, allow one eligible study for an integration check.
    if max_fraction > 0 and max_studies > 0 and len(ids) > 0: limit = max(1, limit)
    proposals.sort(key=lambda t: (t[0], t[1]))
    out = {}
    for i, (_, _, uid, entries) in enumerate(proposals):
        if i >= limit:
            for _, _, _, info in entries: info['exclusion'] = 'cohort_compute_budget'
            continue
        out[uid] = {}
        for _, slot_name, row, info in entries:
            out[uid][slot_name] = row
            info['selected'] = True
    return out, audit

def ma_mix_raw(anchor: np.ndarray, alternate: np.ndarray, indices: np.ndarray,
               alpha: float) -> np.ndarray:
    """Mix probabilities first; rank the COMPLETE cohort later, never the subset."""
    base = ma_check_prob(anchor, 'anchor')
    alt = ma_check_prob(alternate, 'alternate')
    idx = np.asarray(indices, dtype=np.int64)
    if base.ndim != 2 or alt.shape != (len(idx), base.shape[1]):
        raise ValueError('Alternate shape does not match selected rows')
    if len(np.unique(idx)) != len(idx) or np.any(idx < 0) or np.any(idx >= len(base)):
        raise ValueError('Duplicate/out-of-range acquisition row')
    if not 0 <= alpha <= .30: raise ValueError('Unvalidated augmentation alpha must be in [0,.30]')
    out = base.copy()
    if alpha == 0 or not len(idx): return out
    out[idx] = ((1. - alpha) * base[idx].astype(np.float64) + alpha * alt.astype(np.float64)).astype(base.dtype)
    return out

def ma_protocol(metadata: pd.DataFrame, ids: list[str]) -> pd.DataFrame:
    """Exact protocol feature order used by the inherited 88-feature calibrator."""
    md = metadata.copy(); md['StudyInstanceUID'] = md.StudyInstanceUID.astype(str)
    protocol = pd.DataFrame(index=pd.Index(ids, name='StudyInstanceUID'))
    protocol['n_series'] = md.groupby('StudyInstanceUID').size().reindex(ids).fillna(0)
    for plane in ('Sagittal', 'Coronal', 'Axial'):
        part = md[md.Anatomical_Plane.astype(str) == plane]
        protocol[f'n_{plane[:3]}'] = part.groupby('StudyInstanceUID').size().reindex(ids).fillna(0)
    for flag in ('Fat_Suppression', 'Fluid_Sensitive'):
        marked = md[pd.to_numeric(md[flag], errors='coerce').fillna(0) > 0]
        protocol[flag[:3]] = marked.groupby('StudyInstanceUID').size().reindex(ids).fillna(0)
        for plane in ('Sagittal', 'Coronal', 'Axial'):
            part = marked[marked.Anatomical_Plane.astype(str) == plane]
            protocol[f'{flag[:3]}_{plane[:3]}'] = part.groupby('StudyInstanceUID').size().reindex(ids).fillna(0)
    return protocol

def ma_replay_rad(baseline: np.ndarray, e10_raw: np.ndarray, e13_raw: np.ndarray,
                  second_raw: np.ndarray, protocol: pd.DataFrame, cal: dict,
                  *, primary_alpha: float = .55, e13_weight: float = .50,
                  second_alpha: float = .20, calibration_alpha: float = .40) -> np.ndarray:
    """Replay the deployed graph exactly. No coefficients are fitted here."""
    baseline = ma_check_prob(baseline, 'A5 input').astype(np.float64)
    for name, value in [('E10', e10_raw), ('E13', e13_raw), ('second Rad', second_raw)]:
        ma_check_prob(value, name)
        if value.shape != (5, len(baseline), 12): raise ValueError(name + ': five-fold shape required')
    reference = ma_rank((1-e13_weight)*ma_rank(e10_raw.mean(0)) + e13_weight*ma_rank(e13_raw.mean(0)))
    second = ma_rank(second_raw.mean(0))
    parent_rank = ma_rank(baseline)
    candidate = baseline.copy()
    for j, label in enumerate(MA_LABELS):
        if label not in ("Baker's", 'Fracture'):
            candidate[:, j] = (1-primary_alpha)*parent_rank[:, j] + primary_alpha*reference[:, j]
    branch = ma_rank((1-second_alpha)*ma_rank(candidate) + second_alpha*second)
    if list(protocol.columns) != list(cal['protocol_columns']):
        raise ValueError('Calibrator protocol columns changed')
    mean = (parent_rank + reference + second) / 3.
    blocks = [parent_rank, reference, second, reference-parent_rank, second-parent_rank, mean]
    for group in cal['groups']:
        blocks.append(mean[:, [MA_LABELS.index(t) for t in group]].mean(axis=1, keepdims=True))
    blocks.append(protocol.to_numpy(np.float64))
    x = np.concatenate(blocks, axis=1)
    center, spread = np.asarray(cal['mean']), np.asarray(cal['scale'])
    coef, bias = np.asarray(cal['coef']), np.asarray(cal['intercept'])
    if x.shape[1] != coef.shape[1] or not (spread > 0).all():
        raise ValueError('Calibrator feature/scale mismatch')
    adjusted = ma_rank(((x-center)/spread) @ coef.T + bias)
    for j, label in enumerate(MA_LABELS):
        if label in set(cal['gate']):
            branch[:, j] = (1-calibration_alpha)*branch[:, j] + calibration_alpha*adjusted[:, j]
    return ma_rank(branch)

def ma_outer_fuse(transformer: np.ndarray, coat_hybrid: np.ndarray) -> np.ndarray:
    if transformer.shape != coat_hybrid.shape or transformer.shape[1] != 12:
        raise ValueError('Outer-fusion shape mismatch')
    ma_check_prob(coat_hybrid, 'CoAt hybrid')
    return ma_rank((1-MA_OUTER[None,:])*ma_rank(transformer) + MA_OUTER[None,:]*coat_hybrid)

def ma_sha(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(8 << 20), b''): digest.update(block)
    return digest.hexdigest()

def ma_publish(frame: pd.DataFrame, ids: list[str], destination: str | Path,
               receipt: dict) -> dict:
    """Publish once, after every transformation, and hash the actual final bytes."""
    frame = ma_frame(frame, ids)
    path = Path(destination); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name('.' + path.name + '.tmp')
    with temporary.open('w') as f:
        frame.to_csv(f, index=False); f.flush(); os.fsync(f.fileno())
    check = pd.read_csv(temporary, dtype={'StudyInstanceUID':str}, float_precision='round_trip')
    ma_frame(check, ids)
    if not np.array_equal(check[MA_LABELS].to_numpy(), frame[MA_LABELS].to_numpy()):
        raise RuntimeError('Final CSV failed numeric round-trip check')
    digest = ma_sha(temporary)
    os.replace(temporary, path)
    if ma_sha(path) != digest: raise RuntimeError('Final output changed during publication')
    result = {**receipt, 'output':str(path), 'submission_sha256':digest,
              'studies':len(ids), 'labels':MA_LABELS, 'score_measured':None}
    recpath = path.with_suffix('.receipt.json')
    temp = recpath.with_name('.' + recpath.name + '.tmp')
    with temp.open('w') as f:
        json.dump(result, f, indent=2, sort_keys=True, allow_nan=False, default=str)
        f.write('\n'); f.flush(); os.fsync(f.fileno())
    os.replace(temp, recpath)
    return result


In [ ]:
"""MAST v2: checked family fusion, pinned Global96 receipt and process budget.
All new blend coefficients are experiments, not fitted or claimed optimal.
"""
from __future__ import annotations
import os
import signal
import subprocess
import time
from pathlib import Path
import numpy as np
import pandas as pd

V2_GLOBAL96_MANIFEST = 'coatnet_global96_top3_manifest.json'
V2_GLOBAL96_MANIFEST_SHA = '015f09030e76f86274cadae40777b3c3cf5c19f8c835d5ad88e15db41903779e'
V2_TIMM_SHA = '888981753e65cbaacfc07494370138b1700a27b1f0af587f4f9b47bc024161d0'
V2_GLOBAL96_EPOCHS = [16, 23, 18]


def v2_check_settings(settings: dict) -> dict:
    """Validate controls, including booleans and nonfinite numeric values."""
    s = dict(settings)
    bounds = {'acquisition_alpha': (0., .30), 'acquisition_max_fraction': (0., .30),
              'global96_family_share': (0., .50), 'time_budget_hours': (1., 8.5),
              'publication_reserve_seconds': (60., 1800.),
              'minimum_relative_slices': (0., 1.)}
    for key, (low, high) in bounds.items():
        value = s[key]
        if isinstance(value, (bool, np.bool_)):
            raise ValueError(key + ': expected a number, not bool')
        value = float(value)
        if not np.isfinite(value) or not low <= value <= high:
            raise ValueError(f'{key}: expected finite value in [{low}, {high}]')
        s[key] = value
    for key, low in [('acquisition_max_studies', 0), ('acquisition_max_slots', 1),
                     ('minimum_alternate_slices', 1), ('decoder_probe_series', 1)]:
        value = s[key]
        if type(value) is not int or value < low:
            raise ValueError(key + ': invalid integer')
    for key in ('global96_enabled', 'allow_secondary_acquisition', 'exclude_duplicate_context'):
        if type(s[key]) is not bool:
            raise ValueError(key + ': expected bool')
    if s['global96_enabled'] and s['global96_family_share'] == 0:
        raise ValueError('Disable global96_enabled instead of paying for a zero-weight model')
    if not s['global96_enabled'] and s['global96_family_share'] != 0:
        raise ValueError('global96_family_share must be zero when its model is disabled')
    return s


def v2_coat_hybrid(public_raptor: np.ndarray, anchor_family: np.ndarray,
                   global96: np.ndarray | None = None, *, share: float = 0.) -> np.ndarray:
    """Keep the original two-CoAt family as one unit. Rank ONLY full cohorts.

    At share=0 the existing family is untouched. When enabled, blend that
    family with Global96 and rank the family; retain the parent's 60/40
    public/family junction, with NO extra rerank of that hybrid.
    """
    public_raptor = ma_check_prob(np.asarray(public_raptor), 'public Raptor')
    family = ma_check_prob(np.asarray(anchor_family), 'anchor CoAt family')
    if public_raptor.ndim != 2 or public_raptor.shape != family.shape:
        raise ValueError('CoAt family / Raptor shape mismatch')
    if len(family) == 0 or family.shape[1] != len(MA_LABELS):
        raise ValueError('Expected nonempty N x 12 complete-cohort matrices')
    if not np.isfinite(share) or not 0 <= share <= .50:
        raise ValueError('Unvalidated Global96 share must be in [0, .50]')
    if share:
        if global96 is None:
            raise ValueError('Nonzero Global96 share requires its complete predictions')
        g = ma_check_prob(np.asarray(global96), 'Global96')
        if g.shape != family.shape:
            raise ValueError('Global96 must cover every study and label')
        family = ma_rank((1.-share)*family + share*ma_rank(g))
    # Exact coefficient arithmetic and placement retained from the parent.
    return (1.0 - .4)*ma_rank(public_raptor) + .4*family


def _v2_count(value, name):
    if isinstance(value,(bool,np.bool_)):
        raise ValueError(name+': boolean is not a count')
    try: number=float(value)
    except (TypeError,ValueError): raise ValueError(name+': invalid count') from None
    if not np.isfinite(number) or number < 0 or number != int(number):
        raise ValueError(name+': expected nonnegative integer count')
    return int(number)


def v2_validate_global96_receipt(receipt: dict, ids: list[str], output: str | Path,
                               *, expected_status: str | None = None) -> pd.DataFrame:
    """Never admit partial/neutral Global96 outputs into the final ensemble."""
    if not isinstance(receipt, dict):
        raise ValueError('Global96 receipt must be an object')
    if not receipt.get('status') or (expected_status is not None and receipt['status'] != expected_status):
        raise ValueError('Global96 status mismatch')
    if _v2_count(receipt.get('models'),'models') != 3:
        raise ValueError('Global96 must load all three checkpoints')
    checkpoints = receipt.get('checkpoints')
    if not isinstance(checkpoints, list) or [r.get('epoch') for r in checkpoints] != V2_GLOBAL96_EPOCHS:
        raise ValueError('Global96 checkpoint epoch/order mismatch')
    if 'fallback_studies' not in receipt or _v2_count(receipt['fallback_studies'],'fallback_studies') != 0:
        raise ValueError('Global96 contains fallback/failed studies')
    for field in ('studies', 'study_count'):
        if field in receipt and _v2_count(receipt[field],field) != len(ids):
            raise ValueError('Global96 receipt study count mismatch')
    if 'processes' in receipt:
        processes = receipt['processes']
        if not processes or any(p.get('returncode') != 0 for p in processes):
            raise ValueError('Global96 worker subprocess failed')
    for shard in receipt.get('shards', []):
        for field in ('fallback_studies', 'preparation_warnings'):
            if _v2_count(shard.get(field,0),field) != 0:
                raise ValueError('Global96 shard has fallback/warning: ' + field)
    frame = pd.read_csv(output, dtype={'StudyInstanceUID': str}, float_precision='round_trip')
    # Do not silently reorder a worker output: run_submission is expected to preserve input order.
    if frame.get('StudyInstanceUID', pd.Series(dtype=str)).astype(str).tolist() != list(map(str, ids)):
        raise ValueError('Global96 output UID order differs from test.csv')
    frame = ma_frame(frame, ids)
    actual_sha = ma_sha(output)
    if 'output_sha256' in receipt and receipt['output_sha256'] != actual_sha:
        raise ValueError('Global96 output checksum mismatch')
    return frame


def v2_run_checked_child(command: list[str], environment: dict, logfile: str | Path,
                         timeout_seconds: float) -> dict:
    """Kill the entire process group on timeout/interrupt; never keep partial CSVs.

    The caller supplies the remaining whole-notebook budget LESS its reserved
    finalization time. This function is tested with real CPU subprocesses.
    """
    if not command or not np.isfinite(timeout_seconds) or timeout_seconds <= 0:
        raise ValueError('A command and a positive finite child budget are required')
    path = Path(logfile); path.parent.mkdir(parents=True, exist_ok=True)
    began = time.monotonic()
    with path.open('w') as handle:
        process = subprocess.Popen(command, env=environment, stdout=handle,
            stderr=subprocess.STDOUT, start_new_session=True)
        try:
            code = process.wait(timeout=timeout_seconds)
        except BaseException:
            try: os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError: pass
            try: process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                try: os.killpg(process.pid, signal.SIGKILL)
                except ProcessLookupError: pass
                process.wait(timeout=5)
            # A child may have exited while a descendant still owns its session.
            try: os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError: pass
            raise
    # A completed launcher must not leave background GPU workers behind, even on failure.
    try: os.killpg(process.pid, signal.SIGKILL)
    except ProcessLookupError: pass
    if code != 0:
        with path.open('rb') as handle:
            handle.seek(max(0, path.stat().st_size-6000))
            tail = handle.read().decode('utf-8', errors='replace')
        raise RuntimeError(f'Model child exited {code}; {path}\n{tail}')
    return {'returncode': code, 'elapsed_seconds': time.monotonic()-began, 'log': str(path)}


def v2_remaining_budget(started: float, settings: dict, now: float | None = None) -> float:
    now = time.time() if now is None else float(now)
    remaining = settings['time_budget_hours']*3600 - (now-started) - settings['publication_reserve_seconds']
    if not np.isfinite(remaining) or remaining <= 0:
        raise TimeoutError('Insufficient remaining budget for Global96 and final publication')
    return float(remaining)


In [ ]:
"""MAST v3: series-coherent pooling, with frozen global pooling retained.

A triplet is 'same-series' only when all its source slices belong to one
acquisition. This is NOT a claim that the sampled slices are native neighbors,
that the checkpoint was trained without boundary windows, or that masking
improves AUC. No labels or test-set accuracy proxies are used to set weights.
"""
from __future__ import annotations
import hashlib
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd

V3_ARM_NAMES = ('maxspan-v5', 'native384dense-v10', 'maxspan-v5-reverse', 'native384-v8')
V3_ARM_WEIGHTS = np.asarray([.60, .10, .10, .20], dtype=np.float64)
V3_BANK_SCHEMA = 'mast-v3-series-coherent-v1'


def v3_check_settings(settings: dict) -> dict:
    s = v2_check_settings(settings)
    for key, lo, hi in [('series_coherence_alpha', 0., .35),
                        ('series_min_fraction', .25, 1.)]:
        value = s[key]
        if isinstance(value, (bool, np.bool_)) or not np.isscalar(value):
            raise ValueError(key + ': expected scalar number')
        value = float(value)
        if not np.isfinite(value) or not lo <= value <= hi:
            raise ValueError(f'{key}: expected finite value in [{lo}, {hi}]')
        s[key] = value
    if type(s['series_min_windows']) is not int or not 3 <= s['series_min_windows'] <= 94:
        raise ValueError('series_min_windows must be an integer in [3,94]')
    if type(s['raptor_micro_images']) is not int or s['raptor_micro_images'] not in (1,2,4,8):
        raise ValueError('raptor_micro_images must be one of 1,2,4,8')
    return s


def v3_series_mask(source_series, centers=None) -> np.ndarray:
    """Mark exact global-stack triplets whose three sources share an acquisition.

    Center indices must describe the ACTUAL unchanged eval_windows call.
    Source positions and ImageNet channel normalization are not modified.
    """
    groups = np.asarray(source_series)
    if groups.ndim != 1 or len(groups) < 3:
        raise ValueError('Need at least three source-series identifiers')
    if any(not isinstance(g, (str, np.str_)) or not str(g).strip() for g in groups):
        raise ValueError('Source-series identifiers must be nonempty strings')
    if centers is None:
        centers = np.arange(1, len(groups)-1, dtype=np.int64)
    centers = np.asarray(centers)
    if centers.ndim != 1 or len(centers) == 0 or centers.dtype.kind not in 'iu':
        raise ValueError('Need nonempty integer center indices')
    if int(centers.min()) < 1 or int(centers.max()) >= len(groups)-1:
        raise ValueError('A triplet center leaves its source stack')
    return ((groups[centers-1] == groups[centers]) &
            (groups[centers] == groups[centers+1])).astype(bool)


def v3_mask_policy(mask: np.ndarray, settings: dict) -> dict:
    mask = np.asarray(mask)
    if mask.ndim != 1 or mask.dtype != np.bool_ or not 1 <= len(mask) <= 94:
        raise ValueError('Expected 1..94 boolean Raptor window flags')
    count, total = int(mask.sum()), len(mask)
    qualifies = (0 < count < total and count >= settings['series_min_windows']
                 and count/total >= settings['series_min_fraction'])
    active = bool(qualifies and settings['series_coherence_alpha'] > 0.)
    reason = ('DISABLED_ALPHA_ZERO' if settings['series_coherence_alpha'] == 0. else
              'ALL_SAME_SERIES' if count == total else
              'INSUFFICIENT_SAME_SERIES_CONTEXT' if not qualifies else
              'SERIES_COHERENT_HEAD')
    return {'windows':total, 'same_series_windows':count,
            'boundary_windows':total-count, 'applied':active, 'reason':reason}


def v3_blend_raw(anchor, coherent, applied, alpha: float) -> np.ndarray:
    """Mix per-view raw probabilities; never rank only a selected subset.

    Leave bypass rows bit-exact after float64 promotion, including ties.
    The two arrays have the SAME model/view, study and finding order.
    """
    a = np.asarray(anchor); c = np.asarray(coherent); ok = np.asarray(applied)
    ma_check_prob(a, 'Raptor anchor bank'); ma_check_prob(c, 'Raptor coherent bank')
    if a.ndim != 3 or a.shape != c.shape or a.shape[0] != 4 or a.shape[2] != 12 or a.shape[1] < 1:
        raise ValueError('Expected matching 4 x N x 12 Raptor banks')
    if ok.shape != a.shape[:2] or ok.dtype != np.bool_:
        raise ValueError('Applied mask must be boolean 4 x N')
    if isinstance(alpha,(bool,np.bool_)) or not np.isfinite(alpha) or not 0 <= alpha <= .35:
        raise ValueError('series coherence alpha must be finite in [0,.35]')
    if not np.array_equal(a[~ok], c[~ok]):
        raise ValueError('A bypass row must contain its exact original prediction')
    result = a.astype(np.float64, copy=True)
    if alpha and ok.any():
        # Delta form preserves equal inputs exactly; no unrelated labels are mixed.
        result[ok] += float(alpha)*(c[ok].astype(np.float64)-result[ok])
    ma_check_prob(result, 'blended Raptor bank')
    return result


def v3_public_scores(raw) -> np.ndarray:
    """Replay original Raptor weights, rank01, float32 and CSV stage boundary.

    This endpoint rank convention is NOT percentile rank. The distinction is
    retained even though the next CoAt junction reranks the complete cohort.
    """
    raw = np.asarray(raw)
    ma_check_prob(raw, 'Raptor raw bank')
    if raw.ndim != 3 or raw.shape[0] != 4 or raw.shape[2] != 12 or raw.shape[1] < 1:
        raise ValueError('Expected 4 x N x 12 Raptor bank')
    weights = V3_ARM_WEIGHTS / V3_ARM_WEIGHTS.sum()
    avg = np.tensordot(weights, np.stack([np.clip(a,0,1) for a in raw]), axes=(0,0))
    if len(avg) == 1:
        ranked = np.full_like(avg, .5)
    else:
        ranked = (pd.DataFrame(avg).rank(method='average').to_numpy()-1)/(len(avg)-1)
    return ma_csv_values(ranked.astype(np.float32))


def v3_save_bank(folder, ids, anchor, coherent, applied, counts, micro_used):
    """Publish a complete same-run paired prediction bank plus a byte receipt."""
    folder=Path(folder); folder.mkdir(parents=True,exist_ok=True)
    ids=list(map(str,ids))
    a=np.asarray(anchor); c=np.asarray(coherent); ok=np.asarray(applied)
    cnt=np.asarray(counts); micro=np.asarray(micro_used)
    # Strict schema validation before writing is repeated by the reader.
    v3_blend_raw(a,c,ok,0.)
    if len(set(ids)) != len(ids) or a.shape[1] != len(ids) or not all(ids):
        raise ValueError('Invalid bank study identifiers')
    if cnt.shape != (4,len(ids),2) or cnt.dtype.kind not in 'iu':
        raise ValueError('Counts must be integer [4,N,2]: total and same-series')
    if micro.shape != (4,len(ids)) or micro.dtype.kind not in 'iu':
        raise ValueError('Microbatch receipt must be integer 4 x N')
    if (cnt[...,0]<1).any() or (cnt[...,0]>94).any() or (cnt[...,1]<0).any() or (cnt[...,1]>cnt[...,0]).any():
        raise ValueError('Invalid window counts')
    if not np.isin(micro,[1,2,4,8]).all():
        raise ValueError('Invalid executed microbatch size')
    path=folder/'raptor_series_coherent_raw.npz'
    temp=path.with_name('.'+path.name+'.tmp')
    with temp.open('wb') as handle:
        np.savez_compressed(handle, schema=np.asarray(V3_BANK_SCHEMA),
            study_uids=np.asarray(ids), labels=np.asarray(MA_LABELS),
            arm_names=np.asarray(V3_ARM_NAMES), anchor=a, coherent=c,
            applied=ok, window_counts=cnt, micro_images=micro)
        handle.flush(); os.fsync(handle.fileno())
    os.replace(temp,path)
    report={'schema':V3_BANK_SCHEMA,'file':path.name,'sha256':ma_sha(path),
            'studies':len(ids),'views':4,'labels':12,'applied_view_studies':int(ok.sum()),
            'actual_micro_images':sorted(np.unique(micro).astype(int).tolist()),
            'score_improvement_verified':False}
    rec=path.with_suffix('.json'); tmp=rec.with_name('.'+rec.name+'.tmp')
    with tmp.open('w') as handle:
        json.dump(report,handle,indent=2,sort_keys=True,allow_nan=False)
        handle.write('\n'); handle.flush(); os.fsync(handle.fileno())
    os.replace(tmp,rec)
    return report


def v3_load_bank(folder, ids, original_raw_path, settings):
    folder=Path(folder); ids=list(map(str,ids))
    path=folder/'raptor_series_coherent_raw.npz'
    report=json.loads(path.with_suffix('.json').read_text())
    if report.get('sha256')!=ma_sha(path) or report.get('schema')!=V3_BANK_SCHEMA:
        raise RuntimeError('Series-coherent bank checksum/schema mismatch')
    with np.load(path,allow_pickle=False) as z:
        if z['schema'].item()!=V3_BANK_SCHEMA or z['study_uids'].astype(str).tolist()!=ids:
            raise RuntimeError('Series-coherent bank cohort/schema mismatch')
        if z['labels'].astype(str).tolist()!=MA_LABELS or tuple(z['arm_names'].astype(str))!=V3_ARM_NAMES:
            raise RuntimeError('Series-coherent bank finding/view order mismatch')
        a,c,ok,cnt,micro=(z[k].copy() for k in ('anchor','coherent','applied','window_counts','micro_images'))
    if cnt.shape!=(4,len(ids),2) or cnt.dtype.kind not in 'iu':
        raise RuntimeError('Series-coherent count matrix mismatch')
    if micro.shape!=(4,len(ids)) or not np.isin(micro,[1,2,4,8]).all():
        raise RuntimeError('Series-coherent microbatch receipt mismatch')
    v3_blend_raw(a,c,ok,0.)
    total,clean=cnt[...,0],cnt[...,1]
    if (total<1).any() or (total>94).any() or (clean<0).any() or (clean>total).any():
        raise RuntimeError('Invalid series-coherent window counts')
    expected=((clean>=settings['series_min_windows']) & (clean<total) &
              (clean/total>=settings['series_min_fraction']) & (settings['series_coherence_alpha']>0.))
    if not np.array_equal(ok,expected):
        raise RuntimeError('Applied views do not match the frozen geometric eligibility policy')
    with np.load(original_raw_path,allow_pickle=False) as z:
        if z['study_uids'].astype(str).tolist()!=ids or not np.array_equal(z['raw_probabilities'],a):
            raise RuntimeError('Same-run original Raptor bank changed or was reordered')
    if report.get('studies')!=len(ids) or report.get('applied_view_studies')!=int(ok.sum()):
        raise RuntimeError('Series-coherent receipt counts changed')
    return {'anchor':a,'coherent':c,'applied':ok,'counts':cnt,'micro_images':micro,'receipt':report}


def v3_inline_selfcheck():
    groups=['A']*10+['B']*10+['C']*10
    mask=v3_series_mask(groups)
    assert len(mask)==28 and int((~mask).sum())==4
    assert np.flatnonzero(~mask).tolist()==[8,9,18,19]
    r=np.random.default_rng(3)
    a=r.random((4,19,12)).astype(np.float32); c=a.copy()
    ok=np.zeros((4,19),bool); ok[:,::2]=True; c[ok]=r.random((ok.sum(),12))
    frozen=a.copy()
    assert np.array_equal(v3_blend_raw(a,c,ok,0),a)
    mixed=v3_blend_raw(a,c,ok,.20)
    assert np.array_equal(mixed[~ok],a[~ok]) and np.array_equal(a,frozen)
    assert v3_public_scores(mixed).shape==(19,12)
    assert np.array_equal(v3_public_scores(np.full((4,1,12),.4)),np.full((1,12),.5))
    print('[v3 CPU checks] series boundaries, raw fusion, exact bypass and rank01: PASS')


"""MAST v4: bounded gap coverage with unchanged reference preprocessing.

No ground-truth labels are accessed. The new image evidence is an experiment,
not a demonstrated score improvement. Existing studies/views never get dropped.
"""
import hashlib, json, math, os
from pathlib import Path
import numpy as np

V4_BANK_SCHEMA = 'mast-v4-gap-coverage-v1'
V4_GAP_ARM_NAMES = ('maxspan-v5', 'native384-v8')
V4_SETTING_KEYS = ('gap_view_alpha', 'gap_max_windows', 'gap_max_fraction',
                   'gap_attempt_multiplier', 'gap_spacing_ratio',
                   'series_coherence_alpha', 'series_min_windows',
                   'series_min_fraction', 'raptor_micro_images')


def v4_check_settings(settings: dict) -> dict:
    s = v3_check_settings(settings)
    for key, lo, hi in [('gap_view_alpha', 0., .35),
                        ('gap_max_fraction', 0., .20),
                        ('gap_spacing_ratio', 1., 2.)]:
        value = s[key]
        if isinstance(value, (bool, np.bool_)) or not np.isscalar(value):
            raise ValueError(key + ': expected a scalar number')
        value = float(value)
        if not np.isfinite(value) or not lo <= value <= hi:
            raise ValueError(f'{key}: expected finite value in [{lo},{hi}]')
        s[key] = value
    if type(s['gap_max_windows']) is not int or not 0 <= s['gap_max_windows'] <= 16:
        raise ValueError('gap_max_windows must be an integer in [0,16]')
    if type(s['gap_attempt_multiplier']) is not int or not 1 <= s['gap_attempt_multiplier'] <= 4:
        raise ValueError('gap_attempt_multiplier must be an integer in [1,4]')
    return s


def v4_settings_sha(settings: dict) -> str:
    s = v4_check_settings(settings)
    payload = {k: s[k] for k in V4_SETTING_KEYS}
    payload['arms'] = list(V4_GAP_ARM_NAMES)
    return hashlib.sha256(json.dumps(payload, sort_keys=True,
                         separators=(',', ':'), allow_nan=False).encode()).hexdigest()


def v4_gap_budget(arm_name: str, original_windows: int, settings: dict) -> int:
    if isinstance(original_windows, (bool, np.bool_)) or not isinstance(original_windows, (int, np.integer)):
        raise ValueError('original_windows must be an integer')
    if not 1 <= original_windows <= 94:
        raise ValueError('Expected 1..94 original windows')
    if arm_name not in V4_GAP_ARM_NAMES or settings['gap_view_alpha'] == 0.:
        return 0
    return min(settings['gap_max_windows'],
               int(math.floor(original_windows * settings['gap_max_fraction'] + 1.e-12)))


def v4_iter_gap_candidates(records: list[dict], limit: int):
    """Greedy coverage in ordered-index space, not a clinical saliency estimate.

    Each proposed center was omitted by the original encoder input. Expand a
    proposed center to its two consecutive ordered neighbors. Prefer the largest
    normalized uncovered interval and avoid spending two proposals on a covered
    center. Rejected geometric proposals still consume a bounded attempt.
    """
    if type(limit) is not int or limit < 0:
        raise ValueError('Invalid gap proposal limit')
    state = []
    keys = set()
    for record in records:
        files = record['files']; lo = int(record['lo']); hi = int(record['hi'])
        picks = np.asarray(record['picks'])
        if picks.ndim != 1 or picks.dtype.kind not in 'iu' or not len(picks):
            raise ValueError('Gap records need nonempty integer selected slice indices')
        if not 0 <= lo <= hi < len(files) or (picks < lo).any() or (picks > hi).any():
            raise ValueError('Gap pool indices outside ordered acquisition')
        if len(set(picks.tolist())) != len(picks):
            raise ValueError('Original acquisition indices were not unique')
        paths = [str(f[0]) for f in files]
        if len(set(paths)) != len(paths):
            raise ValueError('Duplicate source paths in ordered acquisition')
        parents = {str(Path(p).parent) for p in paths}
        if len(parents) != 1:
            raise ValueError('One gap pool must be exactly one acquisition folder')
        key = next(iter(parents))
        if key in keys:
            raise ValueError('Duplicate acquisition gap pool')
        keys.add(key)
        original = set(map(int, picks))
        state.append({'record':record, 'key':key, 'original':original,
                      'covered':set(original), 'used_centers':set()})
    state.sort(key=lambda r:r['key'])
    for _ in range(limit):
        choices = []
        for s in state:
            r = s['record']; lo = int(r['lo']); hi = int(r['hi'])
            covered = np.asarray(sorted(s['covered']), dtype=np.int64)
            candidates=np.arange(lo+1,hi,dtype=np.int64)
            candidates=candidates[~np.isin(candidates,covered)]
            if not len(candidates):continue
            # Vectorized nearest selected index; avoid a Python/NumPy reduction
            # for every unselected slice in a long acquisition.
            at=np.searchsorted(covered,candidates)
            left=np.abs(candidates-covered[np.maximum(at-1,0)])
            right=np.abs(candidates-covered[np.minimum(at,len(covered)-1)])
            distance=np.minimum(left,right)
            novelty=1+(~np.isin(candidates-1,covered)).astype(np.int64)+(~np.isin(candidates+1,covered)).astype(np.int64)
            scale=max(1,hi-lo)
            midpoint=np.abs(candidates-(lo+hi)/2)/scale
            order=np.lexsort((candidates,midpoint,-novelty,-distance/scale))
            j=int(order[0]);c=int(candidates[j])
            score=(int(distance[j])/scale,int(novelty[j]),-float(midpoint[j]),-c)
            choices.append((score,s['key'],c,s))
        if not choices:
            break
        choices.sort(key=lambda t:(tuple(-float(x) for x in t[0]),t[1]))
        _,_,center,s = choices[0]
        r=s['record']; tri=(center-1,center,center+1)
        s['used_centers'].add(center); s['covered'].update(tri)
        yield {'record':r, 'center':center, 'indices':tri,
               'new_source_indices':[i for i in tri if i not in s['original']]}


def v4_gap_candidates(records: list[dict], limit: int) -> list[dict]:
    return list(v4_iter_gap_candidates(records,limit))


def v4_check_triplet_geometry(headers: list[dict], ratio_limit: float = 1.5) -> dict:
    """Require consistent patient-coordinate geometry; never use InstanceNumber
    as proof of physical adjacency. Consecutive available files may still have
    missing acquisitions/slices. This is not an anatomical localization test.
    """
    if len(headers) != 3:
        raise ValueError('A gap window needs exactly three geometry records')
    try:
        iop=np.asarray([h['iop'] for h in headers],dtype=np.float64)
        ipp=np.asarray([h['ipp'] for h in headers],dtype=np.float64)
        spacing=np.asarray([h['spacing'] for h in headers],dtype=np.float64)
        shape=np.asarray([h['shape'] for h in headers],dtype=np.int64)
    except (KeyError,ValueError,TypeError,OverflowError) as exc:
        raise ValueError('MISSING_OR_MALFORMED_GEOMETRY') from exc
    if iop.shape!=(3,6) or ipp.shape!=(3,3) or spacing.shape!=(3,2) or shape.shape!=(3,2):
        raise ValueError('MALFORMED_GEOMETRY_SHAPES')
    if not all(np.isfinite(x).all() for x in (iop,ipp,spacing)) or (spacing<=0).any() or (shape<2).any():
        raise ValueError('INVALID_GEOMETRY_VALUES')
    if not np.all(shape==shape[0]) or not np.allclose(spacing,spacing[0],rtol=1.e-3,atol=1.e-4):
        raise ValueError('INCONSISTENT_PIXEL_GEOMETRY')
    if not np.allclose(iop,iop[0],rtol=0,atol=1.e-3):
        raise ValueError('INCONSISTENT_ORIENTATION')
    row,col=iop[0,:3],iop[0,3:]
    if not np.isclose(np.linalg.norm(row),1,atol=1.e-3) or not np.isclose(np.linalg.norm(col),1,atol=1.e-3) or abs(float(row@col))>1.e-3:
        raise ValueError('INVALID_DIRECTION_COSINES')
    normal=np.cross(row,col); normal/=np.linalg.norm(normal)
    projected=ipp@normal; gaps=np.diff(projected)
    if (gaps<=1.e-4).any():
        raise ValueError('DUPLICATE_OR_REVERSED_POSITION')
    ratio=float(gaps.max()/gaps.min())
    if ratio>ratio_limit:
        raise ValueError('IRREGULAR_SLICE_GAPS')
    delta=np.diff(ipp,axis=0)
    drift=delta-gaps[:,None]*normal
    if float(np.linalg.norm(drift,axis=1).max())>max(.5,.25*float(spacing[0].min())):
        raise ValueError('IN_PLANE_ORIGIN_DRIFT')
    return {'gaps_mm':gaps.tolist(),'gap_ratio':ratio,
            'row_spacing_mm':float(spacing[0,0]),'column_spacing_mm':float(spacing[0,1])}


def v4_read_geometry(path: str) -> dict:
    import pydicom
    try:
        h=pydicom.dcmread(path,stop_before_pixels=True,specific_tags=[
            'ImageOrientationPatient','ImagePositionPatient','PixelSpacing','Rows','Columns'])
    except MemoryError:
        raise
    except Exception as exc:
        # Normalize parser/IO-specific exceptions only at this optional-file
        # boundary; programming errors in model execution are never suppressed.
        raise OSError(f'Cannot read optional geometry: {path}: {type(exc).__name__}') from exc
    return {'iop':getattr(h,'ImageOrientationPatient',None),
            'ipp':getattr(h,'ImagePositionPatient',None),
            'spacing':getattr(h,'PixelSpacing',None),
            'shape':[getattr(h,'Rows',0),getattr(h,'Columns',0)]}


def v4_prepare_gap_windows(arm: dict, original_windows, records: list[dict],
                           read_pixel, settings: dict, geometry_reader=None):
    """Create ONLY novel gap windows; leave the original tensor untouched.

    Same acquisition-specific percentile anchors, 140 mm crop, uint8 boundary,
    interpolation and ImageNet normalization as the inherited Raptor input.
    Additional unavailable/invalid optional slices are rejected with a receipt.
    Original required decoder/model failures are not caught here.
    """
    import torch
    import torch.nn.functional as F
    import cv2
    budget=v4_gap_budget(arm['name'],len(original_windows),settings)
    empty=original_windows.new_empty((0,*original_windows.shape[1:]))
    report={'budget':budget,'accepted':[],'rejected':[],'new_encoder_source_count':0,
            'attempted':0,'extra_windows':0,'index_policy':'largest_normalized_uncovered_interval',
            'geometry_policy':'iop_ipp_consistent_positive_regular_local_gaps'}
    if budget==0:
        report['reason']='DISABLED_OR_ZERO_BUDGET'
        return empty,report
    # Lazy proposals: do not compute the unused retry budget when images pass.
    proposals=v4_iter_gap_candidates(records,budget*settings['gap_attempt_multiplier'])
    geometry_reader=v4_read_geometry if geometry_reader is None else geometry_reader
    headers={}; rendered={}; accepted=[]; new_sources=set()
    orig_paths={str(r['files'][int(i)][0]) for r in records for i in r['picks']}
    seen=set()
    def get_header(path):
        if path not in headers:
            headers[path]=geometry_reader(path)
        return headers[path]
    for proposal in proposals:
        if len(accepted)>=budget:
            break
        report['attempted']+=1
        record=proposal['record']; indices=proposal['indices']
        paths=tuple(str(record['files'][i][0]) for i in indices)
        identity={'center_index':int(proposal['center']),'sources':list(paths)}
        if paths in seen or paths[1] in orig_paths or len(set(paths))!=3:
            raise RuntimeError('Gap selector supplied duplicate or already encoded center')
        seen.add(paths)
        try:
            geometry=v4_check_triplet_geometry([get_header(p) for p in paths],settings['gap_spacing_ratio'])
        except (ValueError,KeyError,TypeError,AttributeError,OSError,EOFError) as exc:
            report['rejected'].append({**identity,'reason':'GEOMETRY_REJECTED','detail':str(exc)[:200]})
            continue
        low,high=map(float,record['normalization'])
        if not np.isfinite([low,high]).all() or high<=low:
            report['rejected'].append({**identity,'reason':'DEGENERATE_NORMALIZATION'})
            continue
        planes=[]
        try:
            for index,path in zip(indices,paths):
                key=(path,int(arm['img']),low,high)
                if key not in rendered:
                    raw=np.asarray(read_pixel(path))
                    if raw.ndim!=2 or min(raw.shape)<2 or not np.isfinite(raw).all():
                        raise ValueError('Invalid optional gap pixels')
                    if tuple(raw.shape)!=tuple(get_header(path)['shape']):
                        raise ValueError('Pixel/header shape mismatch')
                    ps=float(record['files'][index][1])
                    if not np.isfinite(ps) or ps<=0:
                        raise ValueError('Invalid crop spacing')
                    a=np.clip((raw-low)/(high-low+1.e-6),0,1)
                    h,w=a.shape; c=min(int(round(140.0/max(ps,.001))),min(h,w))
                    if c<2:
                        raise ValueError('Empty optional gap crop')
                    y,x=(h-c)//2,(w-c)//2
                    resized=cv2.resize(a[y:y+c,x:x+c],(int(arm['img']),int(arm['img'])),interpolation=cv2.INTER_AREA)
                    rendered[key]=(resized*255).astype(np.uint8)
                planes.append(rendered[key])
        except (ValueError,TypeError,AttributeError,OSError,EOFError,RuntimeError) as exc:
            report['rejected'].append({**identity,'reason':'OPTIONAL_PIXELS_REJECTED','detail':str(exc)[:200]})
            continue
        window=np.stack(planes).astype(np.float32)/255.0
        if float(np.ptp(window))==0.:
            report['rejected'].append({**identity,'reason':'CONSTANT_EXTRA_WINDOW'})
            continue
        accepted.append(window)
        newly=sorted(set(paths)-orig_paths); new_sources.update(newly)
        report['accepted'].append({**identity,**geometry,'new_encoder_sources':newly})
    if accepted:
        x=torch.from_numpy(np.stack(accepted))
        if tuple(x.shape[-2:])!=(int(arm['res']),int(arm['res'])):
            # Same per-plane bilinear convention as inherited eval_windows.
            flat=x.reshape(-1,1,*x.shape[-2:])
            flat=F.interpolate(flat,size=(int(arm['res']),int(arm['res'])),mode='bilinear',align_corners=False)
            x=flat.reshape(len(x),3,int(arm['res']),int(arm['res']))
        mean=torch.tensor([.485,.456,.406],dtype=x.dtype).view(1,3,1,1)
        std=torch.tensor([.229,.224,.225],dtype=x.dtype).view(1,3,1,1)
        x=(x-mean)/std
        if not torch.isfinite(x).all():
            raise RuntimeError('Nonfinite prepared gap windows')
        if tuple(x.shape[1:])!=tuple(original_windows.shape[1:]):
            raise RuntimeError('Gap/reference window resolution mismatch')
    else:
        x=empty
    report.update(extra_windows=len(x),new_encoder_source_count=len(new_sources),
                  reason='NEW_IMAGE_COVERAGE' if len(x) else 'NO_ELIGIBLE_EXTRA_WINDOWS')
    report['source_receipt_sha256']=hashlib.sha256(json.dumps(report['accepted'],sort_keys=True,
        separators=(',',':'),allow_nan=False).encode()).hexdigest()
    return x,report


def v4_blend_raw(parent_bank: dict, expanded_anchor, expanded_coherent, extra_counts,
                  settings: dict) -> np.ndarray:
    a=parent_bank['anchor']; c=parent_bank['coherent']; ok=parent_bank['applied']
    parent=v3_blend_raw(a,c,ok,settings['series_coherence_alpha'])
    ea=np.asarray(expanded_anchor); ec=np.asarray(expanded_coherent); count=np.asarray(extra_counts)
    if ea.shape!=a.shape or ec.shape!=a.shape or count.shape!=a.shape[:2] or count.dtype.kind not in 'iu':
        raise ValueError('Gap bank shape/count type mismatch')
    ma_check_prob(ea,'expanded anchor'); ma_check_prob(ec,'expanded coherent')
    if (count<0).any(): raise ValueError('Negative gap count')
    enabled=count>0
    if not np.array_equal(ea[~enabled],a[~enabled]) or not np.array_equal(ec[~enabled],c[~enabled]):
        raise ValueError('Bypassed gap rows must preserve exact parent predictions')
    if not np.array_equal(ea[~ok],ec[~ok]):
        raise ValueError('Coherence-ineligible expanded views must use the unmasked head')
    if settings['gap_view_alpha']==0 and enabled.any():
        raise ValueError('Disabled coverage must not have computed added images')
    candidate=parent.copy()
    if enabled.any():
        expanded=v3_blend_raw(ea,ec,ok,settings['series_coherence_alpha'])
        candidate[enabled]+=float(settings['gap_view_alpha'])*(expanded[enabled]-parent[enabled])
    ma_check_prob(candidate,'v4 candidate probabilities')
    return candidate


def v4_save_bank(folder, ids, parent_bank: dict, expanded_anchor, expanded_coherent,
                  extra_counts, extra_micro, settings: dict, source_audit_path):
    folder=Path(folder); folder.mkdir(parents=True,exist_ok=True)
    ids=list(map(str,ids))
    v4_blend_raw(parent_bank,expanded_anchor,expanded_coherent,extra_counts,settings)
    if len(ids)!=parent_bank['anchor'].shape[1] or len(set(ids))!=len(ids) or not all(ids):
        raise ValueError('Invalid v4 bank UIDs')
    counts=np.asarray(extra_counts); micro=np.asarray(extra_micro)
    if micro.shape!=counts.shape or micro.dtype.kind not in 'iu':
        raise ValueError('Invalid v4 microbatch receipt')
    if not np.all(np.where(counts>0,np.isin(micro,[1,2,4,8]),micro==0)):
        raise ValueError('Gap microbatch/coverage mismatch')
    for k,name in enumerate(V3_ARM_NAMES):
        for i,n in enumerate(parent_bank['counts'][k,:,0]):
            if counts[k,i]>v4_gap_budget(name,int(n),settings):
                raise ValueError('Extra image budget exceeded')
    path=folder/'raptor_gap_coverage_raw.npz'; tmp=path.with_name('.'+path.name+'.tmp')
    with tmp.open('wb') as f:
        np.savez_compressed(f,schema=np.asarray(V4_BANK_SCHEMA),study_uids=np.asarray(ids),
            labels=np.asarray(MA_LABELS),arm_names=np.asarray(V3_ARM_NAMES),
            expanded_anchor=expanded_anchor,expanded_coherent=expanded_coherent,
            extra_counts=counts,extra_micro=micro)
        f.flush();os.fsync(f.fileno())
    os.replace(tmp,path)
    report={'schema':V4_BANK_SCHEMA,'sha256':ma_sha(path),'studies':len(ids),
            'parent_bank_sha256':parent_bank['receipt']['sha256'],
            'source_audit_sha256':ma_sha(source_audit_path),
            'settings_sha256':v4_settings_sha(settings),'extra_windows':int(counts.sum()),
            'applied_view_studies':int((counts>0).sum()),'is_oof':False,'measured_score':None}
    rec=path.with_suffix('.json'); tmp=rec.with_name('.'+rec.name+'.tmp')
    with tmp.open('w') as f:
        json.dump(report,f,indent=2,sort_keys=True,allow_nan=False);f.write('\n');f.flush();os.fsync(f.fileno())
    os.replace(tmp,rec)
    return report


def v4_load_bank(folder, ids, parent_bank: dict, settings: dict, source_audit_path):
    folder=Path(folder); ids=list(map(str,ids)); path=folder/'raptor_gap_coverage_raw.npz'
    report=json.loads(path.with_suffix('.json').read_text())
    expected={'schema':V4_BANK_SCHEMA,'sha256':ma_sha(path),'studies':len(ids),
              'parent_bank_sha256':parent_bank['receipt']['sha256'],
              'source_audit_sha256':ma_sha(source_audit_path),'settings_sha256':v4_settings_sha(settings)}
    if any(report.get(k)!=v for k,v in expected.items()):
        raise RuntimeError('Gap bank source/settings/checksum mismatch')
    with np.load(path,allow_pickle=False) as z:
        if z['schema'].item()!=V4_BANK_SCHEMA or z['study_uids'].astype(str).tolist()!=ids or z['labels'].astype(str).tolist()!=MA_LABELS or tuple(z['arm_names'].astype(str))!=V3_ARM_NAMES:
            raise RuntimeError('Gap bank cohort/schema mismatch')
        ea,ec,counts,micro=(z[k].copy() for k in ('expanded_anchor','expanded_coherent','extra_counts','extra_micro'))
    raw=v4_blend_raw(parent_bank,ea,ec,counts,settings)
    if micro.shape!=counts.shape or micro.dtype.kind not in 'iu' or not np.all(np.where(counts>0,np.isin(micro,[1,2,4,8]),micro==0)):
        raise RuntimeError('Invalid gap execution receipt')
    for k,name in enumerate(V3_ARM_NAMES):
        for i,n in enumerate(parent_bank['counts'][k,:,0]):
            if counts[k,i]>v4_gap_budget(name,int(n),settings):
                raise RuntimeError('Gap coverage exceeds frozen budget')
    if report.get('extra_windows')!=int(counts.sum()) or report.get('applied_view_studies')!=int((counts>0).sum()):
        raise RuntimeError('Gap bank count receipt mismatch')
    return {'expanded_anchor':ea,'expanded_coherent':ec,'counts':counts,
            'micro':micro,'raw':raw,'receipt':report}


def v4_inline_selfcheck():
    files=[(f'/test/study/series/{i}.dcm',.5) for i in range(30)]
    rec={'files':files,'lo':0,'hi':29,'picks':np.asarray([0,7,14,21,29])}
    p=v4_gap_candidates([rec],4)
    assert len(p)==4 and all(q['center'] not in set(rec['picks']) for q in p)
    headers=[{'iop':[1,0,0,0,1,0],'ipp':[0,0,i*3.], 'spacing':[.5,.5],'shape':[32,32]} for i in range(3)]
    assert v4_check_triplet_geometry(headers)['gaps_mm']==[3.,3.]
    expected=min(MASTER['gap_max_windows'],int(94*MASTER['gap_max_fraction'])) if MASTER['gap_view_alpha'] else 0
    assert v4_gap_budget('maxspan-v5',94,MASTER)==expected
    assert v4_gap_budget('native384dense-v10',94,MASTER)==0
    print('[v4 CPU checks] novel centers, geometric neighbors and view budgets: PASS')


def v4_validate_source_audit(path, ids, extra_counts, parent_counts, settings):
    """Link each computed extra-view count to the actual preparation record."""
    ids=list(map(str,ids)); lookup={uid:i for i,uid in enumerate(ids)}
    expected={(name,uid) for name in ('maxspan-v5','native384dense-v10','native384-v8') for uid in ids}
    observed=set()
    with Path(path).open() as f:
        for line in f:
            if not line.strip():continue
            r=json.loads(line); key=(r['arm'],r['uid'])
            if key not in expected or key in observed:
                raise RuntimeError('Unexpected or duplicate gap source-audit record')
            observed.add(key);k=V3_ARM_NAMES.index(r['arm']);i=lookup[r['uid']]
            info=r['gap_coverage']; count=int(extra_counts[k,i])
            if int(r['windows'])!=int(parent_counts[k,i,0]):
                raise RuntimeError('Source-audit original-window count mismatch')
            if info['extra_windows']!=count or len(info['accepted'])!=count:
                raise RuntimeError('Source-audit gap count mismatch')
            if info['budget']!=v4_gap_budget(r['arm'],int(r['windows']),settings):
                raise RuntimeError('Source-audit gap budget mismatch')
            if count:
                sha=hashlib.sha256(json.dumps(info['accepted'],sort_keys=True,
                    separators=(',',':'),allow_nan=False).encode()).hexdigest()
                if info.get('source_receipt_sha256')!=sha:
                    raise RuntimeError('Source-audit accepted triplets changed')
                centers=[]
                for q in info['accepted']:
                    paths=q['sources']; new=q['new_encoder_sources']
                    if len(paths)!=3 or len(set(paths))!=3 or not new or paths[1] not in new:
                        raise RuntimeError('Gap input did not contain an unseen center')
                    if len({str(Path(p).parent) for p in paths})!=1:
                        raise RuntimeError('Gap input crossed acquisitions')
                    centers.append(paths[1])
                if len(set(centers))!=len(centers):
                    raise RuntimeError('Repeated gap center in actual inputs')
    if observed!=expected:
        raise RuntimeError('Incomplete gap source-audit coverage')
    if np.asarray(extra_counts)[2].any():
        raise RuntimeError('Reverse view must not consume the forward-only gap images')
    return True


In [ ]:
"""MAST v5: preserve each represented acquisition's sampling mass.

This changes ONLY the gap-expanded head. No extra images, checkpoints, label
coupling, confidence routing, fitted weights, or test-label access are added.
The policy is a count-prior hypothesis, not an established accuracy correction.
"""
V5_PLAN_SCHEMA = 'mast-v5-acquisition-mass-plan-1'
V5_BANK_SCHEMA = 'mast-v5-acquisition-mass-bank-1'


def v5_check_settings(settings: dict) -> dict:
    result = v4_check_settings(settings)
    if type(result.get('sampling_mass_enabled')) is not bool:
        raise ValueError('sampling_mass_enabled must be a boolean')
    return result


def v5_json_sha(value) -> str:
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':'),
                                     allow_nan=False).encode()).hexdigest()


def v5_plan_from_groups(original_groups, extra_groups, *, enabled=True):
    """Each original coherent window has unit reference sampling mass.

    A series with n original and g added coherent windows assigns n/(n+g)
    to each of its n+g windows. Cross-series original windows retain unit mass.
    No reference support (n=0): retain the unweighted v4 treatment, never erase
    a newly available acquisition. No weights depend on model predictions.
    """
    if type(enabled) is not bool:
        raise ValueError('enabled must be bool')
    old, extra = list(original_groups), list(extra_groups)
    if not 1 <= len(old) <= 94 or len(extra) > 16:
        raise ValueError('Invalid original/extra window cardinality')
    if any(x is not None and (not isinstance(x, str) or not x.strip()) for x in old):
        raise ValueError('Original group must be a nonempty UID or None for a boundary')
    if any(not isinstance(x, str) or not x.strip() for x in extra):
        raise ValueError('Extra windows require a nonempty acquisition identity')
    groups = sorted(set(x for x in old if x is not None) | set(extra))
    weights = np.ones(len(old) + len(extra), np.float64)
    combined = old + extra
    records = []
    for sid in groups:
        n, g = old.count(sid), extra.count(sid)
        adjusted = bool(enabled and n > 0 and g > 0)
        value = n / (n + g) if adjusted else 1.0
        positions = [i for i, s in enumerate(combined) if s == sid]
        weights[positions] = value
        records.append({'series': sid, 'original': n, 'extra': g,
                        'unit_weight': value, 'adjusted': adjusted,
                        'reference_mass': float(n),
                        'expanded_mass': float((n + g) * value)})
    plan = {'schema': V5_PLAN_SCHEMA, 'enabled': enabled,
            'original_groups': old, 'extra_groups': extra,
            'original_windows': len(old), 'extra_windows': len(extra),
            'boundary_windows': old.count(None),
            'groups': records, 'log_weights': np.log(weights).tolist(),
            'applied': bool(enabled and np.ptp(weights) > 0),
            'uniform_global_rescale': bool(enabled and len(extra) and
                                          np.ptp(weights) == 0 and weights[0] != 1),
            'reference_mass': float(len(old)),
            'expanded_mass': float(weights.sum()),
            'new_without_reference': sum(r['extra'] for r in records if r['original'] == 0)}
    # A global constant softmax prior cancels exactly; reuse v4 in that case.
    plan['sha256'] = v5_json_sha(plan)
    return plan


def v5_validate_plan(plan, same_series=None, extra_count=None, enabled=None):
    if not isinstance(plan, dict) or plan.get('schema') != V5_PLAN_SCHEMA:
        raise ValueError('Missing/incompatible acquisition-mass plan')
    expected = v5_plan_from_groups(plan.get('original_groups', []),
                                  plan.get('extra_groups', []),
                                  enabled=plan.get('enabled'))
    if plan != expected:
        raise ValueError('Acquisition-mass plan/count/hash mismatch')
    if same_series is not None:
        mask = np.asarray(same_series)
        correct = np.asarray([x is not None for x in plan['original_groups']], bool)
        if mask.dtype != bool or not np.array_equal(mask, correct):
            raise ValueError('Acquisition identities disagree with actual window mask')
    if extra_count is not None and plan['extra_windows'] != int(extra_count):
        raise ValueError('Mass-plan extra count disagrees with actual image input')
    if enabled is not None and plan['enabled'] is not enabled:
        raise ValueError('Mass plan disagrees with frozen control')
    return expected


def v5_build_plan(source_series, centers, accepted, enabled=True):
    sources = list(source_series)
    index = np.asarray(centers)
    if index.ndim != 1 or index.dtype.kind not in 'iu' or not len(index):
        raise ValueError('Original window centers must be an integer vector')
    if any(not isinstance(x, str) or not x.strip() for x in sources):
        raise ValueError('Unknown original acquisition identity')
    if (index < 1).any() or (index >= len(sources) - 1).any():
        raise ValueError('Original window center outside source stack')
    old = [sources[c] if sources[c - 1] == sources[c] == sources[c + 1] else None
           for c in index]
    extras, seen = [], set()
    for item in accepted:
        paths = item.get('sources', [])
        if len(paths) != 3 or len(set(paths)) != 3 or not all(isinstance(p, str) and p for p in paths):
            raise ValueError('Invalid accepted triplet')
        parents = [str(Path(p).parent) for p in paths]
        if len(set(parents)) != 1 or Path(parents[0]).name in ('', '.', '/'):
            raise ValueError('Extra triplet crosses acquisitions')
        if paths[1] in seen:
            raise ValueError('Repeated extra center')
        seen.add(paths[1]); extras.append(Path(parents[0]).name)
    return v5_plan_from_groups(old, extras, enabled=enabled)


def v5_reverse_plan(forward_plan):
    """Channel reversal changes neither window identities nor coherence.

    The inherited reverse arm never receives the forward-only gap pixels.
    """
    p = v5_validate_plan(forward_plan)
    return v5_plan_from_groups(p['original_groups'], [], enabled=p['enabled'])


def v5_validate_source_audit(path, ids, parent_bank, gap_bank, settings):
    """Link frozen priors to actual source-window and accepted-triplet records."""
    ids = list(map(str, ids))
    if not ids or len(set(ids)) != len(ids):
        raise ValueError('Invalid expected study IDs')
    lookup = {uid: i for i, uid in enumerate(ids)}
    shapes = (4, len(ids))
    flags = np.zeros(shapes, bool)
    hashes = np.full(shapes, '', dtype='<U64')
    expected = {(name, uid) for name in (V3_ARM_NAMES[0], V3_ARM_NAMES[1], V3_ARM_NAMES[3]) for uid in ids}
    observed = set()
    with Path(path).open() as f:
        for line in f:
            if not line.strip(): continue
            row = json.loads(line); key = (row['arm'], row['uid'])
            if key not in expected or key in observed:
                raise RuntimeError('Unexpected/duplicate mass source record')
            observed.add(key)
            k, i = V3_ARM_NAMES.index(row['arm']), lookup[row['uid']]
            p = v5_validate_plan(row.get('sampling_mass'),
                                 extra_count=gap_bank['counts'][k, i],
                                 enabled=settings['sampling_mass_enabled'])
            if (p['original_windows'] != int(parent_bank['counts'][k, i, 0]) or
                p['original_windows'] - p['boundary_windows'] != int(parent_bank['counts'][k, i, 1])):
                raise RuntimeError('Mass plan does not describe original window counts')
            # The legacy source audit provides the actual boundary indices.
            boundaries = [j for j, name in enumerate(p['original_groups']) if name is None]
            if boundaries != row['cross_series_window_indices']:
                raise RuntimeError('Mass plan has altered cross-series boundaries')
            accepted = row['gap_coverage']['accepted']
            actual_extra = [Path(item['sources'][1]).parent.name for item in accepted]
            if actual_extra != p['extra_groups']:
                raise RuntimeError('Mass groups disagree with decoded extra acquisitions')
            hashes[k, i] = p['sha256']; flags[k, i] = p['applied']
            if k == 0:
                rev = v5_reverse_plan(p)
                if (rev['original_windows'] != int(parent_bank['counts'][2, i, 0]) or
                    rev['original_windows'] - rev['boundary_windows'] != int(parent_bank['counts'][2, i, 1])):
                    raise RuntimeError('Reverse view changed original window metadata')
                hashes[2, i] = rev['sha256']; flags[2, i] = rev['applied']
    if observed != expected or (hashes == '').any():
        raise RuntimeError('Incomplete sampling-mass source coverage')
    if flags[1].any() or flags[2].any():
        raise RuntimeError('A view with no added images must not use a new mass head')
    return flags, hashes


def v5_blend_raw(parent_bank, gap_bank, balanced, balanced_coherent, applied, settings):
    a, c = np.asarray(balanced), np.asarray(balanced_coherent)
    ok = np.asarray(applied)
    if ok.dtype != bool or ok.shape != gap_bank['counts'].shape:
        raise ValueError('Invalid mass-head applicability mask')
    if a.shape != gap_bank['expanded_anchor'].shape or c.shape != a.shape:
        raise ValueError('Invalid mass-head probability shape')
    ma_check_prob(a, 'mass all-window probabilities'); ma_check_prob(c, 'mass coherent probabilities')
    if (ok & (gap_bank['counts'] == 0)).any():
        raise ValueError('Cannot change mass without added images')
    if not settings['sampling_mass_enabled'] and ok.any():
        raise ValueError('Disabled mass head has active rows')
    if not np.array_equal(a[~ok], gap_bank['expanded_anchor'][~ok]) or not np.array_equal(c[~ok], gap_bank['expanded_coherent'][~ok]):
        raise ValueError('Bypassed mass rows must retain exact v4 probabilities')
    result = v4_blend_raw(parent_bank, a, c, gap_bank['counts'], settings)
    if not ok.any() and not np.array_equal(result, gap_bank['raw']):
        raise RuntimeError('No mass changes must exactly replay v4 raw fusion')
    return result


def v5_settings_sha(settings):
    return v5_json_sha({'v4': v4_settings_sha(settings),
        'series_alpha': settings['series_coherence_alpha'],
        'series_min_windows': settings['series_min_windows'],
        'series_min_fraction': settings['series_min_fraction'],
        'sampling_mass_enabled': settings['sampling_mass_enabled'],
        'policy': V5_PLAN_SCHEMA})


def v5_save_bank(folder, ids, parent_bank, gap_bank, balanced, balanced_coherent,
                 applied, plan_hashes, settings, source_audit_path):
    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)
    ids = list(map(str, ids))
    v5_blend_raw(parent_bank, gap_bank, balanced, balanced_coherent, applied, settings)
    flags, hashes = v5_validate_source_audit(source_audit_path, ids, parent_bank, gap_bank, settings)
    if not np.array_equal(flags, applied) or not np.array_equal(hashes, plan_hashes):
        raise RuntimeError('Executed mass policy does not match actual input plans')
    path = folder / 'raptor_sampling_mass_raw.npz'; tmp = path.with_name('.' + path.name + '.tmp')
    with tmp.open('wb') as f:
        np.savez_compressed(f, schema=np.asarray(V5_BANK_SCHEMA), study_uids=np.asarray(ids),
            labels=np.asarray(MA_LABELS), arm_names=np.asarray(V3_ARM_NAMES),
            balanced=balanced, balanced_coherent=balanced_coherent, applied=applied,
            plan_hashes=hashes)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)
    report = {'schema': V5_BANK_SCHEMA, 'sha256': ma_sha(path), 'studies': len(ids),
        'parent_bank_sha256': parent_bank['receipt']['sha256'],
        'gap_bank_sha256': gap_bank['receipt']['sha256'],
        'source_audit_sha256': ma_sha(source_audit_path), 'settings_sha256': v5_settings_sha(settings),
        'applied_view_studies': int(flags.sum()), 'is_oof': False, 'measured_score': None,
        'additional_encoder_images_vs_v4': 0}
    rec = path.with_suffix('.json'); tmp = rec.with_name('.' + rec.name + '.tmp')
    with tmp.open('w') as f:
        json.dump(report, f, indent=2, sort_keys=True, allow_nan=False); f.write('\n')
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, rec)
    return report


def v5_load_bank(folder, ids, parent_bank, gap_bank, settings, source_audit_path):
    folder = Path(folder); ids = list(map(str, ids)); path = folder / 'raptor_sampling_mass_raw.npz'
    report = json.loads(path.with_suffix('.json').read_text())
    expected = {'schema': V5_BANK_SCHEMA, 'sha256': ma_sha(path), 'studies': len(ids),
        'parent_bank_sha256': parent_bank['receipt']['sha256'],
        'gap_bank_sha256': gap_bank['receipt']['sha256'],
        'source_audit_sha256': ma_sha(source_audit_path), 'settings_sha256': v5_settings_sha(settings),
        'additional_encoder_images_vs_v4': 0, 'is_oof': False, 'measured_score': None}
    if any(report.get(key) != value for key, value in expected.items()):
        raise RuntimeError('Mass bank/receipt/settings/source mismatch')
    with np.load(path, allow_pickle=False) as z:
        if z['schema'].item() != V5_BANK_SCHEMA or z['study_uids'].astype(str).tolist() != ids or z['labels'].astype(str).tolist() != MA_LABELS or tuple(z['arm_names'].astype(str)) != V3_ARM_NAMES:
            raise RuntimeError('Mass bank UID/label/view/schema mismatch')
        a, c, ok, hashes = (z[key].copy() for key in ('balanced','balanced_coherent','applied','plan_hashes'))
    flags, expected_hashes = v5_validate_source_audit(source_audit_path, ids, parent_bank, gap_bank, settings)
    if not np.array_equal(ok, flags) or not np.array_equal(hashes, expected_hashes):
        raise RuntimeError('Saved mass policy does not match audited image inputs')
    raw = v5_blend_raw(parent_bank, gap_bank, a, c, ok, settings)
    if report.get('applied_view_studies') != int(ok.sum()):
        raise RuntimeError('Mass-bank application count mismatch')
    return {'balanced': a, 'balanced_coherent': c, 'applied': ok, 'raw': raw,
            'plan_hashes': hashes, 'receipt': report}


def v5_inline_selfcheck():
    p = v5_plan_from_groups(['A'] * 6 + [None] * 2 + ['B'] * 6, ['A'] * 3)
    w = np.exp(p['log_weights'])
    assert p['applied'] and np.allclose(w.sum(), 14., atol=1e-12)
    assert np.allclose(w[:6], 2/3) and np.array_equal(w[6:14], np.ones(8))
    assert v5_reverse_plan(p)['extra_windows'] == 0
    assert not v5_plan_from_groups(['A'] * 6, ['A'] * 3)['applied']
    assert not v5_plan_from_groups(['A'] * 6 + ['B'] * 6, ['A'], enabled=False)['applied']
    v5_validate_plan(p)
    print('[v5 CPU checks] per-acquisition mass, unchanged boundaries, constant-prior bypass: PASS')


In [ ]:
MASTER=v5_check_settings(MASTER)
v3_inline_selfcheck()
v4_inline_selfcheck()
v5_inline_selfcheck()


In [ ]:
# Quick deterministic CPU checks before model loading. Full build tests are separate.
def _v2_inline_selfcheck():
    rng=np.random.default_rng(20260921)
    r=rng.random((37,12)); f=ma_rank(rng.random((37,12))); g=rng.random((37,12))
    r0,f0,g0=r.copy(),f.copy(),g.copy()
    want=(1.0-.4)*ma_rank(r)+.4*f
    assert np.array_equal(v2_coat_hybrid(r,f,share=0),want)
    got=v2_coat_hybrid(r,f,g,share=.25)
    assert got.shape==(37,12) and np.isfinite(got).all()
    assert np.array_equal(r,r0) and np.array_equal(f,f0) and np.array_equal(g,g0)
    try:v2_coat_hybrid(r,f,g[:-1],share=.25)
    except ValueError:pass
    else:raise AssertionError('Partial Global96 array was not rejected')
    assert ma_flag('0') is False and ma_flag('1') is True and ma_flag('unknown') is None
    assert v2_remaining_budget(100.,MASTER,now=100.) > 0
    print('[v2 CPU checks] family zero-weight parity, complete-cohort schema, immutability: PASS')
MASTER=v2_check_settings(MASTER)
_v2_inline_selfcheck()


In [ ]:
def _master_inline_selfcheck():
    import tempfile
    rng = np.random.default_rng(782)
    x = rng.random((19,12)).astype(np.float32)
    idx = np.asarray([1,7,16]); extra = rng.random((3,12)).astype(np.float32)
    assert ma_flag('unknown') is None and ma_flag(np.nan) is None
    assert ma_flag('1') is True and ma_flag('0') is False
    assert np.array_equal(ma_mix_raw(x,extra,idx,0.),x)
    mixed = ma_mix_raw(x,extra,idx,.15)
    untouched = np.setdiff1d(np.arange(len(x)),idx)
    assert np.array_equal(mixed[untouched],x[untouched])
    tied = np.full((3,12),.5)
    assert np.array_equal(ma_rank(tied),np.full((3,12),2/3))
    assert np.array_equal(ma_outer_fuse(x,x)[:,3],ma_outer_fuse(1-x,x)[:,3])
    ids = ['s'+str(i) for i in range(len(x))]
    frame = pd.DataFrame(ma_rank(x),columns=MA_LABELS);frame.insert(0,'StudyInstanceUID',ids)
    assert ma_frame(frame.iloc[::-1],ids).StudyInstanceUID.tolist()==ids
    with tempfile.TemporaryDirectory() as directory:
        target = Path(directory)/'submission.csv'
        result = ma_publish(frame,ids,target,{'status':'SYNTHETIC_TEST'})
        assert result['submission_sha256']==ma_sha(target)
        assert json.loads(target.with_suffix('.receipt.json').read_text())['score_measured'] is None
    print('PASS: built-in selection/ranking/fusion/publication smoke checks (synthetic CPU data).')
_master_inline_selfcheck()


## 3. Artifact catalogue and offline dependency checks

In [ ]:
import os
for _thread_env in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(_thread_env, '4')
os.environ.setdefault('CUDNN_CONV_WSCAP_DBG', '1024')

def _d4_check_runtime(rt, artifact_root):
    import hashlib
    import json
    from pathlib import Path
    import torch
    root = Path(artifact_root)
    manifest_path = root / 'coatnet_pairfilm_manifest.json'
    digest = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
    if digest != '7ada0605bca6b0530569c6454e988ace479606a3328ed591d090e5764fea661d':
        raise RuntimeError('D4 reference manifest identity changed')
    expected = json.loads(manifest_path.read_text())
    manifest, paths = rt.validate_artifact_root(root, verify_hashes=True)
    if manifest != expected or len(paths) != 1 or rt.N_MODELS != 1:
        raise RuntimeError('D4 must load exactly the reference SWA model')
    if Path(paths[0]).name != 'coatnet_d4_parent_rank8_swa2.pt':
        raise RuntimeError('D4 must use its reference rank-8 parent checkpoint')
    if manifest['slice_depth']['adapter_sha256'] != 'b7c48b19997bc7ecb7e997f8dcf491ec5701ff917d1dac4b369eb85ac97ae89a':
        raise RuntimeError('D4 reference adapter identity changed')
    if rt.GPU_BATCH_STUDIES != 2 or rt.BACKBONE_MICRO_IMAGES != 8:
        raise RuntimeError('D4 reference batching changed')
    if rt.CUDNN_BENCHMARK is not False:
        raise RuntimeError('D4 reference convolution policy changed')
    if rt.base.cv2.__version__ != '4.12.0' or rt.parent.timm.__version__ != '1.0.22':
        raise RuntimeError('D4 reference dependency versions changed')
    if torch.cuda.device_count() < 1:
        raise RuntimeError('D4 requires at least one GPU')
    names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    return (manifest, paths)

def _d4_check_outputs(rt, manifest, receipt, output, competition):
    import hashlib
    from pathlib import Path
    import numpy as np
    import pandas as pd

    def need(condition, message):
        if not condition:
            raise RuntimeError('D4 release check: ' + message)
    need(receipt['status'] == rt.SUBMISSION_STATUS, 'unsuccessful status')
    expected_fields = {'models': 1, 'fallback_studies': 0, 'dicom_preparations_per_study': 1, 'model_passes_per_study': 1, 'rank_after_probability_average': True, 'canonical_sagittal': True, 'common_physical_triplet_warp': False, 'resolution': 384, 'slice_depth_arm': 'd4_zones', 'slice_depth_state_elements': 3255, 'slice_depth_parent_checkpoint_sha256': '5de38e333e3184d7b52fb068183f0cc198d6cbabb6ffbe36b26338622e82d297', 'fsx_attention_heads': 2, 'fsx_attention_width': 128, 'fsx_serving_delta_cap': 0.3, 'fsx_training_delta_cap': 0.5, 'head': 'rank8_swa2_plus_d4_three_zone_expanded_fsx', 'slice_depth_swa_member_epochs': [11, 9, 5], 'slice_depth_checkpoint_sha256': manifest['slice_depth']['adapter_sha256']}
    for key, value in expected_fields.items():
        need(receipt[key] == value, key)
    need(1 <= int(receipt['maximum_eval_windows']) <= 94, 'window limit')
    checkpoints = receipt['checkpoints']
    need(len(checkpoints) == 1, 'parent checkpoint count')
    need(checkpoints[0]['sha256'] == '5de38e333e3184d7b52fb068183f0cc198d6cbabb6ffbe36b26338622e82d297', 'parent checkpoint hash')
    need(checkpoints[0]['member_epochs'] == [15, 16], 'parent SWA member epochs')
    need(receipt['slice_depth_checkpoint_sha256'] == 'b7c48b19997bc7ecb7e997f8dcf491ec5701ff917d1dac4b369eb85ac97ae89a', 'adapter checkpoint hash')
    processes = receipt['processes']
    need(len(processes) >= 1 and all((int(p['returncode']) == 0 for p in processes)), 'worker process failure')
    shards = receipt['shards']
    need(len(shards) >= 1, 'shard count')
    for s in shards:
        need(s['device'] == 'cuda', 'non-CUDA shard')
        need(int(s['fallback_studies']) == 0 and int(s['preparation_warnings']) == 0, 'failed study/preparation')
        need(int(s['models_resident']) == 1, 'resident model count')
        need(int(s['peak_reserved_bytes']) < int(15.5 * 1024 ** 3), 'reference memory gate')
        need(s['specialized_worker_loader_contract'] == manifest['worker_loader_contract'], 'specialized loader')
    output = Path(output)
    prediction_path = output.parent / rt.PREDICTIONS_NAME
    with np.load(prediction_path, allow_pickle=False) as payload:
        uids = payload['study_uids'].astype(str).tolist()
        raw = np.asarray(payload['raw_probabilities'], dtype=np.float32)
        mean = np.asarray(payload['probability_mean'], dtype=np.float32)
        ranked = np.asarray(payload['submission_percentile_rank'], dtype=np.float64)
    ids = pd.read_csv(Path(competition) / 'test.csv', dtype={'StudyInstanceUID': str}).StudyInstanceUID.tolist()
    need(len(ids) > 0 and len(ids) == len(set(ids)), 'test identity')
    need(uids == ids and int(receipt['studies']) == len(ids), 'prediction identity/count')
    need(raw.shape == (1, len(ids), 12), 'raw probability shape')
    need(np.isfinite(raw).all() and ((raw >= 0) & (raw <= 1)).all(), 'raw probabilities')
    need(np.array_equal(raw.astype(np.float64).mean(0).astype(np.float32), mean), 'probability mean')
    need(np.array_equal(rt.percentile_rank64(mean), ranked), 'global percentile ranks')
    frame = pd.read_csv(output, dtype={'StudyInstanceUID': str}, float_precision='round_trip')
    need(frame.columns.tolist() == ['StudyInstanceUID', *rt.base.LABELS], 'CSV labels')
    need(frame.StudyInstanceUID.tolist() == ids, 'CSV identity')
    need(np.array_equal(frame.iloc[:, 1:].to_numpy(np.float64), ranked), 'CSV numeric round trip')
    actual_hash = hashlib.sha256(output.read_bytes()).hexdigest()
    need(actual_hash == receipt['output_sha256'], 'CSV hash')
    receipt['reference_wrapper_checks'] = {'export_sha256': '7f8410fdf1b60cd3831d50f517bb25798a4e4a3404c59dc269193d5fe7cce729', 'validated_manifest_sha256': '7ada0605bca6b0530569c6454e988ace479606a3328ed591d090e5764fea661d', 'probability_mean_exact': True, 'ranks_exact': True, 'csv_roundtrip_exact': True, 'source_and_model_hashes_validated': True, 'input_identity_equals_original_grid': False}
    return receipt

def _run_required_child(command, environment, logfile):
    remaining = v2_remaining_budget(MASTER_STARTED_AT, MASTER)
    report = v2_run_checked_child(command, environment, logfile, remaining)
    return int(report['returncode'])
ASSET_ROOTS = ['/kaggle/input/datasets/mattiaangeli/rsna-knee-coatnet-global96-top3', '/kaggle/input/rsna-knee-coatnet-global96-top3', '/kaggle/input/datasets/dreaddevelopment/raptor-knee-maxspan', '/kaggle/input/raptor-knee-maxspan', '/kaggle/input/datasets/dreaddevelopment/raptor-knee-native384', '/kaggle/input/raptor-knee-native384', '/kaggle/input/datasets/dreaddevelopment/raptor-knee-native384dense', '/kaggle/input/raptor-knee-native384dense', '/kaggle/input/datasets/mattiaangeli/knee-mri-fold-weights', '/kaggle/input/knee-mri-fold-weights', '/kaggle/input/datasets/mattiaangeli/opencv-python-headless-4120088-x86', '/kaggle/input/opencv-python-headless-4120088-x86', '/kaggle/input/datasets/marwanmath/resnet-50-radimagenet-marwan', '/kaggle/input/resnet-50-radimagenet-marwan', '/kaggle/input/datasets/mattiaangeli/rsna-knee-coat-resgated-ep10-top3', '/kaggle/input/rsna-knee-coat-resgated-ep10-top3', '/kaggle/input/datasets/antoinegg1/rsna-knee-e11-diverse-heads-v20', '/kaggle/input/rsna-knee-e11-diverse-heads-v20', '/kaggle/input/datasets/antoinegg1/rsna-knee-e9-radimagenet-heads-v15', '/kaggle/input/rsna-knee-e9-radimagenet-heads-v15', '/kaggle/input/datasets/pilkwang/rsna-knee-llm-labels', '/kaggle/input/rsna-knee-llm-labels', '/kaggle/input/datasets/prvsiyan/rsna-knee-v52-radimagenet-heads-20260812', '/kaggle/input/rsna-knee-v52-radimagenet-heads-20260812', '/kaggle/input/datasets/pilkwang/rsna-knee-weights', '/kaggle/input/rsna-knee-weights', '/kaggle/input/datasets/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2', '/kaggle/input/rsna-knee-coatnet-d4-depthzone-swa3-b2', '/kaggle/input/notebooks/sofiaanjenje/rsna-knee-e11-train', '/kaggle/input/rsna-knee-e11-train', '/kaggle/input/notebooks/sofiaanjenje/rsna-knee-e13-train', '/kaggle/input/rsna-knee-e13-train', '/kaggle/input/models/metaresearch/dinov2/pytorch/small/1', '/kaggle/input/dinov2/pytorch/small/1']

def _asset_walk():
    from pathlib import Path
    seen = set()
    for root in ASSET_ROOTS:
        root = Path(root).resolve()
        if not root.is_dir():
            continue
        level = [root]
        for _depth in range(6):
            following = []
            for directory in sorted(level):
                if str(directory) in seen:
                    continue
                seen.add(str(directory))
                children = sorted(directory.iterdir())
                dirs = [p for p in children if p.is_dir() and p.name not in ('train_series', 'test_series', 'train_images', 'test_images')]
                files = [p.name for p in children if p.is_file()]
                yield (str(directory), [p.name for p in dirs], files)
                following.extend(dirs)
            level = following

def _asset_find_asset(name, digest=None):
    import fnmatch, hashlib
    from pathlib import Path
    hits = []
    for root, _, files in _asset_walk():
        for file in files:
            if fnmatch.fnmatchcase(file, name):
                p = Path(root) / file
                if digest:
                    h = hashlib.sha256()
                    with p.open('rb') as handle:
                        for block in iter(lambda: handle.read(8 << 20), b''):
                            h.update(block)
                    if h.hexdigest() != digest:
                        continue
                if p.resolve() not in [h.resolve() for h in hits]:
                    hits.append(p)
    if len(hits) != 1:
        raise RuntimeError(f'expected one pinned asset {name}, got {hits}')
    return hits[0]

def _asset_half_rank_mix(top3, swa3):
    if top3.shape != swa3.shape or not top3.index.equals(swa3.index):
        raise ValueError('CoAt family alignment changed')
    if top3.isna().any().any() or swa3.isna().any().any():
        raise ValueError('missing CoAt family prediction')
    units = top3.rank(method='average') + swa3.rank(method='average')
    return units.rank(method='average', pct=True)

def _asset_write_coat_runtime(artifact, output, helper_source):
    from pathlib import Path
    import hashlib
    source = Path(artifact) / 'coatnet_resgated_ep10_top3_inference.py'
    text = source.read_text()
    if hashlib.sha256(source.read_bytes()).hexdigest() != 'b11e58f8d7cabe9e264ac70b01e811dc6a846aa01248ace1f0e6536d0d4094d9':
        raise RuntimeError('unexpected parent CoAt runtime')
    edits = {'if not 1 <= windows <= MAX_EVAL_WINDOWS:': 'if not 1 <= windows <= 94:', 'or int(counts.max()) > MAX_EVAL_WINDOWS:': 'or int(counts.max()) > 94:', '"runtime_contract": RUNTIME_CONTRACT,': '"runtime_contract": RUNTIME_CONTRACT, "input_max_windows": 94,', '"eval_grid": "saved_training_gold_unique_center_slot_budgets",': '"eval_grid": "input_unique_native_centers_six_slot_v1",'}
    for old, new in edits.items():
        if old not in text:
            raise RuntimeError('CoAt patch target missing: ' + old)
        text = text.replace(old, new)
    text = text.replace('"gold58_rank_ensemble_auc": manifest["selection"]', '"historical_parent_gold58_rank_ensemble_auc": manifest["selection"]')
    text = text.replace('"t4_gold58_rank_ensemble_auc": manifest["t4_numerical_portability"]', '"historical_parent_t4_gold58_rank_ensemble_auc": manifest["t4_numerical_portability"]')
    marker = 'if __name__ == "__main__":'
    if text.count(marker) != 1:
        raise RuntimeError('CoAt entrypoint changed')
    patch = helper_source + ('\n_original_train_faithful_eval_specs = train_faithful_eval_specs\n'
                             'def _tolerant_coat_specs(study):\n    try:\n        return _dense_coat_specs(study)\n'
                             '    except Exception as exc:\n        print("[dense-fallback] " + str(getattr(study, "study_uid", "?")) + ": " + type(exc).__name__ + ": " + str(exc), flush=True)\n'
                             '        return _original_train_faithful_eval_specs(study)\n'
                             'train_faithful_eval_specs = _tolerant_coat_specs\nRUNTIME_CONTRACT = "resgated_ep10_input_nativecenters_v1"\n')
    text = text.replace(marker, patch + '\n' + marker)
    compile(text, str(output), 'exec')
    Path(output).write_text(text)
    return hashlib.sha256(text.encode()).hexdigest()

def _asset_patch_d4(rt, grid_source):
    import ast
    import hashlib
    import inspect
    import json
    import os
    import textwrap
    from pathlib import Path
    audited = rt.parent.audited
    original = audited.prepare_global96_bag
    original_shard = rt._PARENT_INFER_SHARD
    if getattr(original, '_input_patch_installed', False):
        raise RuntimeError('D4 input preparation was already patched')
    src = textwrap.dedent(inspect.getsource(original))
    tree = ast.parse(src)
    fn = next((n for n in tree.body if isinstance(n, ast.FunctionDef)), None)
    if fn is None or fn.name != 'prepare_global96_bag':
        raise RuntimeError('D4 preparation function identity changed')
    if 'study_uid' not in inspect.signature(original).parameters:
        raise RuntimeError('D4 preparation study-UID contract changed')
    candidates = [n for n in ast.walk(fn) if isinstance(n, ast.Assign) and any((isinstance(t, ast.Name) and t.id == 'specs' for t in n.targets)) and isinstance(n.value, ast.Call) and isinstance(n.value.func, ast.Attribute) and isinstance(n.value.func.value, ast.Name) and (n.value.func.value.id == 'global_stack') and (n.value.func.attr == 'sample_global_windows')]
    if len(candidates) != 1 or candidates[0] not in fn.body:
        raise RuntimeError('D4 expected one top-level Global96 specs constructor')
    call = candidates[0]
    keywords = {k.arg: k.value for k in call.value.keywords}
    if not (isinstance(keywords.get('count'), ast.Constant) and keywords['count'].value is None and isinstance(keywords.get('train'), ast.Constant) and (keywords['train'].value is False)):
        raise RuntimeError('D4 original evaluation must use count=None, train=False')
    exec(compile(grid_source, '<input-grid>', 'exec'), audited.__dict__)
    audit_dir = Path(os.environ['RSNA_D4_INPUT_AUDIT_DIR'])
    audit_dir.mkdir(parents=True, exist_ok=True)

    def record_input(study_uid, stack, audit):
        import numpy as np
        arrays = ('global_indices', 'nominal_slots', 'canonical_depths', 'nominal_steps', 'source_series_rows', 'source_slot_ids')
        digest = hashlib.sha256()
        for name in arrays:
            value = np.ascontiguousarray(getattr(stack, name))
            digest.update(name.encode())
            digest.update(str(value.dtype).encode())
            digest.update(str(value.shape).encode())
            digest.update(value.tobytes())
        record = {'study_uid': str(study_uid), 'pid': os.getpid(), 'positions': int(len(stack.global_indices)), 'windows_expected': int(len(stack.global_indices) - 2), 'slot_widths': list(map(int, stack.slot_widths)), 'stack_sidecars_sha256': digest.hexdigest(), 'slots': audit}
        with (audit_dir / f'input_{os.getpid()}.jsonl').open('a') as handle:
            handle.write(json.dumps(record, sort_keys=True, allow_nan=False) + '\n')
    audited.__dict__['_record_input'] = record_input
    injection = ast.parse('try:\n    stack, input_audit = _dense_stack(stack, study.offsets, study.valid, study.source_depth)\n'
                          'except Exception as _dense_exc:\n    print("[dense-fallback] " + str(study_uid) + ": " + type(_dense_exc).__name__ + ": " + str(_dense_exc), flush=True)\n'
                          '    input_audit = [{"fallback": type(_dense_exc).__name__ + ": " + str(_dense_exc)}]\n'
                          '_record_input(study_uid, stack, input_audit)\n').body
    where = fn.body.index(call)
    fn.body[where:where] = injection
    ast.fix_missing_locations(tree)
    exec(compile(tree, '<D4-input-preparation-only>', 'exec'), audited.__dict__)
    patched = audited.prepare_global96_bag
    patched._input_patch_installed = True
    for module in (rt, rt.parent):
        if getattr(module, 'prepare_global96_bag', None) is original:
            module.prepare_global96_bag = patched
    if rt._PARENT_INFER_SHARD is not original_shard:
        raise RuntimeError('D4 parent shard must remain the official callable')
    files = {}
    for name, module in [('d4_runtime', rt), ('parent_runtime', rt.parent), ('audited_runtime', audited)]:
        path = Path(module.__file__)
        if path.is_file():
            files[name] = {'file': path.name, 'sha256': hashlib.sha256(path.read_bytes()).hexdigest()}
    receipt = {'contract': 'd4-original-runtime-input-preparation-only-v2', 'reference_script_version_id': 348764100, 'reference_export_available': True, 'original_prepare_source_sha256': hashlib.sha256(src.encode()).hexdigest(), 'patched_prepare_source_sha256': hashlib.sha256(ast.unparse(tree).encode()).hexdigest(), 'model_loader_modified': False, 'zone_head_modified': False, 'parent_infer_shard_replaced': False, 'grid_source_sha256': hashlib.sha256(grid_source.encode()).hexdigest(), 'artifact_source_files': files}
    (audit_dir / f'patch_{os.getpid()}.json').write_text(json.dumps(receipt, indent=2))
    return receipt

def _asset_run_d4(artifact, environment_dir, competition, output):
    import hashlib, inspect, json, os, sys
    from pathlib import Path
    import pandas as pd
    artifact, output = (Path(artifact), Path(output))
    manifest = artifact / 'coatnet_pairfilm_manifest.json'
    if hashlib.sha256(manifest.read_bytes()).hexdigest() != '7ada0605bca6b0530569c6454e988ace479606a3328ed591d090e5764fea661d':
        raise RuntimeError('D4 SWA3 manifest changed')
    grid = 'import numpy as np\nfrom raptor_global_stack_smart336 import FixedGlobalStack, INFERRED96_WIDTHS\n' + '\n'.join((inspect.getsource(f) for f in (_dense_quotas, _dense_stack)))
    audit_dir = output.parent / 'study_input_receipts'
    audit_dir.mkdir(parents=True, exist_ok=True)
    for stale in audit_dir.glob('*.json*'):
        stale.unlink()
    output.unlink(missing_ok=True)
    output.with_suffix('.receipt.json').unlink(missing_ok=True)
    wrapper = output.parent / 'd4_worker_entry.py'
    code = f'import sys\nsys.path.insert(0,{str(environment_dir)!r})\nsys.path.insert(0,{str(artifact)!r})\nfrom pathlib import Path\nimport json\nimport coatnet_d4_depthzone_swa_inference as rt\nimport os\n' + inspect.getsource(_asset_patch_d4) + '\n' + inspect.getsource(_d4_check_runtime) + '\n' + inspect.getsource(_d4_check_outputs) + '\n' + f"if __name__ == '__main__' and '--worker-output' not in sys.argv:\n    _checked_manifest, _checked_paths = _d4_check_runtime(rt, Path({str(artifact)!r}))\n" + f'_asset_patch_d4(rt,{grid!r})\n' + 'rt.parent.audited.__file__ = __file__\n' + "if __name__ == '__main__':\n" + "    if '--worker-output' in sys.argv:\n        raise SystemExit(rt.main())\n" + '    manifest, paths = _checked_manifest, _checked_paths\n' + '    r = rt.run_submission(\n' + f'        competition_root=Path({str(competition)!r}), artifact_root=Path({str(artifact)!r}),\n' + f'        output_path=Path({str(output)!r}), gpu_batch_studies=2, backbone_micro_images=8)\n' + f'    r = _d4_check_outputs(rt, manifest, r, Path({str(output)!r}), Path({str(competition)!r}))\n' + f"    Path({str(output.with_suffix('.receipt.json'))!r}).write_text(json.dumps(r,indent=2,allow_nan=False))\n"
    compile(code, str(wrapper), 'exec')
    wrapper.write_text(code)
    env = dict(os.environ)
    env['RSNA_D4_INPUT_AUDIT_DIR'] = str(audit_dir)
    env['PYTHONPATH'] = f'{environment_dir}:/kaggle/working/_coat_env:{artifact}:' + env.get('PYTHONPATH', '')
    env.pop('CUDNN_CONV_WSCAP_DBG', None)
    _run_required_child([sys.executable, str(wrapper)], env, output.with_suffix('.log'))
    receipt = json.loads(output.with_suffix('.receipt.json').read_text())
    expected = pd.read_csv(Path(competition) / 'test.csv', dtype={'StudyInstanceUID': str}).StudyInstanceUID.tolist()
    records = []
    for path in sorted(audit_dir.glob('input_*.jsonl')):
        records.extend((json.loads(line) for line in path.read_text().splitlines() if line.strip()))
    ids = [r['study_uid'] for r in records]
    if len(ids) != len(expected) or len(ids) != len(set(ids)) or set(ids) != set(expected):
        raise RuntimeError('D4 input hook must execute exactly once for every test study')
    if not all((1 <= r['windows_expected'] <= 94 and r['positions'] == r['windows_expected'] + 2 for r in records)):
        raise RuntimeError('D4 input record cardinality drift')
    receipt['input_coverage'] = {'studies': len(records), 'folder': str(audit_dir), 'min_windows': min((r['windows_expected'] for r in records)), 'max_windows': max((r['windows_expected'] for r in records)), 'original_model_loader_retained': True, 'original_shard_scheduler_retained': True}
    output.with_suffix('.receipt.json').write_text(json.dumps(receipt, indent=2, allow_nan=False))
    return receipt
_speed_original_asset_walk = _asset_walk
_speed_original_find_asset = _asset_find_asset
'Catalogue named immutable attachments once; no competition-tree search.'
import threading as _speed_threading
_speed_asset_lock = _speed_threading.RLock()
_speed_asset_catalogue = None
_speed_asset_hits = {}

def _asset_walk():
    global _speed_asset_catalogue
    with _speed_asset_lock:
        if _speed_asset_catalogue is None:
            _speed_asset_catalogue = tuple(((root, tuple(dirs), tuple(files)) for root, dirs, files in _speed_original_asset_walk()))
        catalogue = _speed_asset_catalogue
    for root, dirs, files in catalogue:
        yield (root, list(dirs), list(files))

def _asset_find_asset(name, digest=None):
    key = (str(name), digest)
    with _speed_asset_lock:
        if key not in _speed_asset_hits:
            _speed_asset_hits[key] = _speed_original_find_asset(name, digest)
        return _speed_asset_hits[key]


In [ ]:
# Resolve identical immutable copies without ambiguous-file false failures.
# Deliberately search only named model/helper mounts, never MRI folders.
_SINGLE_ASSET_RESOLVER=s2.AssetResolver(ASSET_ROOTS)
def _asset_find_asset(name,digest=None):
    return _SINGLE_ASSET_RESOLVER.find(name,digest)


In [ ]:
import numpy as np

def _dense_allocate(capacities, proportions, target):
    capacities = np.asarray(capacities, dtype=np.int64)
    proportions = np.asarray(proportions, dtype=np.float64)
    if capacities.ndim != 1 or capacities.shape != proportions.shape or np.any(capacities < 0) or (not np.isfinite(proportions).all()) or np.any(proportions <= 0) or (int(target) < 1):
        raise ValueError('invalid capacity-aware sampling allocation')
    ideal = int(target) * proportions / proportions.sum()
    initial = np.floor(ideal).astype(np.int64)
    for i in np.argsort(-(ideal - initial), kind='stable')[:int(target) - int(initial.sum())]:
        initial[i] += 1
    quotas = np.minimum(initial, capacities)
    wanted = min(int(target), int(capacities.sum()))
    deficit = np.zeros(len(quotas), dtype=np.float64)
    while int(quotas.sum()) < wanted:
        donors = quotas < capacities
        weights = np.where(donors, proportions, 0.0)
        deficit[donors] += weights[donors] / weights.sum()
        chosen = int(np.argmax(np.where(donors, deficit, -np.inf)))
        quotas[chosen] += 1
        deficit[chosen] -= 1
    assert int(quotas.sum()) == wanted and np.all(quotas <= capacities)
    return quotas

def _dense_unique_linspace(capacity, count):
    if not 0 <= count <= capacity:
        raise ValueError('cannot create unique picks beyond physical capacity')
    result = np.linspace(0, capacity - 1, count).round().astype(np.int64)
    if count and (result.min() < 0 or result.max() >= capacity):
        raise AssertionError('out-of-range source')
    assert len(np.unique(result)) == count
    return result

def _dense_coat_specs(study):
    import raptor_light224 as sampler
    capacities = []
    for row in study.series_rows:
        if int(row) < 0:
            capacities.append(0)
        else:
            centers, _, _, _ = sampler._candidate_centers(int(row), study.offsets, study.valid, study.source_depth)
            capacities.append(len(centers))
    if sum(capacities) < 1:
        raise ValueError('no usable CoAt triplets')
    quotas = _dense_allocate(capacities, (20, 12, 12, 8, 16, 8), 94)
    specs = sampler.sample_eval_windows(study.series_rows, study.offsets, study.valid, study.source_depth, count=None, slot_budgets=np.maximum(quotas, 1).tolist())
    if len(specs) != min(94, sum(capacities)):
        raise RuntimeError('capacity-aware sampling window-count drift')
    keys = [(s.series_row, s.center_local) for s in specs]
    if len(keys) != len(set(keys)):
        raise RuntimeError('capacity-aware sampling repeated a native center')
    for s in specs:
        start = int(study.offsets[s.series_row])
        if not np.all(study.valid[start + np.asarray(s.local_indices)]):
            raise RuntimeError('capacity-aware sampling invalid source channel')
    return specs
INFERRED96_WIDTHS = (27, 21, 18, 12, 18)

def _dense_quotas(capacities) -> np.ndarray:
    capacities = np.asarray(capacities, dtype=np.int64)
    base = np.asarray(INFERRED96_WIDTHS, dtype=np.int64)
    if capacities.shape != (5,) or bool((capacities < 0).any()):
        raise ValueError('expected five nonnegative source capacities')
    quotas = np.minimum(base, capacities)
    target = min(96, int(capacities.sum()))
    deficit = np.zeros(5, dtype=np.float64)
    while int(quotas.sum()) < target:
        donors = quotas < capacities
        weights = np.where(donors, base, 0).astype(np.float64)
        deficit[donors] += weights[donors] / weights.sum()
        selected = int(np.argmax(np.where(donors, deficit, -np.inf)))
        quotas[selected] += 1
        deficit[selected] -= 1
    if int(quotas.sum()) != target or bool((quotas > capacities).any()):
        raise RuntimeError('additional quota allocation changed')
    return quotas

def _dense_stack(reference, offsets, valid, source_depth):
    from types import SimpleNamespace
    from raptor_global_stack_smart336 import FixedGlobalStack
    if tuple(reference.slot_widths) != tuple(INFERRED96_WIDTHS):
        raise ValueError('capacity-aware sampling is pinned to the Global96 reference layout')
    pools = []
    audit = []
    begin = 0
    for slot, width in enumerate(reference.slot_widths, 1):
        end = begin + width
        old = reference.global_indices[begin:end]
        row = int(reference.source_series_rows[begin])
        info = {'slot': slot, 'original_width': width, 'source_row': row}
        audit.append(info)
        if row < 0 or bool((old < 0).all()):
            pools.append((np.empty(0, dtype=np.int64), np.empty(0, dtype=np.float32)))
            begin = end
            continue
        start, stop = map(int, offsets[row:row + 2])
        if not 0 <= start < stop <= len(valid) == len(source_depth):
            raise ValueError('source offset drift')
        depth = np.asarray(source_depth[start:stop], dtype=np.float32)
        keep = np.asarray(valid[start:stop], dtype=bool) & np.isfinite(depth) & (depth >= 0.02) & (depth <= 0.98)
        eligible = np.flatnonzero(keep).astype(np.int64) + start
        reverse = bool(old[0] > old[-1] or (old[0] == old[-1] and np.float32(source_depth[old[0]]) != reference.canonical_depths[begin]))
        if reverse:
            eligible = eligible[::-1].copy()
        depth = np.asarray(source_depth[eligible], dtype=np.float32)
        if reverse:
            depth = (1.0 - depth).astype(np.float32)
        pools.append((eligible, depth))
        begin = end
    quotas = _dense_quotas([len(pool[0]) for pool in pools])
    if int(quotas.sum()) < 3:
        raise ValueError('study has fewer than three usable unique source slices')
    parts = {key: [] for key in ('global_indices', 'nominal_slots', 'canonical_depths', 'nominal_steps', 'source_series_rows', 'source_slot_ids')}
    for slot, ((eligible, depth), quota, info) in enumerate(zip(pools, quotas, audit), 1):
        quota = int(quota)
        info.update(eligible=len(eligible), allocated_width=quota, additional=max(0, quota - info['original_width']))
        info['reason'] = 'missing_no_padding' if not quota else 'short_slot_all_unique' if len(eligible) < info['original_width'] else 'uniform_native_centers' if info['additional'] else 'uniform_native_centers'
        positions = np.linspace(0, len(eligible) - 1, quota).round().astype(np.int64)
        if len(positions) != quota or not bool((np.diff(positions) > 0).all()):
            raise RuntimeError('capacity-aware sampling slot selection repeated a source')
        mask = reference.nominal_slots == slot
        source_slot = int(reference.source_slot_ids[mask][0])
        parts['global_indices'].append(eligible[positions])
        parts['canonical_depths'].append(depth[positions])
        parts['nominal_slots'].append(np.full(quota, slot, dtype=np.int8))
        parts['nominal_steps'].append(np.full(quota, 0.96 / max(quota - 1, 1), dtype=np.float32))
        parts['source_series_rows'].append(np.full(quota, info['source_row'], dtype=np.int32))
        parts['source_slot_ids'].append(np.full(quota, source_slot, dtype=np.int8))
    result = FixedGlobalStack(**{key: np.concatenate(value) for key, value in parts.items()}, slot_widths=tuple((int(value) for value in quotas)))
    if len(np.unique(result.global_indices)) != len(result.global_indices) or bool((result.global_indices < 0).any()):
        raise RuntimeError('capacity-aware sampling output contains a repeated/missing source')
    return (result, audit)


In [ ]:
# No network installation. Missing libraries must come from attached compatible wheels.
import importlib.util, subprocess, sys
from packaging.tags import sys_tags
from packaging.utils import parse_wheel_filename
_supported_tags = set(sys_tags())
_wheel_roots = [Path(p) for p in ASSET_ROOTS if Path(p).is_dir()]
if os.environ.get('RSNA_WHEELHOUSE'):
    _wheel_roots.append(Path(os.environ['RSNA_WHEELHOUSE']))
# Limit discovery to known asset directories, not the DICOM tree.
_wheel_dirs = list(dict.fromkeys([p for r in _wheel_roots for p in (r,r/'wheels',r/'packages') if p.is_dir()]))
def _master_offline_import(module, distribution):
    if importlib.util.find_spec(module) is not None: return
    candidates = []
    for root in _wheel_dirs:
        for path in root.glob(distribution.replace('-','_')+'-*.whl'):
            try:
                name, version, build, tags = parse_wheel_filename(path.name)
                if _supported_tags.intersection(tags): candidates.append((version,str(path)))
            except ValueError: pass
    if not candidates:
        raise RuntimeError(f'Missing {module}. Attach a compatible {distribution} wheel under RSNA_WHEELHOUSE; internet stays disabled.')
    selected = sorted(candidates)[-1][1]
    command=[sys.executable,'-m','pip','install','--no-index','--no-deps',selected]
    print('Offline wheel:',Path(selected).name)
    subprocess.run(command,check=True)
    importlib.invalidate_caches()
    if importlib.util.find_spec(module) is None: raise RuntimeError('Offline import still unavailable: '+module)
for _module,_dist in [('pydicom','pydicom'),('timm','timm'),('transformers','transformers'),('cv2','opencv-python-headless')]:
    _master_offline_import(_module,_dist)


## 4. GPU preflight, test-cohort checks, and runtime diagnostics

In [ ]:
from __future__ import annotations
import time as _startup_time
_STARTUP_T0 = _startup_time.perf_counter()
import os
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(_v, '4')
import gc
import hashlib
import json
import re
import time
import traceback
import threading
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F

def _cuda_execution_probe(index):
    dev = torch.device(f'cuda:{index}')
    try:
        major, minor = torch.cuda.get_device_capability(index)
        probe = nn.Conv2d(3, 4, kernel_size=3, padding=1).eval().to(dev)
        with torch.inference_mode():
            out = probe(torch.zeros((1, 3, 16, 16), device=dev))
            if tuple(out.shape) != (1, 4, 16, 16):
                raise RuntimeError(f'unexpected CUDA probe shape {tuple(out.shape)}')
        torch.cuda.synchronize(index)
        print(f'cuda:{index} probe PASS (compute {major}.{minor})')
        del probe, out
        torch.cuda.empty_cache()
        return True
    except Exception as exc:
        import sys; sys.stderr.write('CUDA PROBE FAIL cuda:' + str(index) + ': ' + repr(exc) + '\n'); sys.stderr.flush()
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
        return False
DEVS = []
if torch.cuda.is_available():
    DEVS = [torch.device(f'cuda:{i}') for i in range(torch.cuda.device_count()) if _cuda_execution_probe(i)]
import sys; sys.stderr.write('DEBUG: cuda_avail=' + str(torch.cuda.is_available()) + ', count=' + str(torch.cuda.device_count()) + '\n'); sys.stderr.flush()
if len(DEVS) != 2 or torch.cuda.device_count() != 2:
    raise RuntimeError('Select Kaggle GPU T4 x2: two working CUDA devices are required')
if not all('T4' in torch.cuda.get_device_name(i).upper() for i in range(2)):
    raise RuntimeError('This build preserves the T4 x2 checkpoint/runtime contract; use actual T4 GPUs')
print(f'devices: {[str(d) for d in DEVS]}')
_STARTUP_GPU_IMPORT_S = _startup_time.perf_counter() - _STARTUP_T0
print(f'[startup] imports+GPU preflight: {_STARTUP_GPU_IMPORT_S:.2f}s', flush=True)
T0 = MASTER_STARTED_AT
SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)
TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
CROP_MM = 130.0
CACHE_IMG = 336
GROUP = 3
N_GROUP_MAX = 1
CACHE_FRACTION = 0.45
CACHE_BUDGET_MAX_GB = 24.0
CACHE_BUDGET_GB = 12.0
TEST_SHARE = 0.3
HDR_THREADS = 16
PIX_THREADS = 12
ORDER_THREADS = 32
ORDER_BUDGET_S = 5400
RUNS = [{'name': 'r224', 'img': 224}, {'name': 'r336', 'img': 336}]
EPOCHS = 10
BATCH_STUDIES = 8
AUG_ROT_DEG = 8.0
AUG_SCALE = 0.08
AUG_SHIFT = 0.05
AUG_INTENSITY = 0.1
LAT_MIN_OFFSET_MM = 20.0
SLICE_BAND = (0.2, 0.8)
RULES_NATIVE = {'order': 'normal', 'lat': 'centre', 'slot_fallback': False, 'decode_fill': 'nearest'}
RULES_LEGACY = {'order': 'dominant_axis', 'lat': 'corner_x', 'slot_fallback': True, 'decode_fill': 'zero'}
RULES = dict(RULES_NATIVE)
LEGACY_LAT_OFFSET_MM = 5.0
LR_HEAD = 0.001
LR_BACKBONE = 8e-06
UNFREEZE_LAST = 6
WEIGHT_DECAY = 0.02
EVAL_BATCH = 8
TIME_BUDGET = MASTER['time_budget_hours'] * 3600
SLOTS_RECOVERED = [('SAG_FLUID_FS', 'Sagittal', True, True), ('COR_FLUID_FS', 'Coronal', True, True), ('AX_FLUID_FS', 'Axial', True, True), ('SAG_FLUID_NOFS', 'Sagittal', True, False), ('COR_T1', 'Coronal', False, False), ('SAG_T1', 'Sagittal', False, False)]
SLOTS_PUBLIC = [('SAG_FLUID', 'Sagittal', None, True), ('COR_FLUID', 'Coronal', None, True), ('AX_FLUID', 'Axial', None, True), ('SAG_STRUCT', 'Sagittal', None, False), ('COR_STRUCT', 'Coronal', None, False), ('AX_STRUCT', 'Axial', None, False)]
SLOT_SCHEME = os.environ.get('SLOT_SCHEME', 'recovered')
SLOTS = SLOTS_PUBLIC if SLOT_SCHEME == 'public' else SLOTS_RECOVERED
N_SLOT = len(SLOTS)
POOL_PARTS = {'cls_mean': 2, 'cls_mean_focal': 3}
SLOT_PRIOR_TABLE = {'ACL': (0, 3, 5), 'MCL': (1, 4), 'Medial Meniscus': (0, 1, 3, 4), 'Lateral Meniscus': (0, 1, 3, 4), 'Medial OA': (1, 4, 5), 'Lateral OA': (1, 4, 5), 'PF OA': (0, 2, 5), 'Effusion': (0, 2), 'Synovitis': (0, 2), "Baker's": (0,), 'Contusion': (0, 1, 2), 'Fracture': (0, 1, 2, 4, 5)}
SLOT_PRIOR_STRENGTH = 0.55
FATSAT_OPTS = {'FS', 'FATSAT', 'FAT_SAT', 'FSAT'}
_SEP = re.compile('[_\\-.]')
_FATSAT_RX = re.compile('\\bfs\\b|fatsat|fat sat|\\bstir\\b|\\bspair\\b|\\bspir\\b|\\bwe\\b|water excit|\\btirm\\b|\\bsting\\b|\\bfatsup\\b|smart fat|\\bwater\\b')  # 2026-09-17: GE Dixon water images ('SMART FAT', 'Water: SMART FAT', seq FSEfw) are fat-suppressed; 37/4407 training studies had no fat-sat series detected before
_T1_RX = re.compile('\\bt1\\b|\\bt1w\\b')
_T2_RX = re.compile('\\bt2\\b|\\bt2w\\b')
_PD_RX = re.compile('\\bpd\\b|\\bpdw\\b|proton|\\bdp\\b|dens')

# Runtime integrity: no partial ensemble is ever published as submission.csv.
import os, json, hashlib, time, threading, tempfile
from pathlib import Path
import numpy as np
import pandas as pd

_RSNA_AUDIT_LOCK = threading.RLock()
_RSNA_AUDIT = {'contract': 'btkd_speedy_v558_input_uniform', 'events': [], 'phases': {}}
_RSNA_ORDER_MEMO = {}
_RSNA_HEADERS_MEMO = {}
_RSNA_CACHE_FILES = []
_RSNA_TEST_IDS = None
_RSNA_LABELS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA','PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']

def rsna_sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda: f.read(8 << 20), b''):
            h.update(b)
    return h.hexdigest()

def rsna_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name('.' + path.name + '.tmp')
    with tmp.open('w') as f:
        json.dump(value, f, indent=2, sort_keys=True, allow_nan=False, default=str)
        f.write('\n'); f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)

def rsna_event(kind, **details):
    with _RSNA_AUDIT_LOCK:
        _RSNA_AUDIT['events'].append({'kind':kind, **details})
    if kind in MASTER_FATAL_EVENTS:
        raise RuntimeError('Required ensemble component failed: '+kind+'; '+str(details)[:700])

def rsna_finite(values, tag, probability=False):
    x = np.asarray(values)
    if not np.issubdtype(x.dtype, np.number) or not np.isfinite(x).all():
        raise RuntimeError(f'{tag}: nonfinite/non-numeric values BEFORE ranking')
    if probability and x.size and (x.min() < 0 or x.max() > 1):
        raise RuntimeError(f'{tag}: probability outside [0,1]')
    return x

def rsna_rank01(values):
    """Parent endpoint scale; tied values share their average rank."""
    x = np.asarray(rsna_finite(values, 'rank01'), dtype=np.float64)
    if x.ndim != 2 or not len(x):
        raise ValueError('rank01 requires a nonempty [study,finding] matrix')
    if len(x) == 1:
        return np.full_like(x, .5)
    return (pd.DataFrame(x).rank(method='average').to_numpy() - 1) / (len(x)-1)

def rsna_rankpct(values):
    x = np.asarray(rsna_finite(values, 'rankpct'), dtype=np.float64)
    if x.ndim != 2 or not len(x):
        raise ValueError('rankpct requires a nonempty matrix')
    return pd.DataFrame(x).rank(method='average', pct=True).to_numpy(np.float64)

def rsna_frame(frame, ids, labels, tag):
    ids = [str(u) for u in ids]; labels = list(labels)
    if len(ids) != len(set(ids)):
        raise RuntimeError(f'{tag}: duplicate expected UID')
    if frame.columns.tolist() != ['StudyInstanceUID', *labels]:
        raise RuntimeError(f'{tag}: label order/schema mismatch')
    frame = frame.copy(); frame['StudyInstanceUID'] = frame['StudyInstanceUID'].astype(str)
    if frame.StudyInstanceUID.duplicated().any() or set(frame.StudyInstanceUID) != set(ids):
        raise RuntimeError(f'{tag}: missing/extra/duplicate UID')
    frame = frame.set_index('StudyInstanceUID').loc[ids].reset_index()
    rsna_finite(frame[labels].to_numpy(), tag, probability=True)
    return frame

def rsna_save_predictions(tag, ids, values, labels=None):
    x = rsna_finite(values, tag)
    target = Path('/kaggle/working/diagnostics'); target.mkdir(parents=True, exist_ok=True)
    path = target/(tag+'.npz')
    np.savez_compressed(path, study_uids=np.asarray(ids,dtype=str), labels=np.asarray(labels or _RSNA_LABELS,dtype=str), values=x)
    with _RSNA_AUDIT_LOCK:
        _RSNA_AUDIT['phases'][tag]={'path':str(path),'sha256':rsna_sha(path),'shape':list(x.shape)}

def rsna_strict_load(model, state, tag):
    """A random frozen encoder is never an acceptable missing-state fallback."""
    expected = model.state_dict()
    missing = sorted(set(expected)-set(state)); extra = sorted(set(state)-set(expected))
    bad_shape = [k for k in set(expected)&set(state) if tuple(expected[k].shape)!=tuple(state[k].shape)]
    if missing or extra or bad_shape:
        raise RuntimeError(f'{tag}: incomplete/mismatched checkpoint; missing={missing[:20]}, extra={extra[:20]}, shapes={bad_shape[:20]}. A partial encoder needs its exact pinned source weights; random initialization is forbidden.')
    for k,t in state.items():
        if t.is_floating_point() and not bool(t.isfinite().all()):
            raise RuntimeError(f'{tag}: nonfinite checkpoint tensor {k}')
    return model.load_state_dict(state, strict=True)

def rsna_deadline(tag):
    if time.time() - T0 > TIME_BUDGET:
        raise TimeoutError(f'{tag}: full-run budget exceeded; no incomplete output may be submitted')

def rsna_array(shape, tag):
    """Large temporary pixels belong in scratch space, not saved notebook outputs."""
    size = int(np.prod(shape))
    if size <= float(os.environ.get('RSNA_CACHE_RAM_GIB', '1')) * (1 << 30):
        return np.zeros(shape, np.uint8)
    directory = Path(os.environ.get('RSNA_PIXEL_SCRATCH', '/kaggle/temp/rsna_pixels_v559'))
    try:
        directory.mkdir(parents=True, exist_ok=True)
    except OSError as scratch_error:  # /kaggle/temp may be unavailable; check /tmp capacity explicitly
        rsna_event('scratch_fallback', tag=tag, requested=str(directory), error=f'{type(scratch_error).__name__}: {scratch_error}')
        directory = Path('/tmp/rsna_pixels_v559')
        directory.mkdir(parents=True, exist_ok=True)
    import shutil
    free = shutil.disk_usage(directory).free
    rsna_event('scratch_allocation', tag=tag, bytes=size, directory=str(directory), free_bytes=int(free))
    if free < size + (1 << 30) and str(directory) != '/tmp/rsna_pixels_v559':
        rsna_event('scratch_fallback', tag=tag, requested=str(directory), error=f'insufficient free space ({free} bytes)')
        directory = Path('/tmp/rsna_pixels_v559'); directory.mkdir(parents=True, exist_ok=True); free = shutil.disk_usage(directory).free
    if free < size + (1 << 30):
        raise RuntimeError(f'{tag}: not enough scratch disk for complete cache ({size} bytes; {free} free)')
    fd, path = tempfile.mkstemp(prefix='pixels_', suffix='.npy', dir=directory)
    os.close(fd)
    try:
        a = np.lib.format.open_memmap(path, mode='w+', dtype=np.uint8, shape=tuple(shape))
        a[:] = 0
    except BaseException:
        Path(path).unlink(missing_ok=True)
        raise
    _RSNA_CACHE_FILES.append(path)
    return a

def rsna_release_pixels(array):
    if isinstance(array,np.memmap):
        path=str(array.filename)
        array.flush()
        # POSIX unlink releases the file after the last view is closed.
        Path(path).unlink(missing_ok=True)
        if path in _RSNA_CACHE_FILES: _RSNA_CACHE_FILES.remove(path)

def rsna_initialize(ids):
    global _RSNA_TEST_IDS
    _RSNA_TEST_IDS=[str(u) for u in ids]
    if not _RSNA_TEST_IDS or len(set(_RSNA_TEST_IDS))!=len(_RSNA_TEST_IDS):
        raise RuntimeError('empty or duplicate competition study IDs')
    work=Path('/kaggle/working'); work.mkdir(parents=True,exist_ok=True)
    os.chdir(work)
    for f in ['submission.csv','_pipeline_stage.csv','btkd_v559_complete.json','btkd_v559_complete.json']:
        (work/f).unlink(missing_ok=True)
    (work/'diagnostics').mkdir(exist_ok=True)
    _RSNA_AUDIT['cohort']=len(_RSNA_TEST_IDS)
    _RSNA_AUDIT['environment']={'python':__import__('sys').version,'torch':str(torch.__version__),'cuda':torch.version.cuda,'gpu_names':[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}
    rsna_json(work/'diagnostics/runtime_started.json',_RSNA_AUDIT)

def rsna_asset_preflight():
    """Locate required additions before spending time on the public parent."""
    names=['raptor_ft_coatnet_v5_full_swa.pt','raptor_ft_coatnet_v10_full.pt','raptor_ft_coatnet_v8_full_swa.pt']
    receipt={name:str(_asset_find_asset(name)) for name in names}
    for name,digest in [
        ('coat_resgated_ep10_top3_manifest.json','98511a8fdeb9da0e6e70c78d013ff636e1476f31c80b5dc134d294b18c3f284e'),
        ('coatnet_pairfilm_manifest.json','7ada0605bca6b0530569c6454e988ace479606a3328ed591d090e5764fea661d'),
        ('opencv_python_headless-4.12.0.88-*.whl','236c8df54a90f4d02076e6f9c1cc763d794542e886c576a6fee46ec8ff75a7a9')]:
        receipt[name]={'path':str(_asset_find_asset(name,digest)),'sha256':digest}
    roots=[Path(p) for p in ['/kaggle/input/datasets/mattiaangeli/knee-mri-fold-weights','/kaggle/input/knee-mri-fold-weights'] if (Path(p)/'m_f0.pt').is_file()]
    roots=list({str(p.resolve()):p for p in roots}.values())
    if len(roots)!=1:raise RuntimeError(f'A5: expected one complete attached root, found {roots}')
    missing=[str(roots[0]/f'm_f{i}.pt') for i in range(5) if not (roots[0]/f'm_f{i}.pt').is_file()]
    if missing:raise FileNotFoundError(f'A5 missing folds: {missing}')
    receipt['A5_root']=str(roots[0]);receipt['tensor_loading_and_fingerprints']='validated by mandatory component loaders before predictions'
    d4_manifest_path = _asset_find_asset('coatnet_pairfilm_manifest.json',
        '7ada0605bca6b0530569c6454e988ace479606a3328ed591d090e5764fea661d')
    timm_wheel = d4_manifest_path.parent / 'timm-1.0.22-py3-none-any.whl'
    if not timm_wheel.is_file() or rsna_sha(timm_wheel) != '888981753e65cbaacfc07494370138b1700a27b1f0af587f4f9b47bc024161d0':
        raise RuntimeError('D4 pinned timm wheel missing or changed')
    receipt['D4_timm_wheel_sha256'] = rsna_sha(timm_wheel)
    rsna_json('/kaggle/working/diagnostics/asset_preflight.json',receipt)


def rsna_phase(stage, status='START', **extra):
    """Local phase evidence; this does not expose Kaggle's hidden rerun logs."""
    import json, os, resource, time
    from pathlib import Path
    work = Path('/kaggle/working/diagnostics')
    work.mkdir(parents=True, exist_ok=True)
    current = {'stage': stage, 'status': status,
               'elapsed_seconds': time.time()-T0,
               'max_rss_kib': int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss), **extra}
    for path in ['/sys/fs/cgroup/memory.current', '/sys/fs/cgroup/memory.max']:
        try: current[path.rsplit('/', 1)[-1]] = Path(path).read_text().strip()
        except OSError: pass
    if torch.cuda.is_initialized():
        current['gpus'] = [{'id': i, 'allocated_bytes': torch.cuda.memory_allocated(i),
                            'reserved_bytes': torch.cuda.memory_reserved(i)}
                           for i in range(torch.cuda.device_count())]
    print('[phase] '+json.dumps(current, sort_keys=True), flush=True)
    rsna_json(work/'current_phase.json', current)
    with (work/'phase_events.jsonl').open('a') as f:
        f.write(json.dumps(current, sort_keys=True)+'\n')
    return current

# Record uncaught cell errors locally; Kaggle may still withhold hidden-run logs.
def _rsna_capture_cell_error(result):
    error = getattr(result, 'error_in_exec', None) or getattr(result, 'error_before_exec', None)
    if error is None:
        return
    try:
        path = Path('/kaggle/working/diagnostics/current_phase.json')
        previous = json.loads(path.read_text()) if path.is_file() else {}
        rsna_phase(previous.get('stage', 'unknown'), 'FAILED', error_type=type(error).__name__, error=str(error))
    except Exception as logging_error:
        print('[diagnostic-write-failed]', type(logging_error).__name__, flush=True)
try:
    _rsna_ip = get_ipython()
    if _rsna_ip is not None:
        _rsna_ip.events.register('post_run_cell', _rsna_capture_cell_error)
except NameError:
    pass


In [ ]:
def log(msg):
    print(f'[{time.time() - T0:7.1f}s] {msg}', flush=True)

def _one_direct(tag, candidates, required):
    matches = []
    for candidate in map(Path, candidates):
        if candidate.is_dir() and all((candidate / item).exists() for item in required):
            matches.append(candidate)
    unique = []
    for match in matches:
        if str(match.resolve()) not in {str(item.resolve()) for item in unique}:
            unique.append(match)
    if len(unique) != 1:
        raise FileNotFoundError(
            f'{tag}: expected exactly one direct mounted artifact, found '
            f'{[str(item) for item in unique]}; checked {[str(Path(item)) for item in candidates]}'
        )
    return unique[0]

def find_root():
    return _one_direct('competition', [
        os.environ.get('RSNA_COMP_ROOT', '/kaggle/input/competitions/rsna-knee-abnormality-detection'),
        '/kaggle/input/rsna-knee-abnormality-detection',
    ], ['test.csv', 'test_series.csv', 'test_series'])

def find_dinov2(variant='small'):
    if variant != 'small':
        raise ValueError(f'unpinned DINOv2 variant: {variant}')
    return _one_direct('DINOv2-S model', [
        '/kaggle/input/models/metaresearch/dinov2/PyTorch/small/1',
        '/kaggle/input/models/metaresearch/dinov2/pytorch/small/1',
        '/kaggle/input/dinov2/PyTorch/small/1',
        '/kaggle/input/dinov2/pytorch/small/1',
        '/kaggle/input/dinov2-small',
        '/kaggle/input/models/metaresearch/dinov2/PyTorch/small/1',
    ], ['config.json', 'pytorch_model.bin'])
ROOT = find_root()
DINOV2_SOURCE = find_dinov2('small')
log(f'input root: {ROOT}')
_STARTUP_PATH_S = _startup_time.perf_counter() - _STARTUP_T0 - _STARTUP_GPU_IMPORT_S
print(f'[startup] competition-root preflight: {_STARTUP_PATH_S:.2f}s', flush=True)
IMG = CACHE_IMG

def available_gb():
    try:
        with open('/proc/meminfo') as fh:
            info = {k.strip(): v for k, v in (l.split(':', 1) for l in fh if ':' in l)}
        return int(info['MemAvailable'].split()[0]) / 1024 ** 2
    except Exception:
        return CACHE_BUDGET_GB / CACHE_FRACTION

def plan_cache(n_study, n_test=0):
    avail = available_gb()
    budget = min(avail * CACHE_FRACTION, CACHE_BUDGET_MAX_GB)
    n_total = n_study + max(n_test, int(TEST_SHARE * n_study))
    per_slice = n_total * N_SLOT * IMG * IMG
    afford = int(budget * 1024 ** 3 // max(per_slice, 1))
    groups = max(1, min(N_GROUP_MAX, afford // GROUP))
    log(f'memory: {avail:.1f} GB available, {budget:.1f} GB to the cache; sizing for {n_study} train + {n_total - n_study} test studies -> {groups} group(s) of {GROUP} = {groups * GROUP} slices per slot' + (f' (wanted {N_GROUP_MAX})' if groups < N_GROUP_MAX else ''))
    return groups
N_GROUP = plan_cache(len(pd.read_csv(ROOT / 'train.csv')), len(pd.read_csv(ROOT / 'test.csv')))
CACHE_SLICES = GROUP * N_GROUP
log(f'cache layout: {N_GROUP} groups x {GROUP} slices = {CACHE_SLICES} per slot')
rsna_initialize(pd.read_csv(ROOT/'test.csv',dtype={'StudyInstanceUID':str}).StudyInstanceUID.tolist())
os.environ['RSNA_COMP_ROOT']=str(ROOT)
rsna_asset_preflight()

# Resolve unsupported A5 metadata before long model inference, without inventing labels.
_meta = pd.read_csv(ROOT/'test_series.csv', dtype={'StudyInstanceUID':str})
_required = {'StudyInstanceUID','Anatomical_Plane','Fat_Suppression'}
if not _required.issubset(_meta.columns):
    raise RuntimeError('Missing required series metadata columns: '+str(sorted(_required-set(_meta.columns))))
_a5_matches = _meta.Anatomical_Plane.isin(['Sagittal','Coronal','Axial']) & _meta.Fat_Suppression.isin([0,1])
_a5_ids = set(_meta.loc[_a5_matches,'StudyInstanceUID'])
_a5_unavailable = [u for u in _RSNA_TEST_IDS if u not in _a5_ids]
rsna_json('/kaggle/working/diagnostics/a5_metadata_applicability.json', {
    'studies':len(_RSNA_TEST_IDS), 'no_matching_acquisition_uids':_a5_unavailable,
    'policy':'base-only contribution for explicit A5 absence; model failures remain fatal'})
rsna_phase('input_metadata', 'COMPLETE', a5_without_matching_acquisition=len(_a5_unavailable),
           missing_fluid_flags=int(_meta.Fluid_Sensitive.isna().sum()) if 'Fluid_Sensitive' in _meta else len(_meta))
del _meta, _a5_matches, _a5_ids, _a5_unavailable

_SINGLE_TEST_CONTRACT=s2.check_test_tables(ROOT)
rsna_json('/kaggle/working/diagnostics/complete_v2_test_contract.json',_SINGLE_TEST_CONTRACT)
rsna_json('/kaggle/working/diagnostics/complete_v2_environment.json',s2.environment_report())
print('Test contract:',_SINGLE_TEST_CONTRACT)


In [ ]:
def v2_asset_preflight():
    """Resolve the exact extra dataset before any long inference stage."""
    if not MASTER['global96_enabled']:
        return None
    try:
        manifest = _asset_find_asset(V2_GLOBAL96_MANIFEST, V2_GLOBAL96_MANIFEST_SHA)
    except Exception as exc:
        raise FileNotFoundError(
            'MAST v2 requires the EXTRA Kaggle input '
            'mattiaangeli/rsna-knee-coatnet-global96-top3 (dataset version 19459763 '
            'from the supplied Original notebook), in addition to all prior inputs. '
            'Its pinned manifest was not found. Attach the asset before Run All. '
            'To deliberately replay without it, set global96_enabled=False AND '
            'global96_family_share=0.0; that is not the default v2 challenger.'
        ) from exc
    root = manifest.parent
    runtime = root/'coatnet_global96_baseline_top3_inference.py'
    wheel = root/'timm-1.0.22-py3-none-any.whl'
    if not runtime.is_file() or not wheel.is_file() or ma_sha(wheel) != V2_TIMM_SHA:
        raise RuntimeError('Incomplete/changed Global96 runtime or its pinned timm wheel')
    record = {'root':str(root), 'manifest_sha256':ma_sha(manifest),
              'runtime_sha256':ma_sha(runtime), 'timm_wheel_sha256':ma_sha(wheel),
              'expected_models':3, 'expected_epochs':V2_GLOBAL96_EPOCHS,
              'share':MASTER['global96_family_share'],
              'source':'supplied bend-the-knee-to-speedy-raptors-the-original.ipynb'}
    rsna_json('/kaggle/working/diagnostics/v2_global96_asset.json', record)
    return record


def v2_decoder_preflight(root, max_series=64):
    """Decode up to two samples per encountered transfer syntax before inference.

    This is a sampled environment check, NOT proof that every DICOM is healthy.
    It does not change any pretrained reader's pixel preprocessing.
    """
    root = Path(root)
    table = pd.read_csv(root/'test_series.csv', dtype={'StudyInstanceUID':str, 'SeriesInstanceUID':str})
    table = table.sort_values(['StudyInstanceUID','SeriesInstanceUID'], kind='stable')
    if not len(table): raise RuntimeError('Empty test-series descriptors')
    chosen = np.unique(np.linspace(0, len(table)-1, min(int(max_series),len(table))).astype(int))
    by_syntax, header_errors = {}, []
    for row in table.iloc[chosen].itertuples(index=False):
        directory = root/'test_series'/str(row.StudyInstanceUID)/str(row.SeriesInstanceUID)
        paths = sorted(directory.glob('*.dcm'))
        if not paths:
            header_errors.append({'study':str(row.StudyInstanceUID), 'series':str(row.SeriesInstanceUID),
                                  'error':'no DICOM slices'})
            continue
        path = paths[len(paths)//2]
        try:
            ds = pydicom.dcmread(path, stop_before_pixels=True, force=True)
            syntax = str(ds.file_meta.TransferSyntaxUID)
            if len(by_syntax.setdefault(syntax,[])) < 2:
                by_syntax[syntax].append(path)
        except Exception as exc:
            header_errors.append({'path':str(path),'error':str(exc)})
    result = {'sampled_series':len(chosen), 'total_series':len(table),
              'header_errors':header_errors, 'decoded':[],
              'coverage':'sampled transfer syntaxes, not exhaustive'}
    if not by_syntax:
        raise RuntimeError('Decoder preflight found no readable test DICOM headers')
    failures = []
    for syntax, paths in sorted(by_syntax.items()):
        for path in paths:
            try:
                ds = pydicom.dcmread(path, force=True)
                pixels = ds.pixel_array
                if pixels.ndim != 2 or not np.isfinite(pixels).all():
                    raise ValueError('Expected finite 2-D decoded pixels')
                result['decoded'].append({'transfer_syntax':syntax,'path':str(path),
                                          'shape':list(pixels.shape),'dtype':str(pixels.dtype)})
            except Exception as exc:
                failures.append({'transfer_syntax':syntax,'path':str(path),
                                  'error':f'{type(exc).__name__}: {exc}'})
    result['failures'] = failures
    rsna_json('/kaggle/working/diagnostics/v2_decoder_preflight.json', result)
    if failures:
        raise RuntimeError('Sampled DICOM decode failed before model inference. '
            'See diagnostics/v2_decoder_preflight.json. Supply compatible offline '
            'decoder wheels (e.g. pylibjpeg-libjpeg/openjpeg) when the error is a missing codec; '
            'a corrupt file is a distinct data issue. Internet must stay off.')
    return result

_V2_GLOBAL96_ASSET = v2_asset_preflight()
_V2_DECODER_CHECK = v2_decoder_preflight(ROOT, MASTER['decoder_probe_series'])
print('[v2 preflight] extra model resolved; sampled decoder checks passed')


## 5. DICOM preparation and twenty pretrained DINO members

In [ ]:
HDR_TAGS = ['SeriesDescription', 'SequenceName', 'ScanOptions', 'ScanningSequence', 'RepetitionTime', 'EchoTime', 'Laterality', 'PixelSpacing', 'Rows', 'Columns', 'RescaleSlope', 'RescaleIntercept', 'ImagePositionPatient', 'ImageOrientationPatient']

def _hdr_vec(s, n):
    if not isinstance(s, str):
        return None
    try:
        v = [float(x) for x in s.split('|')]
    except ValueError:
        return None
    return np.array(v) if len(v) >= n else None

def side_from_geometry(h):
    cx = {}
    for r in h.itertuples(index=False):
        ipp = _hdr_vec(getattr(r, 'ImagePositionPatient', None), 3)
        iop = _hdr_vec(getattr(r, 'ImageOrientationPatient', None), 6)
        ps = _hdr_vec(getattr(r, 'PixelSpacing', None), 2)
        rows, cols = (getattr(r, 'Rows', None), getattr(r, 'Columns', None))
        if ipp is None or iop is None or ps is None or (not rows) or (not cols):
            continue
        try:
            c = ipp[:3] + iop[:3] * ps[1] * float(cols) / 2 + iop[3:6] * ps[0] * float(rows) / 2
        except (TypeError, ValueError):
            continue
        cx.setdefault(r.StudyInstanceUID, []).append(float(c[0]))
    out = {}
    for st, xs in cx.items():
        m = float(np.median(xs))
        out[st] = None if abs(m) < LAT_MIN_OFFSET_MM else 'R' if m < 0 else 'L'
    return out

def side_from_corner_x(h):
    out = {}
    for st, g in h.groupby('StudyInstanceUID'):
        xs = []
        for r in g.itertuples(index=False):
            ipp = _hdr_vec(getattr(r, 'ImagePositionPatient', None), 3)
            if ipp is not None and np.isfinite(ipp).all():
                xs.append(float(ipp[0]))
        if not xs:
            out[st] = None
            continue
        x = float(np.median(xs))
        out[st] = None if abs(x) < LEGACY_LAT_OFFSET_MM else 'R' if x < 0 else 'L'
    return out

def lat_of(h, tag=''):
    geo = side_from_corner_x(h) if RULES['lat'] == 'corner_x' else side_from_geometry(h)
    d, n_tag, n_geo, n_none, n_disagree = ({}, 0, 0, 0, 0)
    for st, g in h.groupby('StudyInstanceUID'):
        v = [str(x).strip().upper() for x in g['Laterality'].dropna()]
        if RULES['lat'] == 'corner_x' and 'ImageLaterality' in g.columns:
            v += [str(x).strip().upper() for x in g['ImageLaterality'].dropna()]
        v = [x[0] for x in v if x and x[0] in ('L', 'R')]
        side = v[0] if v else None
        if side is not None:
            n_tag += 1
            if geo.get(st) is not None and geo[st] != side:
                n_disagree += 1
        else:
            side = geo.get(st)
            n_geo += side is not None
            n_none += side is None
        d[st] = side
    log(f'{tag}laterality: {n_tag} from the tag, {n_geo} from geometry, {n_none} unresolved; tag and geometry disagree on {n_disagree} ({n_disagree / max(n_tag, 1):.1%} of the tagged)')
    return d

def probe(item):
    split, study, series, path = item
    row = {'split': split, 'StudyInstanceUID': study, 'SeriesInstanceUID': series, 'dir': path}
    try:
        files = sorted((e.name for e in os.scandir(path) if e.name.endswith('.dcm')))
        row['files'] = files
        row['n_slices'] = len(files)
        if not files:
            return row
        ds = pydicom.dcmread(os.path.join(path, files[len(files) // 2]), stop_before_pixels=True, force=True)
        for t in HDR_TAGS:
            v = getattr(ds, t, None)
            if v is None:
                row[t] = None
            elif isinstance(v, (list, tuple)) or type(v).__name__ == 'MultiValue':
                row[t] = '|'.join((str(x) for x in v))
            else:
                row[t] = str(v)
    except Exception as exc:
        row['err'] = str(exc)[:120]
    return row

def walk(split):
    key=(str(ROOT.resolve()),str(split))
    if key in _RSNA_HEADERS_MEMO:
        return _RSNA_HEADERS_MEMO[key].copy(deep=True)
    base = ROOT / split
    items = []
    if not base.is_dir():
        return pd.DataFrame(columns=['split', 'StudyInstanceUID', 'SeriesInstanceUID', 'dir', 'files', 'n_slices'] + HDR_TAGS)
    for study in os.scandir(base):
        if study.is_dir():
            for series in os.scandir(study.path):
                if series.is_dir():
                    items.append((split, study.name, series.name, series.path))
    with ThreadPoolExecutor(max_workers=HDR_THREADS) as pool:
        rows = list(pool.map(probe, items))
    result=pd.DataFrame(rows)
    _RSNA_HEADERS_MEMO[key]=result.copy(deep=True)
    return result

def annotate(df):
    desc = df['SeriesDescription'].fillna('') + ' ' + df['SequenceName'].fillna('')
    desc = desc.str.lower().str.replace(_SEP, ' ', regex=True)
    opts = df['ScanOptions'].fillna('').str.upper().str.split('|')
    opts_fs = opts.apply(lambda ts: any((t.strip() in FATSAT_OPTS for t in ts)))
    df['fatsat'] = desc.str.contains(_FATSAT_RX) | opts_fs
    tr = pd.to_numeric(df['RepetitionTime'], errors='coerce')
    te = pd.to_numeric(df['EchoTime'], errors='coerce')
    gre = df['ScanningSequence'].fillna('').str.upper().str.contains('GR')
    t1, t2, pdw = (desc.str.contains(_T1_RX), desc.str.contains(_T2_RX), desc.str.contains(_PD_RX))
    df['weight'] = np.where(t1 & ~t2 & ~pdw, 'T1', np.where(t2 & ~pdw, 'T2', np.where(pdw, 'PD', np.where(gre, 'GRE', np.where(tr < 800, 'T1', np.where(te > 60, 'T2', np.where(tr >= 800, 'PD', 'UNK')))))))
    df['fluid'] = np.isin(df['weight'], ['PD', 'T2'])
    df['px'] = pd.to_numeric(df['PixelSpacing'].fillna('').str.split('|').str[0].replace('', np.nan), errors='coerce')
    return df


In [ ]:
# The original read_slot preprocessing is unchanged. Only the actual float32
# decoded/rescaled plane is memoized, so repeated source reads can be reused.
def _single_decode_pixels(path):
    key=s2.pixel_key(path)
    def load():
        ds=pydicom.dcmread(path,force=True)
        a=ds.pixel_array.astype(np.float32)
        a=a*float(getattr(ds,'RescaleSlope',1) or 1)+float(getattr(ds,'RescaleIntercept',0) or 0)
        if a.ndim!=2 or not np.isfinite(a).all():
            raise ValueError('expected finite 2-D pixels')
        return a
    return _SINGLE_DECODE_CACHE.get(key,load)


In [ ]:
def pick_slots(series_df, plane_map):
    series_df = series_df.copy()
    series_df['plane'] = series_df['SeriesInstanceUID'].map(plane_map)
    out = {}
    for study, g in series_df.groupby('StudyInstanceUID'):
        chosen = {}
        for name, plane, fluid, fs in SLOTS:
            sel = (g['plane'] == plane) & (g['fatsat'] == fs)
            if fluid is not None:
                sel &= g['fluid'] == fluid
            cand = g[sel]
            if len(cand) == 0 and RULES['slot_fallback'] and (fluid is False):
                cand = g[(g['plane'] == plane) & ~g['fatsat']]
            if len(cand):
                chosen[name] = cand.sort_values('n_slices', ascending=False).iloc[0]
        out[study] = chosen
    return out


In [ ]:
ORDER_TAGS = [(32, 50), (32, 55), (32, 19), (40, 48)]
DECODE_FAILED = []
RAPTOR_HEADER_CACHE = {}


def _natural_key(name):
    return tuple((int(x) if x.isdigit() else x.lower() for x in re.split('(\\d+)', str(name))))

def _order_dominant_axis(rec):
    files, d = (rec['files'], rec['dir'])
    rows = []
    for pos, f in enumerate(files):
        ipp = inst = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True, specific_tags=['ImagePositionPatient', 'InstanceNumber'])
            raw = getattr(ds, 'ImagePositionPatient', None)
            if raw is not None and len(raw) >= 3:
                c = np.asarray(raw[:3], dtype=np.float64)
                if np.isfinite(c).all():
                    ipp = c
            n = getattr(ds, 'InstanceNumber', None)
            if n is not None:
                inst = float(n)
        except Exception:
            pass
        rows.append((f, ipp, inst, pos))
    placed = [r for r in rows if r[1] is not None]
    need = max(2, int(0.8 * len(rows)))
    if len(placed) >= need:
        xyz = np.stack([r[1] for r in placed])
        axis = int(np.argmax(np.ptp(xyz, axis=0)))
        spare = float(np.nanmedian(xyz[:, axis]))
        rows.sort(key=lambda r: (float(r[1][axis]) if r[1] is not None else spare, r[2] if r[2] is not None else float('inf'), r[3]))
    elif sum((r[2] is not None for r in rows)) >= need:
        rows.sort(key=lambda r: (r[2] if r[2] is not None else float('inf'), r[3]))
    else:
        rows.sort(key=lambda r: _natural_key(r[0]))
    return ([r[0] for r in rows], True)

def order_slices(rec):
    if RULES['order'] == 'dominant_axis':
        return _order_dominant_axis(rec)
    files, d = (rec['files'], rec['dir'])
    keyed = []
    raptor_records = []
    raptor_safe = True
    for f in files:
        k = None
        ds = None
        file_raptor_safe = True
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True, specific_tags=ORDER_TAGS)
            iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
            ipp = np.asarray(ds.ImagePositionPatient, dtype=float)
            k = float(np.dot(ipp, np.cross(iop[:3], iop[3:])))
        except Exception:
            file_raptor_safe = False
            try:
                k = float(ds.InstanceNumber)
            except Exception:
                k = None
        if file_raptor_safe:
            try:
                spacing = getattr(ds, 'PixelSpacing', None)
                spacing = float(spacing[0]) if spacing is not None else 0.5
                raptor_records.append((k, os.path.join(d, f), spacing))
            except Exception:
                file_raptor_safe = False
        raptor_safe &= file_raptor_safe
        keyed.append((k, f))
    if any((k is None for k, _ in keyed)):
        return (sorted(files,key=_natural_key), False)
    # Cache only the unambiguous geometric order. Every fallback still uses
    # Raptor's original reader, so slice selection cannot change.
    if raptor_safe and len({item[0] for item in raptor_records}) == len(files):
        raptor_records.sort(key=lambda item: item[0])
        spacings = [item[2] for item in raptor_records]
        RAPTOR_HEADER_CACHE[str(d)] = (
            [(item[1], item[2]) for item in raptor_records],
            float(np.median(spacings)) if spacings else 0.5,
        )
    return ([f for _, f in sorted(keyed, key=lambda t: t[0])], True)

def read_slot(rec, n_slice=None, out_size=None):
    """Same healthy pixels as BTKD; failed slices cannot erase good native planes."""
    n_slice = GROUP if n_slice is None else int(n_slice)
    out_size = IMG if out_size is None else int(out_size)
    files, d, px = rec.get('ordered') or rec['files'], rec['dir'], rec['px']
    if not files:
        rsna_event('slot_no_files', series=str(rec.get('SeriesInstanceUID', d)), dir=str(d))
        return None
    lo, hi = int(SLICE_BAND[0]*(len(files)-1)), int(SLICE_BAND[1]*(len(files)-1))
    idx = np.unique(np.linspace(lo,hi,n_slice).astype(int)) if hi>lo else np.array([len(files)//2])
    while len(idx)<n_slice: idx=np.append(idx,idx[-1])
    decoded, errors = {}, []
    for i in sorted(set(int(v) for v in idx[:n_slice])):
        path=os.path.join(d,files[i])
        try:
            a=_single_decode_pixels(path)
            if a.ndim!=2 or not np.isfinite(a).all(): raise ValueError('expected finite 2-D pixels')
            decoded[i]=a
        except Exception as exc:
            decoded[i]=None; errors.append({'path':path,'error':f'{type(exc).__name__}: {exc}'})
    planes=[decoded[int(i)] for i in idx[:n_slice]]
    got=[i for i,p in enumerate(planes) if p is not None]
    if errors:
        DECODE_FAILED.append(rec.get('SeriesInstanceUID',d))
        rsna_event('partial_slice_decode',series=str(rec.get('SeriesInstanceUID',d)),errors=errors)
    if not got:
        rsna_event('slot_all_decode_failed', series=str(rec.get('SeriesInstanceUID', d)), dir=str(d), errors=errors[:5])
        return None
    from collections import Counter
    shape=Counter(planes[i].shape for i in got).most_common(1)[0][0]
    odd=[i for i in got if planes[i].shape!=shape]
    if odd:
        rsna_event('slot_shape_mismatch', series=str(rec.get('SeriesInstanceUID', d)), dir=str(d), majority=[int(v) for v in shape], dropped=len(odd))
        for i in odd: planes[i]=None
        got=[i for i in got if planes[i] is not None]
    for i,p in enumerate(planes):
        if p is None:
            planes[i]=np.zeros(shape,np.float32) if RULES['decode_fill']=='zero' else planes[min(got,key=lambda j:abs(j-i))]
    vol=np.stack(planes)
    if px and np.isfinite(px) and px>0:
        want=int(round(CROP_MM/px)); h,w=shape
        if 16<want<min(h,w):
            cy,cx=h//2,w//2; half=want//2
            vol=vol[:,max(0,cy-half):cy+half,max(0,cx-half):cx+half]
    lo_v,hi_v=np.percentile(vol,[1,99])
    vol=np.clip((vol-lo_v)/max(hi_v-lo_v,1e-6),0,1)
    t=torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)
    t=F.interpolate(t,size=(out_size,out_size),mode='bilinear',align_corners=False)
    return (t.squeeze(0)*255).round().clamp(0,255).to(torch.uint8)


In [ ]:
def normalise_laterality(img, plane, lat):
    if lat != 'R':
        return img
    if plane in ('Coronal', 'Axial'):
        return torch.flip(img, dims=[-1])
    return torch.flip(img, dims=[0])


In [ ]:
def build_cache(slot_map, plane_map, lat_map, tag):
    studies = list(_RSNA_TEST_IDS) if _RSNA_TEST_IDS is not None else sorted(slot_map)
    if set(studies)!=set(slot_map):
        rsna_event('coverage_mismatch',tag=tag,missing=[str(s) for s in studies if s not in slot_map][:20],extra=len(set(slot_map)-set(studies)))
        slot_map={s:slot_map.get(s,{}) for s in studies}
    sidx={s:i for i,s in enumerate(studies)}
    cache=rsna_array((len(studies),N_SLOT,CACHE_SLICES,IMG,IMG),tag)
    mask=np.zeros((len(studies),N_SLOT),np.float32)
    jobs=[(st,k,plane,slot_map[st][name]) for st in studies for k,(name,plane,_,_) in enumerate(SLOTS) if name in slot_map[st]]
    log(f'{tag}: {len(studies)} studies, {len(jobs)} slot-series, {cache.nbytes/2**30:.2f} GiB pixels')
    missing=[]; reused=0
    for job in jobs:
        rec=job[3]
        key=(str(Path(rec['dir']).resolve()),RULES['order'],tuple(sorted(rec['files'])))
        saved=_RSNA_ORDER_MEMO.get(key)
        if saved is not None:
            rec['ordered']=list(saved[0]);reused+=1
        else: missing.append((job,key))
    def ordered(item):
        job,key=item
        try:
            files,good=order_slices(job[3])
            if len(files)!=len(job[3]['files']) or set(files)!=set(job[3]['files']):
                raise ValueError(f'ordering lost/added files ({len(files)} vs {len(job[3]["files"])})')
        except Exception as exc:
            rsna_event('ordering_fallback',tag=tag,series=str(job[3].get('SeriesInstanceUID')),error=f'{type(exc).__name__}: {exc}')
            return key,sorted(job[3]['files']),False
        return key,files,bool(good)
    with ThreadPoolExecutor(max_workers=ORDER_THREADS) as pool:
        for start in range(0,len(missing),256):
            rsna_deadline(tag+' ordering')
            block=missing[start:start+256]
            for (job,_),(key,files,good) in zip(block,pool.map(ordered,block)):
                job[3]['ordered']=files;_RSNA_ORDER_MEMO[key]=(tuple(files),good)
                if not good: rsna_event('deterministic_order_fallback',series=str(job[3].get('SeriesInstanceUID')))
    done=0; unfilled=0
    def _tolerant_read_slot(j):
        try: return read_slot(j[3],CACHE_SLICES,IMG)
        except Exception as exc:
            rsna_event('slot_read_error',tag=tag,study=str(j[0]),series=str(j[3].get('SeriesInstanceUID')),error=f'{type(exc).__name__}: {exc}'); return None
    with ThreadPoolExecutor(max_workers=PIX_THREADS) as pool:
        for start in range(0,len(jobs),128):
            rsna_deadline(tag+' decoding')
            block=jobs[start:start+128]
            for (st,k,plane,rec),img in zip(block,pool.map(_tolerant_read_slot,block)):
                done+=1
                if img is None:
                    unfilled+=1; rsna_event('slot_unfilled',tag=tag,study=str(st),series=str(rec.get('SeriesInstanceUID')),slot=int(k)); continue
                cache[sidx[st],k]=normalise_laterality(img,plane,lat_map.get(st)).numpy()
                mask[sidx[st],k]=1
    if unfilled: log(f'{tag}: {unfilled} slot(s) left empty and masked (flagged)')
    _empty=[studies[i] for i in np.flatnonzero(mask.sum(1)==0)]
    if _empty:
        rsna_event('empty_study_rows',tag=tag,count=len(_empty),studies=[str(s) for s in _empty])
        log(f'{tag}: {len(_empty)} study(ies) with no series matching this layout kept as all-zero masked rows (parent policy): {_empty[:5]}')
    if done!=len(jobs):
        raise RuntimeError(f'{tag}: incomplete cache; {done}/{len(jobs)} slots')
    if isinstance(cache,np.memmap): cache.flush()
    rsna_event('cache_complete',tag=tag,studies=len(studies),slot_jobs=len(jobs),filled=done,order_cache_hits=reused,bytes=int(cache.nbytes))
    log(f'{tag}: COMPLETE {done}/{len(jobs)}; reused ordering for {reused} series')
    return studies,cache,mask


In [ ]:
class SlotHead(nn.Module):

    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2, prior=False):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden
        p_ = torch.zeros(n_out, n_slot)
        if prior and n_slot == len(SLOTS) and (n_out == len(TARGETS)):
            for t, slots in SLOT_PRIOR_TABLE.items():
                if t in TARGETS:
                    p_[TARGETS.index(t), list(slots)] = SLOT_PRIOR_STRENGTH
        self.prior = prior
        if prior:
            self.register_buffer('slot_prior', p_)

    def forward(self, x, mask):
        h = self.proj(x) + self.slot_emb
        att = torch.einsum('bsh,oh->bos', h, self.query) / self.hidden ** 0.5
        if self.prior:
            att = att + self.slot_prior.unsqueeze(0)
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -10000.0).softmax(-1)
        ctx = self.drop(torch.einsum('bos,bsh->boh', att, h))
        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias


In [ ]:
class Model(nn.Module):

    def __init__(self, backbone, dim, pool='cls_mean', prior=False):
        super().__init__()
        self.backbone = backbone
        self.pool = pool
        self.head = SlotHead(dim * POOL_PARTS[pool], N_SLOT, len(TARGETS), prior=prior)
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, imgs, mask, img_size=None):
        B, S = imgs.shape[:2]
        x = imgs.reshape(B * S, *imgs.shape[2:]).float().div_(255.0)
        if img_size is not None and img_size != x.shape[-1]:
            x = F.interpolate(x, size=(img_size, img_size), mode='bilinear', align_corners=False)
        x = (x - self.mean) / self.std
        out = self.backbone(pixel_values=x).last_hidden_state
        patch = out[:, 1:]
        parts = [out[:, 0], patch.mean(1)]
        if self.pool == 'cls_mean_focal':
            k = max(1, patch.shape[1] // 8)
            parts.append(patch.topk(k, dim=1).values.mean(1))
        feat = torch.cat(parts, dim=1).reshape(B, S, -1)
        return self.head(feat, mask)


In [ ]:
def build_model(unfreeze_last, source=None, variant='small', pool='cls_mean', prior=False):
    from transformers import AutoModel
    p = source if source is not None else DINOV2_SOURCE
    if p is None:
        raise FileNotFoundError('DINOv2 weights not attached')
    bb = AutoModel.from_pretrained(str(p))
    n_layer = len(bb.encoder.layer)
    for prm in bb.parameters():
        prm.requires_grad = False
    for blk in bb.encoder.layer[max(0, n_layer - unfreeze_last):]:
        for prm in blk.parameters():
            prm.requires_grad = True
    for prm in bb.layernorm.parameters():
        prm.requires_grad = True
    dim = bb.config.hidden_size
    trainable = sum((p.numel() for p in bb.parameters() if p.requires_grad))
    log(f'backbone: {n_layer} blocks, last {unfreeze_last} trainable ({trainable / 1000000.0:.1f}M params), feature dim {dim * POOL_PARTS[pool]}')
    return Model(bb, dim, pool=pool, prior=prior)
_speed_original_build_model = build_model
"""Reuse a CPU architecture template; every member still strict-loads and fingerprints."""
import copy as _speed_copy
import threading as _speed_dino_threading

_speed_dino_templates = {}
_speed_dino_lock = _speed_dino_threading.RLock()


def build_model(unfreeze_last, source=None, variant='small', pool='cls_mean', prior=False):
    key = (int(unfreeze_last), None if source is None else str(source), variant,
           pool, bool(prior), N_SLOT, tuple(TARGETS))
    with _speed_dino_lock:
        if key not in _speed_dino_templates:
            template = _speed_original_build_model(unfreeze_last, source, variant, pool, prior)
            assert all(p.device.type == 'cpu' for p in template.parameters())
            _speed_dino_templates[key] = template
        # No mutable state or GPU tensors shared between independently trained models.
        return _speed_copy.deepcopy(_speed_dino_templates[key])


In [ ]:
SHARED_DINO_PREFIX_LAYERS = 6
SHARED_DINO_PREFIX_SHA256 = '0a55b893bde971c864ea5aeea443f075b17c084d13aa36f30bd1c7863f655cf9'
def _shared_prefix_state_sha256(state):
    digest = hashlib.sha256()
    prefixes = (
        'backbone.embeddings.',
        *(
            f'backbone.encoder.layer.{index}.'
            for index in range(
                SHARED_DINO_PREFIX_LAYERS
            )
        ),
    )
    keys = sorted(
        key
        for key in state
        if key.startswith(prefixes)
    )
    if len(keys) != 113:
        raise WeightsError(
            'shared DINOv2 prefix key-count drift: '
            f'{len(keys)} != 113'
        )
    for key in keys:
        tensor = state[key].detach().cpu().contiguous()
        digest.update(key.encode())
        digest.update(str(tensor.dtype).encode())
        digest.update(str(tuple(tensor.shape)).encode())
        digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()

def _shared_prefix_eligible(members):
    if not members or not DEVS:
        return False
    return all(
        'state' not in member
        and int(member['config']['unfreeze_last'])
        == SHARED_DINO_PREFIX_LAYERS
        and member['config']['variant'] == 'small'
        and member['config'].get('pool', 'cls_mean')
        in POOL_PARTS
        for member in members
    )

def _shared_dino_prefix(model, images, img_size):
    batch, slots = images.shape[:2]
    values = images.reshape(
        batch * slots,
        *images.shape[2:],
    ).float().div_(255.0)
    if (
        img_size is not None
        and img_size != values.shape[-1]
    ):
        values = F.interpolate(
            values,
            size=(img_size, img_size),
            mode='bilinear',
            align_corners=False,
        )
    values = (
        values - model.mean
    ) / model.std
    hidden = model.backbone.embeddings(values)
    for layer in model.backbone.encoder.layer[
        :SHARED_DINO_PREFIX_LAYERS
    ]:
        hidden = layer(hidden)
        hidden = hidden[0] if isinstance(hidden, (tuple, list)) else hidden
    return hidden

def _shared_dino_tail(
    model,
    hidden,
    batch,
    slots,
    slot_mask,
):
    value = hidden
    for layer in model.backbone.encoder.layer[
        SHARED_DINO_PREFIX_LAYERS:
    ]:
        value = layer(value)
        value = value[0] if isinstance(value, (tuple, list)) else value
    value = model.backbone.layernorm(value)
    patch = value[:, 1:]
    parts = [
        value[:, 0],
        patch.mean(1),
    ]
    if model.pool == 'cls_mean_focal':
        count = max(
            1,
            patch.shape[1] // 8,
        )
        parts.append(
            patch.topk(
                count,
                dim=1,
            ).values.mean(1)
        )
    feature = torch.cat(
        parts,
        dim=1,
    ).reshape(
        batch,
        slots,
        -1,
    )
    return model.head(feature, slot_mask)

@torch.no_grad()
def _predict_shared_dino_members(
    entries,
    cache,
    mask,
    idx,
    dev,
    img_size,
    group,
    starts,
):
    target_idx = {
        target: index
        for index, target in enumerate(TARGETS)
    }
    states = {}
    for member, model, jitter in entries:
        generator = torch.Generator(device=dev)
        jitter_seed = (
            SEED
            + int(
                hashlib.sha256(
                    str(member['id']).encode()
                ).hexdigest()[:8],
                16,
            )
        )
        generator.manual_seed(
            int(jitter_seed) % (2 ** 63 - 1)
        )
        states[member['id']] = {
            'out': [],
            'public': [],
            'soft': [],
            'generator': generator,
            'jitter': jitter,
        }
        model.eval()

    reference = entries[0][1]
    for batch_start in range(
        0,
        len(idx),
        EVAL_BATCH,
    ):
        selected = idx[
            batch_start:
            batch_start + EVAL_BATCH
        ]
        slot_mask = torch.from_numpy(
            mask[selected]
        ).to(dev)
        batch = len(selected)
        slots = cache.shape[1]
        member_windows = {
            member['id']: (
                [],
                [],
                [],
            )
            for member, _, _ in entries
        }

        for start in starts:
            rows = torch.from_numpy(
                np.ascontiguousarray(
                    cache[
                        selected,
                        :,
                        start:start + group,
                    ]
                )
            ).to(dev)
            def _dino_shared_forward(_amp):
                with torch.autocast('cuda', enabled=_amp):
                    _common = _shared_dino_prefix(reference, rows, img_size)
                    return {member['id']: _shared_dino_tail(model, _common, batch, slots, slot_mask).float() for member, model, _ in entries}
            original_logits = _dino_shared_forward(dev.type == 'cuda')
            if dev.type == 'cuda' and not all(bool(torch.isfinite(v).all()) for v in original_logits.values()):
                rsna_event('dino_fp16_nonfinite_retry_fp32', window_start=int(start))
                original_logits = _dino_shared_forward(False)

            for member, model, jitter in entries:
                member_id = member['id']
                view_logits = [
                    original_logits[member_id]
                ]
                if jitter:
                    jittered = augment(
                        rows,
                        generator=states[
                            member_id
                        ]['generator'],
                    )
                    with torch.autocast(
                        'cuda',
                        enabled=dev.type == 'cuda',
                    ):
                        jitter_hidden = (
                            _shared_dino_prefix(
                                reference,
                                jittered,
                                img_size,
                            )
                        )
                        view_logits.append(
                            _shared_dino_tail(
                                model,
                                jitter_hidden,
                                batch,
                                slots,
                                slot_mask,
                            ).float()
                        )
                    if dev.type == 'cuda' and not bool(torch.isfinite(view_logits[-1]).all()):
                        rsna_event('dino_jitter_fp16_nonfinite_dropped', member=str(member_id))
                        view_logits.pop()
                view_probs = [
                    torch.sigmoid(value)
                    for value in view_logits
                ]
                probabilities, logits, originals = (
                    member_windows[member_id]
                )
                logits.append(
                    torch.stack(
                        view_logits
                    ).mean(0)
                )
                probabilities.append(
                    torch.stack(
                        view_probs
                    ).mean(0)
                )
                originals.append(
                    view_probs[0]
                )

        for member, _, _ in entries:
            member_id = member['id']
            win_probs, win_logits, win_originals = (
                member_windows[member_id]
            )
            probabilities = torch.stack(win_probs)
            logits = torch.stack(win_logits)
            originals = torch.stack(win_originals)
            value = (
                torch.sigmoid(logits.mean(0))
                if TTA_POOL == 'logit'
                else probabilities.mean(0)
            )
            value = apply_target_window_pool(
                value,
                probabilities,
                logits,
                originals,
                TTA_TARGET_POOL,
                target_idx,
            )
            states[member_id]['out'].append(
                value.cpu().numpy()
            )
            public_value = apply_target_window_pool(
                originals.mean(0),
                originals,
                logits,
                originals,
                PUBLIC_FRONTIER_TARGET_POOL,
                target_idx,
            )
            states[member_id]['public'].append(
                public_value.cpu().numpy()
            )
            states[member_id]['soft'].append(
                legacy_fold_soft_window_pool(
                    originals,
                    target_idx,
                ).cpu().numpy()
            )

    return {
        member['id']: (
            np.concatenate(states[member['id']]['out']),
            np.concatenate(states[member['id']]['public']),
            np.concatenate(states[member['id']]['soft']),
        )
        for member, _, _ in entries
    }

@torch.inference_mode()
def _check_shared_dino_path(model,dev,img_size,tag):
    model.eval()
    g=torch.Generator().manual_seed(SEED)
    im=torch.randint(0,256,(2,N_SLOT,GROUP,img_size,img_size),generator=g,dtype=torch.uint8).to(dev)
    mask=torch.ones(2,N_SLOT,device=dev);mask[1,-1]=0
    direct=model(im,mask,img_size).float()
    split=_shared_dino_tail(model,_shared_dino_prefix(model,im,img_size),2,N_SLOT,mask).float()
    if not torch.isfinite(split).all() or not torch.allclose(split,direct,rtol=1e-5,atol=2e-5):
        raise RuntimeError(f'{tag}: shared-prefix execution differs from full forward')
    rsna_event('dino_shared_path_parity',member=tag,max_abs=float((split-direct).abs().max()))

def _run_shared_dino_group(
    path,
    members,
    cache,
    mask,
    idx,
    starts,
):
    ordered = sorted(
        members,
        key=lambda member: -(
            member.get('holdout') or 0
        ),
    )
    # run_dinov2 retains only submission_public_0899.csv, whose member
    # predictions are built exclusively from the unaugmented views.
    # Jittering here affected only the intermediate submission.csv that the
    # retained public-frontier file replaces at the end of this stage.
    plans = [
        (
            member,
            False,
        )
        for member in ordered
    ]
    assignments = [
        plans[index::len(DEVS)]
        for index in range(len(DEVS))
    ]
    results = {}
    result_lock = threading.Lock()

    def worker(dev, plans_for_device):
        global _DINOV2_MATCHED_MEMBERS
        loaded = []
        for member, jitter in plans_for_device:
            with BUILD_LOCK:
                checkpoint = torch.load(
                    Path(path) / member['file'],
                    map_location='cpu',
                    weights_only=False,
                )
                state = checkpoint['model']
                prefix_sha256 = (
                    _shared_prefix_state_sha256(
                        state
                    )
                )
                if (
                    prefix_sha256
                    != SHARED_DINO_PREFIX_SHA256
                ):
                    raise WeightsError(
                        f"{member['id']}: frozen prefix "
                        'hash mismatch: '
                        f'{prefix_sha256}'
                    )
                model = build_model(
                    int(
                        member['config'][
                            'unfreeze_last'
                        ]
                    ),
                    variant=member['config'][
                        'variant'
                    ],
                    pool=member['config'].get(
                        'pool',
                        'cls_mean',
                    ),
                    prior=bool(
                        member['config'].get(
                            'prior',
                            False,
                        )
                    ),
                ).to(dev)
                model.load_state_dict(state)
                check_fingerprint(
                    model,
                    dev,
                    IMG,
                    checkpoint.get('fingerprint'),
                    tag=f"{member['id']}: ",
                )
                _check_shared_dino_path(model, dev, IMG, member['id'])
                _DINOV2_MATCHED_MEMBERS += 1
                del checkpoint, state
            loaded.append(
                (member, model, jitter)
            )

        predicted = _predict_shared_dino_members(
            loaded,
            cache,
            mask,
            idx,
            dev,
            IMG,
            GROUP,
            starts,
        )
        with result_lock:
            results.update(predicted)
        del loaded
        gc.collect()
        with torch.cuda.device(dev):
            torch.cuda.empty_cache()

    with ThreadPoolExecutor(max_workers=len(DEVS)) as pool:
        futures=[pool.submit(worker,dev,plan) for dev,plan in zip(DEVS,assignments)]
        for future in futures: future.result()
    if len(results) != len(ordered):
        raise WeightsError(
            'shared DINOv2 worker did not return '
            f'all members: {len(results)} / '
            f'{len(ordered)}'
        )
    log(
        'shared first 6 DINOv2 blocks for '
        f'{len(ordered)} members; original views '
        f'computed once per {len(DEVS)} device(s)'
    )
    return [
        (
            member,
            results[member['id']],
            jitter,
        )
        for member, jitter in plans
    ]

FINGERPRINT_TOL = 0.002
EXPECTED_DINOV2_MEMBERS = 20
_DINOV2_MATCHED_MEMBERS = 0

def fingerprint(model, dev, img_size, n_slot=None, group=None, seed=None):
    n_slot = N_SLOT if n_slot is None else n_slot
    group = GROUP if group is None else group
    seed = SEED if seed is None else seed
    g = torch.Generator().manual_seed(seed)
    imgs = torch.randint(0, 256, (2, n_slot, group, img_size, img_size), generator=g, dtype=torch.uint8).to(dev)
    mask = torch.ones(2, n_slot, device=dev)
    mask[1, -1] = 0.0
    was_training = model.training
    model.eval()
    with torch.no_grad():
        out = model(imgs, mask, img_size).float().cpu().numpy()
    if was_training:
        model.train()
    return out

def check_fingerprint(model, dev, img_size, expected, tol=FINGERPRINT_TOL, tag=''):
    got = fingerprint(model, dev, img_size)
    exp = np.asarray(expected, np.float32)
    if not np.isfinite(got).all() or not np.isfinite(exp).all():
        raise WeightsError(f'{tag}: nonfinite fingerprint')
    if got.shape != exp.shape:
        raise WeightsError(f'{tag}fingerprint shape {got.shape} != stored {exp.shape}: the architecture is not the one these weights were fitted to')
    d = float(np.abs(got - exp).max())
    if d > tol:
        raise WeightsError(f'{tag}fingerprint differs by {d:.4g} (tolerance {tol:g}). The weights load but do not compute what they computed when fitted - preprocessing, resolution or architecture has moved between the two runs.')
    log(f'{tag}fingerprint matches within {d:.2g}')
    return d

class WeightsError(RuntimeError):
    pass

def find_weights(name='manifest.json'):
    import json
    roots = [
        Path('/kaggle/input/datasets/pilkwang/rsna-knee-weights'),
        Path('/kaggle/input/rsna-knee-weights'),
    ]
    valid = []
    for root in roots:
        path = root / name
        if not path.is_file():
            continue
        try:
            man = json.loads(path.read_text())
        except (OSError, ValueError) as exc:
            raise WeightsError(f'invalid pinned weights manifest at {path}: {exc}') from exc
        if not isinstance(man.get('members'), list) or len(man['members']) != EXPECTED_DINOV2_MEMBERS:
            raise WeightsError(f'{path} must list exactly {EXPECTED_DINOV2_MEMBERS} members')
        if len({m.get('id') for m in man['members']}) != EXPECTED_DINOV2_MEMBERS:
            raise WeightsError(f'{path} has duplicate or missing member ids')
        missing = [m['file'] for m in man['members'] if not (root / m['file']).is_file()]
        if missing:
            raise WeightsError(
                f'{root} lists {len(man["members"])} members but misses {missing[0]!r}'
            )
        valid.append(root)
    if len(valid) != 1:
        raise WeightsError(f'expected one pinned rsna-knee-weights root, found {valid}')
    return valid[0]
TTA_OVERLAP = True
TTA_POOL = 'prob'
PUBLIC_FRONTIER_TARGET_POOL = {'Fracture': 'max', 'Contusion': 'max', 'Medial Meniscus': 'max', 'Lateral Meniscus': 'max', 'ACL': 'top2', 'MCL': 'top2', "Baker's": 'max'}
TTA_TARGET_POOL = {**PUBLIC_FRONTIER_TARGET_POOL, 'Synovitis': 'original_mean'}
# No-extra-pass diversity branch: smooth focal pooling is evaluated from
# the same no-jitter public-member windows already used by the parent.
LEGACY_FOLD_SOFTPOOL_BETA = {
    'ACL': 6.0, 'MCL': 6.0,
    'Medial Meniscus': 8.0, 'Lateral Meniscus': 8.0,
    "Baker's": 8.0, 'Contusion': 8.0, 'Fracture': 10.0,
}
LEGACY_FOLD_SOFTPOOL_ALPHA = {
    'ACL': 0.20, 'MCL': 0.20,
    'Medial Meniscus': 0.25, 'Lateral Meniscus': 0.25,
    "Baker's": 0.20, 'Contusion': 0.20, 'Fracture': 0.15,
}
LEGACY_MEMBER_WEIGHT_BY_TARGET = {'Lateral Meniscus': 15.0, 'Medial OA': 2.5, 'Lateral OA': 15.0, 'Contusion': 5.0}

def window_starts(n_slice, group, overlap=None):
    overlap = TTA_OVERLAP if overlap is None else overlap
    if overlap and n_slice >= group:
        return list(range(n_slice - group + 1))
    return [g * group for g in range(max(n_slice // group, 1))]

def apply_target_window_pool(values, probs, logits, original_probs, mapping, target_idx):
    for target, mode in mapping.items():
        j = target_idx[target]
        if mode == 'max':
            values[:, j] = probs[:, :, j].max(0).values
        elif mode == 'mean':
            values[:, j] = probs[:, :, j].mean(0)
        elif mode == 'logit_mean':
            values[:, j] = torch.sigmoid(logits[:, :, j].mean(0))
        elif mode == 'original_mean':
            values[:, j] = original_probs[:, :, j].mean(0)
        elif mode in ('top2', 'top3'):
            k = min(int(mode[3:]), probs.shape[0])
            values[:, j] = probs[:, :, j].topk(k, dim=0).values.mean(0)
        else:
            raise ValueError(f'unknown TTA pooling mode for {target}: {mode}')
    return values

def legacy_fold_soft_window_pool(original_probs, target_idx):
    values = original_probs.mean(0).clone()
    for target, beta in LEGACY_FOLD_SOFTPOOL_BETA.items():
        j = target_idx[target]
        x = original_probs[:, :, j]
        weight = torch.softmax(float(beta) * x, dim=0)
        values[:, j] = (weight * x).sum(0)
    return values

@torch.no_grad()
def predict_member(model, cache, mask, idx, dev, img_size, group=None, pool=None, starts=None, jitter=False, jitter_seed=SEED, return_public_frontier=False):
    group = GROUP if group is None else group
    pool = TTA_POOL if pool is None else pool
    starts = window_starts(cache.shape[2], group) if starts is None else list(starts)
    if not starts:
        raise ValueError('predict_member was given no windows to average over')
    target_idx = {t: j for j, t in enumerate(TARGETS)}
    unknown = (set(TTA_TARGET_POOL) | set(PUBLIC_FRONTIER_TARGET_POOL)) - set(target_idx)
    if unknown:
        raise ValueError(f'unknown target(s) in TTA_TARGET_POOL: {unknown}')
    jitter_gen = torch.Generator(device=dev)
    jitter_gen.manual_seed(int(jitter_seed) % (2 ** 63 - 1))
    model.eval()
    out, public_frontier_out, public_soft_out = ([], [], [])
    for b in range(0, len(idx), EVAL_BATCH):
        sel = idx[b:b + EVAL_BATCH]
        m = torch.from_numpy(mask[sel]).to(dev)
        win_probs, win_logits, win_original_probs = ([], [], [])
        for st in starts:
            rows = torch.from_numpy(np.ascontiguousarray(cache[sel, :, st:st + group])).to(dev)
            views = [rows] + ([augment(rows, generator=jitter_gen)] if jitter else [])
            view_probs, view_logits = ([], [])
            for view in views:
                with torch.autocast('cuda', enabled=dev.type == 'cuda'):
                    z = model(view, m, img_size).float()
                if dev.type == 'cuda' and not bool(torch.isfinite(z).all()):
                    rsna_event('dino_legacy_fp16_nonfinite_retry_fp32')
                    with torch.autocast('cuda', enabled=False):
                        z = model(view, m, img_size).float()
                view_logits.append(z)
                view_probs.append(torch.sigmoid(z))
            win_logits.append(torch.stack(view_logits).mean(0))
            win_probs.append(torch.stack(view_probs).mean(0))
            win_original_probs.append(view_probs[0])
        probs = torch.stack(win_probs)
        logits = torch.stack(win_logits)
        original_probs = torch.stack(win_original_probs)
        v = torch.sigmoid(logits.mean(0)) if pool == 'logit' else probs.mean(0)
        v = apply_target_window_pool(v, probs, logits, original_probs, TTA_TARGET_POOL, target_idx)
        out.append(v.cpu().numpy())
        if return_public_frontier:
            public_v = apply_target_window_pool(original_probs.mean(0), original_probs, logits, original_probs, PUBLIC_FRONTIER_TARGET_POOL, target_idx)
            public_frontier_out.append(public_v.cpu().numpy())
            public_soft = legacy_fold_soft_window_pool(original_probs, target_idx)
            public_soft_out.append(public_soft.cpu().numpy())
    primary = np.concatenate(out) if out else np.zeros((0, len(TARGETS)), np.float32)
    if not return_public_frontier:
        return primary
    public_frontier = np.concatenate(public_frontier_out) if public_frontier_out else np.zeros((0, len(TARGETS)), np.float32)
    public_soft = np.concatenate(public_soft_out) if public_soft_out else np.zeros((0, len(TARGETS)), np.float32)
    return (primary, public_frontier, public_soft)
BUILD_LOCK = threading.Lock()
STATE_LOCK = threading.Lock()
LEGACY_BUNDLE_FILE = 'rsna_20260807_v1.pt'
LEGACY_WEIGHT = 0.5

def find_legacy_bundle():
    candidates = [
        Path('/kaggle/input/datasets/pilkwang/rsna-knee-weights') / LEGACY_BUNDLE_FILE,
        Path('/kaggle/input/rsna-knee-weights') / LEGACY_BUNDLE_FILE,
    ]
    hits = [path for path in candidates if path.is_file()]
    if len(hits) > 1:
        raise WeightsError(f'ambiguous legacy bundle: {hits}')
    return hits[0] if hits else None

def legacy_group_members():
    return {}

def _run_member(path, m, dev, Cte, Mte, idx, starts, jitter):
    t0 = time.time()
    with BUILD_LOCK:
        if 'state' in m:
            raise WeightsError(f"{m['id']}: inline legacy state is forbidden")
        ck = torch.load(Path(path) / m['file'], map_location='cpu', weights_only=False)
        state, fp = (ck['model'], ck.get('fingerprint'))
        if fp is None:
            raise WeightsError(f"{m['id']}: stored fingerprint is required")
        model = build_model(int(m['config']['unfreeze_last']), variant=m['config']['variant'], pool=m['config'].get('pool', 'cls_mean'), prior=bool(m['config'].get('prior', False))).to(dev)
        model.load_state_dict(state)
        check_fingerprint(model, dev, IMG, fp, tag=f"{m['id']}: ")
        global _DINOV2_MATCHED_MEMBERS
        _DINOV2_MATCHED_MEMBERS += 1
    t_ready = time.time()
    jitter_seed = SEED + int(hashlib.sha256(str(m['id']).encode()).hexdigest()[:8], 16)
    public_member = 'state' not in m
    predicted = predict_member(model, Cte, Mte, idx, dev, IMG, starts=starts, jitter=jitter, jitter_seed=jitter_seed, return_public_frontier=public_member)
    if public_member:
        p, public_p, public_soft = predicted
    else:
        p, public_p, public_soft = (predicted, None, None)
    if public_member:
        _ma_predict_single_extra(model, m, public_p, dev, starts)
    t_done = time.time()
    del model, state
    gc.collect()
    if dev.type == 'cuda':
        with torch.cuda.device(dev):
            torch.cuda.empty_cache()
    passes = len(starts) * (2 if jitter else 1)
    return (p, public_p, public_soft, (t_ready - t0, (t_done - t_ready) / max(passes, 1)))

def _combine(per_member):
    all_ids = sorted({s for m in per_member for s in m['ids']})
    pos = {s: i for i, s in enumerate(all_ids)}
    acc = np.zeros((len(all_ids), len(TARGETS)), np.float64)
    tot = np.zeros(len(TARGETS), np.float64)
    for m in per_member:
        target_weight = m.get('target_weight')
        w = np.asarray(target_weight if target_weight is not None else [float(m.get('weight', 1.0))] * len(TARGETS), dtype=np.float64)
        if w.shape != (len(TARGETS),) or np.any(w < 0):
            raise ValueError(f"invalid target weights for {m.get('id')}: {w}")
        rsna_finite(m['pred'],str(m['id'])+' retained raw',probability=True)
        r = pd.DataFrame(m['pred']).rank(pct=True).to_numpy()
        acc[[pos[s] for s in m['ids']]] += r * w[None, :]
        tot += w
    if np.any(tot <= 0):
        raise ValueError(f'at least one target has no ensemble vote: {tot}')
    return (all_ids, acc / tot[None, :])

def combine_public_members_by_fold(per_member, pred_key='pred'):
    # Raw-average the four members within each fold, rank each fold,
    # then give all five folds equal weight.
    all_ids = sorted({study for member in per_member for study in member['ids']})
    position = {study: i for i, study in enumerate(all_ids)}
    groups = {}
    for i, member in enumerate(per_member):
        fold = member.get('fold')
        key = f'fold_{fold}' if fold is not None else f'member_{i}'
        groups.setdefault(key, []).append(member)
    fold_ranks, diagnostics = ([], [])
    for key, members_in_fold in sorted(groups.items()):
        matrices = []
        for member in members_in_fold:
            values = np.full((len(all_ids), len(TARGETS)), np.nan, np.float64)
            values[[position[study] for study in member['ids']]] = np.asarray(member[pred_key], np.float64)
            if np.isnan(values).any():
                raise WeightsError(f"{member.get('id')}: incomplete {pred_key} coverage")
            matrices.append(values)
        raw_fold_mean = np.mean(matrices, axis=0)
        fold_ranks.append(pd.DataFrame(raw_fold_mean).rank(method='average', pct=True).to_numpy(np.float64))
        diagnostics.append({'ensemble_group': key, 'members': len(members_in_fold)})
    if len(fold_ranks) != 5:
        raise WeightsError(f'legacy branch requires five folds, found {len(fold_ranks)}')
    return all_ids, np.mean(fold_ranks, axis=0), pd.DataFrame(diagnostics)

def blend_legacy_frontier_and_soft(frontier_rank, soft_rank):
    output = np.asarray(frontier_rank, np.float64).copy()
    for j, target in enumerate(TARGETS):
        alpha = float(LEGACY_FOLD_SOFTPOOL_ALPHA.get(target, 0.0))
        if alpha:
            output[:, j] = (1.0 - alpha) * frontier_rank[:, j] + alpha * soft_rank[:, j]
    return output

def infer_from_package(path, dev=None):
    man = json.loads((Path(path) / 'manifest.json').read_text())
    members = man['members']
    if len(members) != EXPECTED_DINOV2_MEMBERS:
        raise WeightsError(f'expected {EXPECTED_DINOV2_MEMBERS} DINOv2 members, found {len(members)}')
    log(f'weights package: {len(members)} member(s) from {path}; {len(DEVS)} device(s)')
    test_df = pd.read_csv(ROOT / 'test.csv')
    test_series = pd.read_csv(ROOT / 'test_series.csv')
    plane_map = dict(zip(test_series['SeriesInstanceUID'], test_series['Anatomical_Plane']))
    hte = annotate(walk('test_series'))
    log(f'test header pass: {len(hte)} series')
    groups = {}
    for m in members:
        groups.setdefault(m['pixel_group'], []).append(m)
    groups.update(legacy_group_members())
    per_member, public_frontier_members = ([], [])
    failures = []
    abort = threading.Event()
    est = {'fixed': None, 'win': None}

    def bank(m, ids, pred, starts, jitter, public_pred=None, public_soft=None):
        if not np.isfinite(pred).all():
            rsna_event('dino_member_nonfinite_neutral', member=str(m['id']), rows=int((~np.isfinite(pred)).reshape(len(pred), -1).any(axis=1).sum()))
            pred = np.where(np.isfinite(pred), pred, 0.5)
        if public_pred is not None and not np.isfinite(public_pred).all():
            rsna_event('dino_member_public_nonfinite_neutral', member=str(m['id']))
            public_pred = np.where(np.isfinite(public_pred), public_pred, 0.5)
        rsna_finite(pred, m['id']+' DINO raw', probability=True)
        if public_pred is not None: rsna_finite(public_pred,m['id']+' DINO retained raw',probability=True)
        rsna_save_predictions('dino_'+str(m['id']),ids,public_pred if public_pred is not None else pred,TARGETS)
        if float(np.std(pred)) < 1e-09:
            raise WeightsError(f"{m['id']}: degenerate predictions")
        with STATE_LOCK:
            per_member.append({'id': m['id'], 'fold': m.get('fold'), 'ids': ids, 'pred': pred, 'weight': m.get('weight', 1.0), 'target_weight': m.get('target_weight'), 'holdout': m.get('holdout')})
            if public_pred is not None and len(starts) == len(starts_full):
                if float(np.std(public_pred)) < 1e-09:
                    raise WeightsError(f"{m['id']}: degenerate public-frontier prediction")
                public_frontier_members.append({'id': m['id'], 'fold': m.get('fold'), 'ids': ids, 'pred': public_pred, 'soft_pred': public_soft})
            elif public_pred is not None:
                raise WeightsError(f"{m['id']}: incomplete public-frontier windows")
            log(f"  banked {m['id']} fold {m.get('fold', '?')} ({len(starts)} window(s); {len(per_member)} member(s)")
    for gi, (key, gm) in enumerate(groups.items(), 1):
        cfg = json.loads(key)
        adopt_config_globals(cfg)
        log(f"decode group {gi}/{len(groups)}: {cfg['img']}px x {cfg['slices']} slices, crop {cfg['crop_mm']} mm -> {len(gm)} member(s)")
        _ma_anchor_slots = pick_slots(hte, plane_map)
        _ma_side = lat_of(hte, 'test ')
        st_te, Cte, Mte = build_cache(_ma_anchor_slots, plane_map, _ma_side, f'test g{gi}')
        _ma_prepare_group(hte, test_series, _ma_anchor_slots, st_te, Cte, Mte, _ma_side, f'g{gi}')
        idx = np.arange(len(st_te))
        starts_full = window_starts(Cte.shape[2], GROUP)
        if len(groups) == 1 and _shared_prefix_eligible(gm):
            shared_results = _run_shared_dino_group(
                path,
                gm,
                Cte,
                Mte,
                idx,
                starts_full,
            )
            for member, predicted, jitter in shared_results:
                prediction, public, soft = predicted
                bank(
                    member,
                    st_te,
                    prediction,
                    starts_full,
                    jitter,
                    public,
                    soft,
                )
            _ma_release_group()
            rsna_release_pixels(Cte)
            del Cte, Mte, shared_results
            gc.collect()
            continue
        pending = sorted(gm, key=lambda m: -(m.get('holdout') or 0))
        left_after = sum((len(g) for j, (_, g) in enumerate(groups.items(), 1) if j > gi))

        def pop_next():
            with STATE_LOCK:
                if not pending:
                    return (None, None, False)
                rsna_deadline('DINO complete-member inference')
                starts, jit = starts_full, False
                return (pending.pop(0), starts, jit)

        def worker(dev):
            while not abort.is_set():
                m, starts, jit = pop_next()
                if m is None:
                    return
                try:
                    p, public_p, public_soft, (fs, ws) = _run_member(path, m, dev, Cte, Mte, idx, starts, jit)
                    with STATE_LOCK:
                        est['fixed'], est['win'] = (fs, ws)
                    bank(m, st_te, p, starts, jit, public_p, public_soft)
                except Exception as exc:
                    with STATE_LOCK:
                        failures.append((m['id'], str(dev), type(exc).__name__, str(exc)))
                    abort.set()
                    return
        threads = [threading.Thread(target=worker, args=(d,)) for d in DEVS]
        for t in threads:
            t.start()
        for t in threads:
            t.join()
        if failures:
            raise WeightsError(f'DINOv2 member failure; no fallback permitted: {failures[0]}')
        _ma_release_group()
        rsna_release_pixels(Cte)
        del Cte, Mte
        gc.collect()
    if _DINOV2_MATCHED_MEMBERS != EXPECTED_DINOV2_MEMBERS:
        raise WeightsError(f'fingerprint gate failed: {_DINOV2_MATCHED_MEMBERS} / {EXPECTED_DINOV2_MEMBERS}')
    if len(public_frontier_members) != EXPECTED_DINOV2_MEMBERS:
        raise WeightsError(f'public-frontier inference incomplete: {len(public_frontier_members)} / {EXPECTED_DINOV2_MEMBERS} members')
    log(f'DINOv2 fail-closed gate PASS: {_DINOV2_MATCHED_MEMBERS}/{EXPECTED_DINOV2_MEMBERS} fingerprints and members')
    _ma_capture_members(public_frontier_members)
    frontier_ids, frontier_acc = _combine(public_frontier_members)
    sub = write_submission(frontier_acc, frontier_ids, test_df, '_pipeline_stage.csv')
    log(f'submission.csv = exact no-jitter public-frontier rank mean of {len(public_frontier_members)} member(s); {sub.shape}')
    return sub

def adopt_config_globals(cfg):
    global IMG, CACHE_IMG, GROUP, CACHE_SLICES, N_GROUP, CROP_MM, SLICE_BAND, RULES
    CACHE_IMG = IMG = int(cfg['img'])
    GROUP = int(cfg['group'])
    CACHE_SLICES = int(cfg['slices'])
    N_GROUP = max(CACHE_SLICES // GROUP, 1)
    CROP_MM = float(cfg['crop_mm'])
    SLICE_BAND = tuple((float(x) for x in cfg['band']))
    rules = cfg.get('rules') or RULES_NATIVE
    unknown = {k: v for k, v in rules.items() if k not in RULES_NATIVE or v not in (RULES_NATIVE[k], RULES_LEGACY[k])}
    if unknown:
        raise WeightsError(f'the members record pixel rules this pipeline cannot reproduce: {unknown}')
    RULES = {**RULES_NATIVE, **rules}
    if [s[0] for s in SLOTS] != list(cfg['slots']):
        raise WeightsError(f"the members were fitted on slots {cfg['slots']} and this pipeline defines {[s[0] for s in SLOTS]}; a weight would be read against the wrong slot")


In [ ]:
def write_submission(pred, studies, test_df, path):
    sub = pd.DataFrame(pd.DataFrame(pred).rank(pct=True).values, columns=TARGETS)
    sub.insert(0, 'StudyInstanceUID', studies)
    sub = test_df[['StudyInstanceUID']].merge(sub, on='StudyInstanceUID', how='left')
    if sub[TARGETS].isna().any().any():
        rsna_event('submission_neutral_fill', studies=int(sub[TARGETS].isna().any(axis=1).sum()))
        sub[TARGETS] = sub[TARGETS].fillna(0.5)
    sub.to_csv(path, index=False)
    return sub

def _v37_validate_submission(path, test_df, tag):
    frame = pd.read_csv(path)
    expected = ['StudyInstanceUID'] + TARGETS
    if list(frame.columns) != expected:
        raise ValueError(f'{tag}: columns differ from the competition contract')
    if len(frame) != len(test_df) or not frame['StudyInstanceUID'].is_unique:
        raise ValueError(f'{tag}: row count or StudyInstanceUID uniqueness failed')
    if set(frame['StudyInstanceUID'].astype(str)) != set(test_df['StudyInstanceUID'].astype(str)):
        raise ValueError(f'{tag}: StudyInstanceUID set differs from test.csv')
    values = frame[TARGETS].to_numpy(np.float64)
    if not np.isfinite(values).all():
        raise ValueError(f'{tag}: non-finite prediction')
    return frame

def main():
    pkg = find_weights()
    if pkg is None:
        raise WeightsError('required public checkpoint manifest not found')
    infer_from_package(pkg, DEVS[0])
    _v37_validate_submission('_pipeline_stage.csv', pd.read_csv(ROOT / 'test.csv'), 'public frontier')


In [ ]:
# Retained-public-only extra-view inference: no unused TTA or soft-pooling branch.
# This only changes how the OPTIONAL acquisition is computed, not the anchor.
# Live first-batch numerical parity is required for every loaded model group.
@torch.inference_mode()
def _v2_shared_public(entries, cache, mask, idx, dev, img_size, group, starts):
    if not entries or not starts or len(idx) == 0:
        raise ValueError('Public-only inference needs models, rows and windows')
    for _, model, _ in entries: model.eval()
    target_idx = {name:j for j,name in enumerate(TARGETS)}
    # The retained public score must be independent of the unused jittered logits.
    if any(mode not in {'max','mean','original_mean','top2','top3'}
           for mode in PUBLIC_FRONTIER_TARGET_POOL.values()):
        raise RuntimeError('Public-only optimization does not support logit-based pooling')
    saved = {str(member['id']): [] for member, _, _ in entries}
    reference = entries[0][1]
    for lo in range(0, len(idx), EVAL_BATCH):
        selected = idx[lo:lo+EVAL_BATCH]
        slot_mask = torch.from_numpy(np.ascontiguousarray(mask[selected])).to(dev)
        windows = {str(member['id']): [] for member, _, _ in entries}
        for start in starts:
            rows = torch.from_numpy(np.ascontiguousarray(
                cache[selected, :, start:start+group])).to(dev)
            def forward(amp):
                with torch.autocast('cuda', enabled=amp):
                    common = _shared_dino_prefix(reference, rows, img_size)
                    return {str(m['id']): _shared_dino_tail(model, common, len(selected),
                        cache.shape[1], slot_mask).float() for m, model, _ in entries}
            logits = forward(dev.type == 'cuda')
            if dev.type == 'cuda' and not all(bool(torch.isfinite(z).all()) for z in logits.values()):
                rsna_event('v2_extra_public_fp32_retry', start=int(start))
                logits = forward(False)
            if not all(bool(torch.isfinite(z).all()) for z in logits.values()):
                raise RuntimeError('Nonfinite extra-acquisition logits after retry')
            for key, z in logits.items(): windows[key].append(torch.sigmoid(z))
        for key, window_values in windows.items():
            probabilities = torch.stack(window_values)
            value = apply_target_window_pool(probabilities.mean(0), probabilities, None,
                probabilities, PUBLIC_FRONTIER_TARGET_POOL, target_idx)
            saved[key].append(value.cpu().numpy())
    return {key: np.concatenate(chunks) for key,chunks in saved.items()}


def _v2_public_parity(optimized: dict, reference: dict) -> dict:
    if set(optimized) != set(reference):
        raise RuntimeError('Extra-view optimization changed model membership')
    results = {}
    for key, got in optimized.items():
        want = reference[key][1]
        if got.shape != want.shape or not np.isfinite(got).all():
            raise RuntimeError('Extra-view optimized output contract failed: ' + key)
        delta = float(np.max(np.abs(got-want)))
        # Tight FP32/AMP output comparison, not a clinical equivalence criterion.
        if not np.allclose(got, want, rtol=1.e-6, atol=2.e-7):
            raise RuntimeError('Extra-view optimization differs from retained public path: '+key)
        results[key] = {'exact':bool(np.array_equal(got,want)), 'max_abs_difference':delta}
    return results


# Additional acquisition view: original model inputs/predictions stay immutable.
import threading as _ma_threading
_MA_LOCK = _ma_threading.RLock()
_MA_ACTIVE = None
_MA_ALTERNATES = {}
_MA_SELECTION_AUDIT = []
_MA_GROUP_RECEIPTS = []
_MA_ANCHOR_MEMBERS = []
_MA_EXTRA_SECONDS = {}

def _ma_prepare_group(headers, metadata, anchor_slots, ids, pixels, masks, side_map, tag):
    global _MA_ACTIVE
    if _MA_ACTIVE is not None:
        raise RuntimeError('Previous alternate cache was not released')
    if MASTER['acquisition_alpha'] == 0:
        return
    proposals, audit = ma_select_alternates(
        headers, metadata, anchor_slots, list(SLOTS), ids,
        max_fraction=MASTER['acquisition_max_fraction'],
        max_studies=MASTER['acquisition_max_studies'],
        max_slots=MASTER['acquisition_max_slots'],
        allow_secondary=MASTER['allow_secondary_acquisition'],
        min_slices=MASTER['minimum_alternate_slices'],
        min_relative_slices=MASTER['minimum_relative_slices'],
        exclude_duplicate_context=MASTER['exclude_duplicate_context'])
    for r in audit: r['pixel_group'] = str(tag)
    _MA_SELECTION_AUDIT.extend(audit)
    picked = np.asarray([i for i, u in enumerate(ids) if u in proposals], dtype=np.int64)
    if not len(picked):
        _MA_GROUP_RECEIPTS.append(dict(group=tag, selected=0, usable=0, extra_slots=0))
        log(f'[MAST] {tag}: no eligible alternative; anchor pixels untouched')
        return
    alternate = rsna_array((len(picked), *pixels.shape[1:]), 'MAST_' + str(tag))
    for lo in range(0, len(picked), 8):
        alternate[lo:lo+8] = pixels[picked[lo:lo+8]]
    alternate_mask = np.asarray(masks[picked]).copy()
    used = np.zeros(len(picked), bool)
    jobs = []
    for local, original in enumerate(picked):
        uid = ids[int(original)]
        for k, (slot_name, plane, _, _) in enumerate(SLOTS):
            if slot_name in proposals[uid]:
                jobs.append((local, k, uid, plane, proposals[uid][slot_name].copy(deep=True)))
    def read_alternate(job):
        local, k, uid, plane, record = job
        try:
            key = (str(Path(record['dir']).resolve()), RULES['order'], tuple(sorted(record['files'])))
            saved = _RSNA_ORDER_MEMO.get(key)
            if saved is None:
                ordered, valid = order_slices(record)
                if len(ordered) != len(record['files']) or set(ordered) != set(record['files']):
                    raise ValueError('Alternate ordering changed the file set')
                # Do not add questionable-order evidence to a healthy anchor.
                if not valid: raise ValueError('Alternate has ambiguous geometric order')
                _RSNA_ORDER_MEMO[key] = (tuple(ordered), bool(valid))
            else:
                ordered, valid = saved
                if not valid: raise ValueError('Alternate has a cached ordering fallback')
            record['ordered'] = list(ordered)
            image = read_slot(record, CACHE_SLICES, IMG)
            if image is None: raise ValueError('No decodable alternate pixels')
            image = normalise_laterality(image, plane, side_map.get(uid)).numpy()
            if image.shape != tuple(pixels.shape[2:]):
                raise ValueError('Alternate shape violates the checkpoint pixel contract')
            if not np.isfinite(image).all(): raise ValueError('Nonfinite alternate image')
            if image.max() == image.min(): raise ValueError('Constant optional acquisition image')
            return local, k, image, None
        except Exception as exc:
            # This is an unavailable extra acquisition, not a crashed model.
            return local, k, None, dict(study_uid=str(uid), series_uid=str(record.SeriesInstanceUID),
                error=f'{type(exc).__name__}: {exc}')
    extra_slots = 0
    with ThreadPoolExecutor(max_workers=min(PIX_THREADS, max(1, len(jobs)))) as pool:
        for local, k, image, error in pool.map(read_alternate, jobs):
            if image is None:
                rsna_event('mast_extra_acquisition_unavailable', **error)
                continue
            # Unchanged context slots are byte-identical to the anchor cache.
            alternate[local, k] = image
            alternate_mask[local, k] = 1.
            used[local] = True
            extra_slots += 1
    if isinstance(alternate, np.memmap): alternate.flush()
    active = np.flatnonzero(used)
    record = dict(group=tag, selected=len(picked), usable=len(active), extra_slots=extra_slots,
                  array_bytes=int(alternate.nbytes), ensemble_members_total=20,
                  alpha=MASTER['acquisition_alpha'],
                  uid_order=[ids[int(i)] for i in picked[active]])
    _MA_GROUP_RECEIPTS.append(record)
    if not len(active):
        rsna_release_pixels(alternate)
        log(f'[MAST] {tag}: optional acquisitions unavailable; anchor retained')
        return
    _MA_ACTIVE = dict(cache=alternate, mask=alternate_mask, local_indices=active,
                      full_indices=picked[active], ids=list(ids), tag=str(tag))
    log(f'[MAST] {tag}: {len(active)}/{len(ids)} studies; {extra_slots} new slot acquisitions; '
        f'original pixels preserved; extra passes use loaded models')

def _ma_release_group():
    global _MA_ACTIVE
    if _MA_ACTIVE is not None:
        rsna_release_pixels(_MA_ACTIVE['cache'])
        _MA_ACTIVE = None

def _ma_bank_alternate(member, raw_anchor, raw_alternate, elapsed):
    context = _MA_ACTIVE
    if context is None: raise RuntimeError('Missing alternate-view context')
    name = str(member['id'])
    raw_anchor = ma_check_prob(raw_anchor, name + ' anchor raw')
    raw_alternate = ma_check_prob(raw_alternate, name + ' alternate raw')
    index = context['full_indices']
    if raw_alternate.shape != (len(index), 12) or raw_anchor.shape != (len(context['ids']),12):
        raise RuntimeError('Alternate member UID/shape mismatch')
    entry = dict(indices=index.copy(), alternate=raw_alternate.copy(),
                 anchor_sha256=hashlib.sha256(np.ascontiguousarray(raw_anchor).tobytes()).hexdigest(),
                 ids=list(context['ids']), group=context['tag'])
    with _MA_LOCK:
        if name in _MA_ALTERNATES: raise RuntimeError('Duplicate alternate model member: ' + name)
        _MA_ALTERNATES[name] = entry
        _MA_EXTRA_SECONDS[name] = float(elapsed)

# Shared-prefix path: one extra bounded cohort, with the existing models resident.
_MA_BASE_SHARED_PREDICT = _predict_shared_dino_members

def _predict_shared_dino_members(entries, cache, mask, idx, dev, img_size, group, starts):
    anchor = _MA_BASE_SHARED_PREDICT(entries, cache, mask, idx, dev, img_size, group, starts)
    context = _MA_ACTIVE
    if context is not None:
        if not np.array_equal(idx, np.arange(len(context['ids']))):
            raise RuntimeError('Alternate shared-prefix branch expects the complete anchor cohort')
        rsna_deadline('MAST additional acquisition inference')
        began = time.monotonic()
        extra = _v2_shared_public(entries, context['cache'], context['mask'],
            context['local_indices'], dev, img_size, group, starts)
        # Compare against the ORIGINAL retained-public computation on one real batch.
        probe_count = min(EVAL_BATCH,len(context['local_indices']))
        probe_entries = [(member,model,False) for member,model,_ in entries]
        reference = _MA_BASE_SHARED_PREDICT(probe_entries, context['cache'], context['mask'],
            context['local_indices'][:probe_count], dev, img_size, group, starts)
        parity = _v2_public_parity({key:value[:probe_count] for key,value in extra.items()}, reference)
        rsna_event('v2_extra_public_parity', group=context['tag'],device=str(dev),results=parity)
        elapsed = time.monotonic() - began
        for member, _, _ in entries:
            _ma_bank_alternate(member, anchor[member['id']][1], extra[str(member['id'])],
                               elapsed/max(1,len(entries)))
        del extra, reference
    return anchor

def _ma_predict_single_extra(model, member, raw_anchor, dev, starts):
    context = _MA_ACTIVE
    if context is None: return
    rsna_deadline('MAST single-member additional acquisition')
    began = time.monotonic()
    extra = predict_member(model, context['cache'], context['mask'], context['local_indices'],
                           dev, IMG, starts=starts, jitter=False, return_public_frontier=True)
    _ma_bank_alternate(member, raw_anchor, extra[1], time.monotonic()-began)

# Preserve pure prediction functions before A5 temporarily replaces notebook globals.
_MA_A5_BLEND_FN = None

def _ma_capture_members(public_members):
    global _MA_ANCHOR_MEMBERS
    _MA_ANCHOR_MEMBERS = [{**m, 'ids':list(map(str,m['ids'])), 'pred':m['pred'].copy()}
                          for m in public_members]
    if len(_MA_ANCHOR_MEMBERS) != 20:
        raise RuntimeError('MAST requires all twenty anchor DINO members')
    expected = {str(m['id']) for m in _MA_ANCHOR_MEMBERS}
    if set(_MA_ALTERNATES) - expected:
        raise RuntimeError('Unknown additional-acquisition member')
    rsna_json('/kaggle/working/diagnostics/mast_acquisitions.json', {
        'method':'MAST-v2-repair-only', 'score_evidence':'not yet measured',
        'settings':MASTER, 'groups':_MA_GROUP_RECEIPTS,
        'selections':_MA_SELECTION_AUDIT, 'extra_compute_owner_seconds':_MA_EXTRA_SECONDS})
    if _MA_ALTERNATES:
        target = Path('/kaggle/working/diagnostics/mast_members'); target.mkdir(exist_ok=True)
        for name, rec in _MA_ALTERNATES.items():
            safe = hashlib.sha256(name.encode()).hexdigest()[:20]
            np.savez_compressed(target/(safe+'.npz'), member_id=np.asarray(name),
                study_uids=np.asarray(rec['ids'], dtype=str), indices=rec['indices'],
                alternate_probabilities=rec['alternate'], labels=np.asarray(MA_LABELS, dtype=str))


In [ ]:
rsna_phase('dinov2', 'START')
main()
log('done')

_speed_dino_templates.clear()
gc.collect()

rsna_phase('dinov2', 'COMPLETE')


## 6. Original five-fold A5 reader

In [ ]:
if globals().get('_DINOV2_MATCHED_MEMBERS') != 20:
    raise RuntimeError('DINOv2 20/20 fingerprint gate did not pass')
rsna_phase('a5', 'START')
_A5_SAVED = dict(globals())
import gc, os, time, warnings
from concurrent.futures import ThreadPoolExecutor as A5DecodePool, as_completed
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import pydicom
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
warnings.filterwarnings('ignore')
cv2.setNumThreads(1)
CROP_MM = 130.0
SIZE = 336
SLICE_BAND = (0.12, 0.88)
N_SLICE = 16
INTENSITY = 'slice'
SLOTS = [('Sagittal', 1), ('Sagittal', 0), ('Coronal', 1), ('Coronal', 0), ('Axial', 1), ('Axial', 0)]
N_SLOT = len(SLOTS)
LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']

def _one_a5_root(tag, candidates, marker):
    hits = [Path(path) for path in candidates if Path(path).is_dir() and (Path(path) / marker).exists()]
    if len(hits) != 1:
        raise RuntimeError(f'{tag}: expected one direct root, found {hits}')
    return hits[0]

COMP = Path(ROOT)
CKPT = _one_a5_root('A5 weights', [
    '/kaggle/input/datasets/mattiaangeli/knee-mri-fold-weights',
    '/kaggle/input/knee-mri-fold-weights',
], 'm_f0.pt')
assert COMP is not None, 'competition data not attached'
assert CKPT is not None, 'fold weights not attached'
assert (COMP / 'sample_submission.csv').exists(), f'no competition data at {COMP}'
assert list(CKPT.glob('*_f*.pt')), f'no checkpoints at {CKPT}'
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'


In [ ]:
SERIES_ROOT = COMP / 'test_series'
if not SERIES_ROOT.is_dir():
    raise FileNotFoundError('Missing test_series; train-series fallback is forbidden')
print('series root:', SERIES_ROOT)

def ordered_files(sdir, cap=64):
    # Preserve the parent's InstanceNumber ordering; remove pre-sort truncation.
    # The unused cap argument remains only for call compatibility.
    keyed=[]; unreadable=[]
    for f in sorted(sdir.glob('*.dcm')):
        try:
            ds=pydicom.dcmread(str(f),stop_before_pixels=True)
            key=int(ds.InstanceNumber)
        except Exception as exc:
            unreadable.append(str(f)); continue
        keyed.append((key,str(f)))
    if unreadable:
        rsna_event('a5_order_unreadable_dropped',series=str(sdir),unreadable=len(unreadable),readable=len(keyed),examples=unreadable[:3])
    return [f for _,f in sorted(keyed)]

def series_side(path):
    try:
        return float(pydicom.dcmread(path, stop_before_pixels=True).ImagePositionPatient[0])
    except Exception:
        return 0.0

def read_crop(path):
    try:
        ds = pydicom.dcmread(path)
        arr = ds.pixel_array.astype(np.float32)
    except Exception:
        return None
    try:
        ps = float(ds.PixelSpacing[0])
    except Exception:
        ps = CROP_MM / max(arr.shape)
    half = int(round(CROP_MM / ps / 2))
    cy, cx = (arr.shape[0] // 2, arr.shape[1] // 2)
    y0, y1 = (max(0, cy - half), min(arr.shape[0], cy + half))
    x0, x1 = (max(0, cx - half), min(arr.shape[1], cx + half))
    crop = arr[y0:y1, x0:x1]
    return None if crop.size == 0 else crop

def window(crop, lo, hi, flip):
    c = np.clip((crop - lo) / max(hi - lo, 1e-06), 0, 1)
    img = cv2.resize(c, (SIZE, SIZE), interpolation=cv2.INTER_AREA)
    return img[:, ::-1].copy() if flip else img

def render(path, flip):
    crop = read_crop(path)
    if crop is None:
        return None
    lo, hi = np.percentile(crop[::4, ::4], [1, 99])
    return window(crop, lo, hi, flip)

def build_study(args):
    idx,study,recs=args
    out=np.zeros((N_SLOT,N_SLICE,SIZE,SIZE),np.uint8);mask=np.zeros(N_SLOT,np.uint8)
    rows=pd.DataFrame(recs)
    if not len(rows):
        rsna_event('a5_no_metadata',study=str(study)); return idx,out,mask
    for s_i,(plane,fs) in enumerate(SLOTS):
        sub=rows[(rows.Anatomical_Plane==plane)&(rows.Fat_Suppression==fs)]
        if sub.empty: continue  # Actual acquisition absence, not a decoding failure.
        series=str(sub.iloc[0].SeriesInstanceUID)
        files=ordered_files(SERIES_ROOT/str(study)/series)
        if not files:
            rsna_event('a5_slot_no_files',study=str(study),series=series); continue
        flip=plane!='Sagittal' and series_side(files[0])<0
        i0=int(round(SLICE_BAND[0]*(len(files)-1)));i1=int(round(SLICE_BAND[1]*(len(files)-1)))
        avail=list(range(i0,i1+1))
        if len(avail)>=N_SLICE:
            picks=[avail[int(round(t))] for t in np.linspace(0,len(avail)-1,N_SLICE)];off=0
        else:picks,off=avail,(N_SLICE-len(avail))//2
        rendered=0;errors=[]
        if INTENSITY=='series':
            crops=[read_crop(files[p]) for p in picks];got=[x for x in crops if x is not None]
            if not got:
                rsna_event('a5_slot_all_crops_failed',study=str(study),series=series); continue
            samp=np.concatenate([x[::4,::4].ravel() for x in got]);lo_,hi_=np.percentile(samp,[1,99])
            for c,x in enumerate(crops):
                if x is None:x=read_crop(files[min(len(files)-1,picks[c]+1)])
                if x is None:errors.append(files[picks[c]]);continue
                img=window(x,lo_,hi_,flip)
                rsna_finite(img,'A5 rendered pixels')
                out[s_i,off+c]=(img*255).astype(np.uint8);rendered+=1
        else:
            for c,p in enumerate(picks):
                img=render(files[p],flip)
                if img is None:img=render(files[min(len(files)-1,p+1)],flip)
                if img is None:errors.append(files[p]);continue
                rsna_finite(img,'A5 rendered pixels')
                out[s_i,off+c]=(img*255).astype(np.uint8);rendered+=1
        if errors or rendered!=len(picks):
            rsna_event('a5_decode_failure',study=str(study),series=series,failed=errors,rendered=int(rendered),wanted=int(len(picks)))
            if not rendered: continue
        mask[s_i]=rendered
    if not mask.any():
        rsna_event('a5_empty_study',study=str(study))
        print(f'[a5] {study}: no acquired slots; all-zero mask kept (parent policy)', flush=True)
    return idx,out,mask
sub_df = pd.read_csv(COMP / 'sample_submission.csv',dtype={'StudyInstanceUID':str})
ser_csv = pd.read_csv(COMP / 'test_series.csv',dtype={'StudyInstanceUID':str,'SeriesInstanceUID':str})
if set(sub_df.StudyInstanceUID)!=set(_RSNA_TEST_IDS):
    raise RuntimeError('A5 test/sample UID mismatch')
ser_csv = ser_csv.loc[:, ~ser_csv.columns.duplicated()]
studies = sub_df.StudyInstanceUID.tolist()
by = {s: g.to_dict('records') for s, g in ser_csv[ser_csv.StudyInstanceUID.isin(set(studies))].groupby('StudyInstanceUID')}
print(f'{len(studies):,} test studies, {len(by):,} with series metadata')


In [ ]:
N_SLOT_TYPES, MASK_IDX = (6, 0)

def segment_softmax(scores, sidx, B):
    T, K = scores.shape
    idx = sidx.unsqueeze(1).expand(-1, K)
    m = torch.full((B, K), float('-inf'), device=scores.device, dtype=scores.dtype)
    m = m.scatter_reduce(0, idx, scores, reduce='amax', include_self=True)
    e = (scores - m[sidx]).exp()
    s = torch.zeros(B, K, device=scores.device, dtype=scores.dtype).index_add_(0, sidx, e)
    return e / s[sidx].clamp(min=1e-06)

class MeanMaxPool(nn.Module):

    def forward(self, f, sidx, B, slot=None, return_attn=False):
        D = f.shape[1]
        cnt = torch.zeros(B, device=f.device, dtype=f.dtype).index_add_(0, sidx, torch.ones(f.shape[0], device=f.device, dtype=f.dtype))
        mean = torch.zeros(B, D, device=f.device, dtype=f.dtype).index_add_(0, sidx, f)
        mean = mean / cnt.clamp(min=1).unsqueeze(1)
        mx = torch.full((B, D), -10000.0, device=f.device, dtype=f.dtype)
        mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), f, reduce='amax', include_self=True)
        return (torch.cat([mean, mx], 1), None)

class LabelAttentionPool(nn.Module):

    def __init__(self, d, n_labels=12, n_heads=4, slot_bias=True):
        super().__init__()
        self.d, self.k, self.h = (d, n_labels, n_heads)
        self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)
        self.key, self.val = (nn.Linear(d, d), nn.Linear(d, d))
        self.slot_bias = nn.Parameter(torch.zeros(n_labels, N_SLOT_TYPES + 1)) if slot_bias else None

    def forward(self, f, sidx, B, slot=None, return_attn=False):
        scores = self.key(f) @ self.q.t() / self.d ** 0.5
        if self.slot_bias is not None and slot is not None:
            scores = scores + self.slot_bias.t()[slot]
        a = segment_softmax(scores, sidx, B)
        out = torch.zeros(B, self.k, self.d, device=f.device, dtype=f.dtype)
        out = out.index_add_(0, sidx, a.unsqueeze(-1) * self.val(f).unsqueeze(1))
        return (out, a)

class TokenXAttnPool(nn.Module):

    def __init__(self, d, n_labels=12, n_heads=6, dropout=0.2):
        super().__init__()
        self.d, self.k = (d, n_labels)
        self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)
        self.slot_emb = nn.Embedding(N_SLOT_TYPES + 1, d, padding_idx=0)
        self.kv_norm = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)

    def forward(self, tok, sidx, B, slot=None, return_attn=False):
        T, N, D = tok.shape
        cnt = torch.bincount(sidx, minlength=B)
        S = int(cnt.max().item())
        starts = torch.cumsum(cnt, 0) - cnt
        pos = torch.arange(T, device=tok.device) - starts[sidx]
        kv = tok + self.slot_emb(slot).unsqueeze(1)
        pad = tok.new_zeros(B, S, N, D)
        pad[sidx, pos] = kv
        keep = torch.zeros(B, S, dtype=torch.bool, device=tok.device)
        keep[sidx, pos] = True
        kpm = ~keep.repeat_interleave(N, dim=1)
        pad = self.kv_norm(pad.reshape(B, S * N, D))
        q = self.q.unsqueeze(0).expand(B, -1, -1)
        att, w = self.attn(q, pad, pad, key_padding_mask=kpm, need_weights=return_attn, average_attn_weights=True)
        cls = tok[:, 0]
        mean = torch.zeros(B, D, device=tok.device, dtype=tok.dtype).index_add_(0, sidx, cls) / cnt.clamp(min=1).unsqueeze(1)
        mx = torch.full((B, D), -10000.0, device=tok.device, dtype=tok.dtype)
        mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), cls, reduce='amax', include_self=True)
        base = torch.cat([mean, mx], 1).unsqueeze(1).expand(-1, self.k, -1)
        return (torch.cat([att, base], -1), w)

class ViTSlotToken(nn.Module):

    def __init__(self, vit, n_cat, dim=None):
        super().__init__()
        self.vit = vit
        d = dim or vit.embed_dim
        self.tok = nn.Embedding(n_cat + 1, d, padding_idx=MASK_IDX)
        self.num_features = vit.num_features
        self._orig_prefix = getattr(vit, 'num_prefix_tokens', 1)
        vit.num_prefix_tokens = self._orig_prefix + 1
        for blk in vit.blocks:
            a = getattr(blk, 'attn', None)
            if a is not None and hasattr(a, 'num_prefix_tokens'):
                a.num_prefix_tokens = a.num_prefix_tokens + 1

    @staticmethod
    def _maybe(mod, x):
        return x if mod is None else mod(x)

    def forward_features(self, x, cat):
        v = self.vit
        x = v.patch_embed(x)
        pos = v._pos_embed(x)
        rope = None
        if isinstance(pos, tuple):
            x, rope = pos
        else:
            x = pos
        x = self._maybe(getattr(v, 'patch_drop', None), x)
        x = self._maybe(getattr(v, 'norm_pre', None), x)
        npt = self._orig_prefix
        tok = self.tok(cat).unsqueeze(1)
        x = torch.cat([x[:, :npt], tok, x[:, npt:]], dim=1)
        if rope is not None:
            if getattr(v, 'rope_mixed', False):
                for i, blk in enumerate(v.blocks):
                    x = blk(x, rope=rope[i])
            else:
                for blk in v.blocks:
                    x = blk(x, rope=rope)
        else:
            x = v.blocks(x)
        return v.norm(x)

    def forward_head(self, x, pre_logits=True):
        return self.vit.forward_head(x, pre_logits=pre_logits)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

class _GatedDepthBlock(nn.Module):

    def __init__(self, n_slice, dropout=0.0, ls_init=0.1):
        super().__init__()
        self.norm = nn.GroupNorm(1, n_slice)
        self.v = nn.Conv2d(n_slice, n_slice, 1)
        self.g = nn.Conv2d(n_slice, n_slice, 1)
        self.out = nn.Conv2d(n_slice, n_slice, 1)
        self.gamma = nn.Parameter(torch.full((n_slice, 1, 1), ls_init))
        self.drop = nn.Dropout2d(dropout) if dropout else nn.Identity()

    def forward(self, x):
        z = self.norm(x)
        return x + self.gamma * self.drop(self.out(self.v(z) * F.silu(self.g(z))))

class DepthCompress(nn.Module):

    def __init__(self, n_slice=16, out_ch=3, depth=1, dropout=0.0, ls_init=0.1, imagenet=True, proj_noise=0.25):
        super().__init__()
        self.imagenet = imagenet
        self.blocks = nn.ModuleList([_GatedDepthBlock(n_slice, dropout, ls_init) for _ in range(depth)])
        self.proj = nn.Conv2d(n_slice, out_ch, 1, bias=True)
        if imagenet:
            self.register_buffer('mu', torch.tensor(IMAGENET_MEAN).view(1, -1, 1, 1))
            self.register_buffer('sd', torch.tensor(IMAGENET_STD).view(1, -1, 1, 1))

    def forward(self, x):
        keep = (x.amax(dim=1, keepdim=True) > 0).to(x.dtype)
        z = x
        for b in self.blocks:
            z = b(z)
        z = self.proj(z)
        if self.imagenet:
            z = (z - self.mu.to(z.dtype)) / self.sd.to(z.dtype)
        return z * keep
N_PLANE, N_CONTRAST = (3, 2)
_PLANE_OF = lambda s: torch.clamp(s - 1, 0, 5) // 2
_CONTRAST_OF = lambda s: torch.clamp(s - 1, 0, 5) % 2

class SlotDepthMixer(nn.Module):

    def __init__(self, n_slice=16, ksize=5, alpha_max=0.25):
        super().__init__()
        self.n_slice, self.ksize, self.r = (n_slice, ksize, ksize // 2)
        self.alpha_max = alpha_max
        b = torch.tensor([1.0, 4.0, 6.0, 4.0, 1.0])
        self.register_buffer('base', b.log()[self.r:])
        n_u = self.r + 1
        self.shared = nn.Parameter(torch.zeros(n_u))
        self.plane_k = nn.Parameter(torch.zeros(N_PLANE, n_u))
        self.contrast_k = nn.Parameter(torch.zeros(N_CONTRAST, n_u))
        self.g0 = nn.Parameter(torch.zeros(()))
        self.gate_p = nn.Parameter(torch.zeros(N_PLANE))
        self.gate_c = nn.Parameter(torch.zeros(N_CONTRAST))
        idx = torch.arange(n_slice)
        self.register_buffer('off', idx[None, :] - idx[:, None])

    def kernel(self, slot):
        p, c = (_PLANE_OF(slot), _CONTRAST_OF(slot))
        half = self.base + self.shared + self.plane_k[p] + self.contrast_k[c]
        full = torch.cat([half.flip(-1)[..., :self.r], half], dim=-1)
        return F.softmax(full, dim=-1)

    def alpha(self, slot):
        p, c = (_PLANE_OF(slot), _CONTRAST_OF(slot))
        return self.alpha_max * torch.tanh(self.g0 + self.gate_p[p] + self.gate_c[c])

    def forward(self, x, slot, vmask):
        T, S, H, W = x.shape
        if vmask is None:
            raise ValueError('stem=mixer requires the padding mask')
        k = self.kernel(slot)
        v = vmask.to(k.dtype)
        d = self.off + self.r
        inb = (d >= 0) & (d < self.ksize)
        kk = k[:, d.clamp(0, self.ksize - 1)] * inb
        M = kk * v[:, None, :]
        den = M.sum(-1, keepdim=True)
        eye = torch.eye(S, device=x.device, dtype=M.dtype).expand(T, S, S)
        ok = (den > 1e-06) & v[:, :, None].bool()
        M = torch.where(ok, M / den.clamp(min=1e-06), eye)
        a = self.alpha(slot)[:, None, None]
        Aop = ((1.0 - a) * eye + a * M).to(x.dtype)
        if x.is_contiguous(memory_format=torch.channels_last) and (not x.is_contiguous()):
            y = torch.bmm(x.permute(0, 2, 3, 1).reshape(T, H * W, S), Aop.transpose(1, 2))
            return y.reshape(T, H, W, S).permute(0, 3, 1, 2)
        return torch.bmm(Aop, x.reshape(T, S, H * W)).reshape(T, S, H, W)

def _seg_mean_max(v, sidx, B):
    D = v.shape[1]
    cnt = torch.zeros(B, device=v.device, dtype=v.dtype).index_add_(0, sidx, torch.ones(v.shape[0], device=v.device, dtype=v.dtype))
    mean = torch.zeros(B, D, device=v.device, dtype=v.dtype).index_add_(0, sidx, v)
    mean = mean / cnt.clamp(min=1).unsqueeze(1)
    mx = torch.full((B, D), -10000.0, device=v.device, dtype=v.dtype)
    mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), v, reduce='amax', include_self=True)
    return torch.cat([mean, mx], 1)

def _pad_kv(x, sidx, B, norm):
    T, P, D = x.shape
    cnt = torch.bincount(sidx, minlength=B)
    S = int(cnt.max().item())
    starts = torch.cumsum(cnt, 0) - cnt
    pos = torch.arange(T, device=x.device) - starts[sidx]
    pad = x.new_zeros(B, S, P, D)
    pad[sidx, pos] = x
    keep = torch.zeros(B, S, dtype=torch.bool, device=x.device)
    keep[sidx, pos] = True
    return (norm(pad.reshape(B, S * P, D)), ~keep.repeat_interleave(P, dim=1))

class _GatedDelta(nn.Module):

    def __init__(self, d, n_labels, n_heads, dropout):
        super().__init__()
        self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)
        self.kv_norm = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)
        self.d_norm = nn.LayerNorm(d)
        self.dw = nn.Parameter(torch.randn(n_labels, d) * (1.0 / d ** 0.5))
        self.db = nn.Parameter(torch.zeros(n_labels))
        self.gate = nn.Parameter(torch.zeros(n_labels))

    def delta(self, pat, sidx, B, return_attn):
        kv, kpm = _pad_kv(pat, sidx, B, self.kv_norm)
        q = self.q.unsqueeze(0).expand(B, -1, -1)
        att, w = self.attn(q, kv, kv, key_padding_mask=kpm, need_weights=return_attn, average_attn_weights=True)
        return ((self.d_norm(att) * self.dw).sum(-1) + self.db, w)

class TokenResidualPool(_GatedDelta):

    def __init__(self, d, n_labels=12, n_heads=6, pe=64, dropout=0.2):
        super().__init__(d, n_labels, n_heads, dropout)
        self.base = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(dropout), nn.Linear(2 * d + pe, n_labels))

    def forward(self, tok, slot, sidx, B, pres, return_attn=False):
        base = self.base(torch.cat([_seg_mean_max(tok[:, 1:].mean(1), sidx, B), pres], 1))
        d_, w = self.delta(tok[:, 1:], sidx, B, return_attn)
        return (base + self.gate * d_, w)

class CodexResidualPool(_GatedDelta):

    def __init__(self, d, n_labels=12, n_heads=6, pe=64, dropout=0.2):
        super().__init__(d, n_labels, n_heads, dropout)
        self.base = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(dropout), nn.Linear(2 * d + pe, n_labels))

    def forward(self, tok, slot, sidx, B, pres, return_attn=False):
        base = self.base(torch.cat([_seg_mean_max(tok[:, 0], sidx, B), pres], 1))
        d_, w = self.delta(tok[:, 1:], sidx, B, return_attn)
        return (base + self.gate * d_, w)

class ClsAddPool(nn.Module):

    def __init__(self, d, n_labels=12, pe=64, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(nn.LayerNorm(4 * d + pe), nn.Dropout(dropout), nn.Linear(4 * d + pe, n_labels))

    def forward(self, tok, slot, sidx, B, pres, return_attn=False):
        return (self.net(torch.cat([_seg_mean_max(tok[:, 1:].mean(1), sidx, B), _seg_mean_max(tok[:, 0], sidx, B), pres], 1)), None)

class Readout(nn.Module):

    def __init__(self, pool, d, n_labels=12, pe=64):
        super().__init__()
        self.pool_kind, self.k = (pool, n_labels)
        self.pres_emb = nn.Embedding(N_SLOT_TYPES + 1, pe, padding_idx=0)
        if pool in ('xres', 'clsadd', 'xcodex'):
            self.pool = {'xres': TokenResidualPool, 'clsadd': ClsAddPool, 'xcodex': CodexResidualPool}[pool](d, n_labels, pe=pe)
        elif pool in ('attn', 'xattn'):
            if pool == 'xattn':
                self.pool = TokenXAttnPool(d, n_labels)
                wd = 3 * d + pe
            else:
                self.pool = LabelAttentionPool(d, n_labels)
                wd = d + pe
            self.norm = nn.LayerNorm(wd)
            self.w = nn.Parameter(torch.randn(n_labels, wd) * (1.0 / wd ** 0.5))
            self.b = nn.Parameter(torch.zeros(n_labels))
        else:
            self.pool = MeanMaxPool()
            self.net = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(0.2), nn.Linear(2 * d + pe, n_labels))
        self.drop = nn.Dropout(0.2)

    def forward(self, f, slot, sidx, B, return_attn=False):
        pe = self.pres_emb(slot)
        pres = torch.zeros(B, pe.shape[1], device=f.device, dtype=f.dtype).index_add_(0, sidx, pe)
        if self.pool_kind in ('xres', 'clsadd', 'xcodex'):
            return self.pool(f, slot, sidx, B, pres)[0]
        pooled, attn = self.pool(f, sidx, B, slot=slot, return_attn=return_attn)
        if self.pool_kind in ('attn', 'xattn'):
            x = torch.cat([pooled, pres.unsqueeze(1).expand(-1, self.k, -1)], -1)
            x = self.drop(self.norm(x))
            return (x * self.w).sum(-1) + self.b
        return self.net(torch.cat([pooled, pres], 1))

class Net(nn.Module):

    def __init__(self, enc, cond, n_meta=0, pool='mean_max', stem='native', n_slice=16):
        super().__init__()
        self.enc, self.cond = (enc, cond)
        self.compress = DepthCompress(n_slice, 3) if stem == 'compress' else None
        self.mixer = SlotDepthMixer(n_slice) if stem == 'mixer' else None
        self.tokens = pool in ('xattn', 'xres', 'clsadd', 'xcodex')
        D = enc.num_features
        self.meta_mlp = nn.Sequential(nn.LayerNorm(n_meta), nn.Linear(n_meta, 128), nn.GELU(), nn.Linear(128, D)) if n_meta > 0 else None
        self.readout = Readout(pool, D)
        if cond == 'post':
            self.slot_emb = nn.Embedding(N_SLOT_TYPES + 1, D, padding_idx=MASK_IDX)

    def forward(self, im, slot, smeta, sidx, B, vm=None):
        if self.mixer is not None:
            im = self.mixer(im, slot, vm)
        if self.compress is not None:
            im = self.compress(im)
        f = self.enc.forward_features(im, slot) if self.cond == 'token' else self.enc.forward_features(im)
        if self.tokens:
            inner = getattr(self.enc, 'vit', self.enc)
            orig = getattr(self.enc, '_orig_prefix', getattr(inner, 'num_prefix_tokens', 1))
            f = torch.cat([f[:, :1], f[:, orig:]], 1)
        else:
            f = self.enc.forward_head(f, pre_logits=True)
            if f.dim() > 2:
                f = f.flatten(1)
        ex = (lambda v: v.unsqueeze(1)) if self.tokens else lambda v: v
        if self.cond == 'post':
            f = f + ex(self.slot_emb(slot))
        if self.meta_mlp is not None and smeta.shape[1] > 0:
            mt = self.meta_mlp(smeta)
            f = torch.cat([f, mt.unsqueeze(1)], 1) if self.tokens else f + mt
        return self.readout(f, slot, sidx, B)
models=[]
_A5_LOAD_RECEIPT=[]
_a5_paths=[CKPT/f'm_f{i}.pt' for i in range(5)]
if not all(p.is_file() for p in _a5_paths):
    raise RuntimeError(f'A5 requires m_f0.pt..m_f4.pt; missing {[str(p) for p in _a5_paths if not p.is_file()]}')
_a5_config=None
for ckpt_path in _a5_paths:
    z=torch.load(ckpt_path,map_location='cpu',weights_only=False)
    cfg=z['cfg'];fold=int(z['fold'])
    if ckpt_path.name!=f'm_f{fold}.pt': raise RuntimeError('A5 fold/file identity mismatch')
    fields={'backbone':cfg['backbone'],'cond':cfg['cond'],'pool':cfg['pool'],'img':int(cfg['img']),'n_slice':int(cfg.get('n_slice',16)),'n_meta':int(cfg.get('n_meta',0)),'meta':cfg.get('meta','none'),'stem':cfg.get('stem','native'),'norm':cfg.get('norm','none')}
    if _a5_config is None:_a5_config=fields
    if fields!=_a5_config:raise RuntimeError(f'A5 per-fold input/model contract differs: {ckpt_path}')
    if fields['img']!=SIZE or fields['n_slice']!=N_SLICE or fields['n_meta']!=0:
        raise RuntimeError(f'A5 checkpoint input contract incompatible with loader: {fields}')
    if 'labels' in z and list(z['labels'])!=list(LABELS):raise RuntimeError('A5 label order differs')
    _stem=fields['stem'];_in=3 if _stem=='compress' else fields['n_slice']
    enc=timm.create_model(cfg['backbone'],pretrained=False,num_classes=0,in_chans=_in,**({'img_size':cfg['img']} if 'vit_' in cfg['backbone'] else {}))
    if cfg['cond']=='token':enc=ViTSlotToken(enc,N_SLOT_TYPES)
    m=Net(enc,cfg['cond'],cfg.get('n_meta',0),cfg['pool'],stem=_stem,n_slice=cfg.get('n_slice',16))
    rsna_strict_load(m,z['state_dict'],f'A5 fold {fold}')
    models.append(m.eval());_A5_LOAD_RECEIPT.append({'fold':fold,'file':str(ckpt_path),'sha256':rsna_sha(ckpt_path),'tensor_keys':len(z['state_dict']),'strict':True,'config':fields})
    print(f'A5 strict-loaded {ckpt_path.name}; all encoder and head tensors present')
CFG=dict(cfg)
rsna_json('/kaggle/working/diagnostics/a5_loads.json',_A5_LOAD_RECEIPT)


In [ ]:
AMP_PREF = 'bf16'  # BTKD precision retained, including T4 execution; no unvalidated AMP change.

def amp_for(dev):
    if not str(dev).startswith('cuda'):
        return (torch.float32, False)
    cc = torch.cuda.get_device_capability(dev)
    if AMP_PREF == 'bf16':
        return (torch.bfloat16, True)
    if AMP_PREF == 'fp16':
        return (torch.float16, True)
    if AMP_PREF == 'fp32':
        return (torch.float32, False)
    return (torch.bfloat16 if cc >= (8, 0) else torch.float16, True)
AMP_DT, AMP_ON = amp_for(DEV)
WORKERS = max(1, min(4, os.cpu_count() or 4))
CHUNK = 48
MICRO = 8
import copy as _speed_a5_copy
from concurrent.futures import ThreadPoolExecutor as _SpeedA5Pool
if torch.cuda.device_count() >= 2:
    _speed_a5_second = [_speed_a5_copy.deepcopy(m).to('cuda:1').eval() for m in models]
    _speed_a5_models = [[m.to('cuda:0').eval() for m in models], _speed_a5_second]
    del _speed_a5_second
    _speed_a5_pool = _SpeedA5Pool(max_workers=2)
else:
    _speed_a5_models = [[m.to('cuda:0').eval() for m in models]]
    _speed_a5_pool = _SpeedA5Pool(max_workers=1)
models = _speed_a5_models[0]
print(f"device {DEV} | amp {str(AMP_DT).split('.')[-1]} (on={AMP_ON}) | workers {WORKERS} | chunk {CHUNK} | micro {MICRO}")

def _norm_(im):
    k = CFG.get('norm', 'none')
    if k == 'zscore':
        m = (im > 0).float()
        n = m.sum(dim=(1, 2, 3), keepdim=True).clamp(min=1.0)
        mu = (im * m).sum(dim=(1, 2, 3), keepdim=True) / n
        var = (((im - mu) * m) ** 2).sum(dim=(1, 2, 3), keepdim=True) / n
        return (im - mu) / (var.sqrt() + 1e-06) * m
    if k == 'imagenet':
        m = (im > 0).float()
        return (im - 0.485) / 0.229 * m
    return im

@torch.no_grad()
def _speed_a5_micro(images, masks, *, dev, model_group, amp_dtype):
    """Predict acquired rows only; NaN is a sentinel exclusively for absent inputs."""
    active = np.flatnonzero((masks > 0).any(axis=1))
    out = np.full((len(model_group), len(masks), len(LABELS)), np.nan, np.float32)
    if not len(active):
        return out
    ims, slots, sidx, vms = [], [], [], []
    for compact_index, original_index in enumerate(active):
        present = np.flatnonzero(masks[original_index] > 0)
        blk = images[original_index][present]
        ims.append(torch.from_numpy(blk))
        vms.append(torch.from_numpy(blk.reshape(blk.shape[0], blk.shape[1], -1).max(2) > 0))
        slots.append(torch.from_numpy(present + 1).long())
        sidx.append(torch.full((len(present),), compact_index, dtype=torch.long))
    im = _norm_(torch.cat(ims).to(dev, non_blocking=True).float().div_(255.0))
    sl = torch.cat(slots).to(dev)
    si = torch.cat(sidx).to(dev)
    vm = torch.cat(vms).to(dev)
    sm = torch.zeros(len(sl), CFG.get('n_meta', 0), device=dev)
    per = torch.zeros(len(model_group), len(active), len(LABELS), device=dev, dtype=torch.float32)
    def _a5_forward(enabled):
        with torch.autocast('cuda' if str(dev).startswith('cuda') else 'cpu',
                            dtype=amp_dtype, enabled=enabled):
            for fold_index, model in enumerate(model_group):
                per[fold_index] = torch.sigmoid(model(im, sl, sm, si, len(active), vm=vm).float())
        return per.cpu().numpy()
    got = _a5_forward(AMP_ON)
    if AMP_ON and not np.isfinite(got).all():
        rsna_event('a5_fp16_nonfinite_retry_fp32', rows=int(len(active)))
        got = _a5_forward(False)
    _bad = ~np.isfinite(got)
    if _bad.any():
        rsna_event('a5_nonfinite_neutral', rows=int(_bad.any(axis=(0, 2)).sum()))
        got = np.where(_bad, 0.5, got)
    rsna_finite(got, 'A5 acquired-row probabilities', probability=True)
    out[:, active] = got
    return out

"""Two GPU owners, preserving every original eight-study microbatch."""


def speed_a5_ranges(n, micro=8, devices=2):
    if n < 0 or micro < 1 or devices < 1:
        raise ValueError((n, micro, devices))
    return [[(start, min(start + micro, n))
             for i, start in enumerate(range(0, n, micro)) if i % devices == d]
            for d in range(devices)]


def speed_a5_predict(images, masks, models_by_device, executor, dtype=None):
    """No shard-local ranks; output remains [fold, original study, finding]."""
    count = len(masks)
    assert len(images) == count
    output = np.full((len(models_by_device[0]), count, len(LABELS)), np.nan, np.float32)
    plans = speed_a5_ranges(count, MICRO, len(models_by_device))

    def owner(device_index, ranges):
        device = torch.device(f"cuda:{device_index}")
        result = []
        with torch.cuda.device(device):
            for start, stop in ranges:
                value = _speed_a5_micro(images[start:stop], masks[start:stop],
                    dev=device, model_group=models_by_device[device_index],
                    amp_dtype=AMP_DT if dtype is None else dtype)
                result.append((start, stop, value))
        return result

    tasks = [executor.submit(owner, d, ranges) for d, ranges in enumerate(plans) if ranges]
    for future in tasks:
        for start, stop, value in future.result():
            output[:, start:stop] = value
    return output

def predict(images, masks):
    return speed_a5_predict(images, masks, _speed_a5_models, _speed_a5_pool)

def _a5_validate_applicability(raw, eligible):
    """Never confuse an absent input with failed inference on an acquired input."""
    x = np.asarray(raw)
    m = np.asarray(eligible, dtype=bool)
    if x.ndim != 3 or x.shape[1] != len(m):
        raise ValueError('A5 eligibility/raw shape mismatch')
    if m.any():
        rsna_finite(x[:, m], 'A5 acquired-row complete tensor', probability=True)
    if (~m).any() and not np.isnan(x[:, ~m]).all():
        raise RuntimeError('A5 absent rows must retain explicit NaN sentinels')
    return m

def _a5_rank_available(raw, eligible):
    eligible = _a5_validate_applicability(raw, eligible)
    result = np.full((len(eligible), raw.shape[-1]), .5, dtype=np.float64)
    if eligible.any():
        result[eligible] = 0.0
        for fold in raw:
            result[eligible] += rsna_rank01(fold[eligible])
        result[eligible] /= raw.shape[0]
    return result

def _a5_blend_available(parent, a5_values, eligible, weight):
    """Parent-only on metadata-proven absence; no silent fallback on model error."""
    base = np.asarray(parent, dtype=np.float64)
    a5 = np.asarray(a5_values, dtype=np.float64)
    mask = np.asarray(eligible, dtype=bool)
    if base.shape != a5.shape or mask.shape != (len(base),):
        raise ValueError('A5 merge shape mismatch')
    rsna_finite(base, 'A5 parent probabilities', probability=True)
    if not mask.any() or weight == 0:
        return base.copy()
    rsna_finite(a5[mask], 'A5 applicable ranks', probability=True)
    base_rank = pd.DataFrame(base).rank(method='average', pct=True).to_numpy()
    a5_rank = pd.DataFrame(a5[mask]).rank(method='average', pct=True).to_numpy()
    result = base_rank.copy()
    result[mask] = (1-weight)*base_rank[mask] + weight*a5_rank
    rsna_finite(result, 'A5 applicability-aware merge', probability=True)
    return result

# Macro ROC-AUC depends on ordering, so combine fold orderings rather
# than allowing a fold's probability scale to dominate the mean.
preds = np.full((len(models), len(studies), len(LABELS)), np.nan, np.float32)
_a5_eligible = np.zeros(len(studies), dtype=bool)
t0, done = (time.time(), 0)
def submit_study_block(executor, start):
    """Keep one bounded CPU decode block ahead of GPU inference."""
    block = studies[start:start + CHUNK]
    futures = [
        executor.submit(
            build_study,
            (index, study, by.get(study, [])),
        )
        for index, study in enumerate(block)
    ]
    return start, block, futures

# CPU decode threads avoid forking a process after CUDA/model initialization.
with A5DecodePool(max_workers=WORKERS) as ex:
    pending_block = submit_study_block(ex, 0)
    while pending_block is not None:
        rsna_deadline('A5 full-cohort inference')
        c0, block, futs = pending_block
        next_start = c0 + len(block)
        pending_block = (
            submit_study_block(ex, next_start)
            if next_start < len(studies)
            else None
        )
        imgs = np.zeros((len(block), N_SLOT, N_SLICE, SIZE, SIZE), np.uint8)
        msks = np.zeros((len(block), N_SLOT), np.uint8)
        _fut_index = {f: i for i, f in enumerate(futs)}
        for f in as_completed(futs):
            try:
                i, a, k = f.result()
                imgs[i], msks[i] = (a, k)
            except Exception as e:
                i = _fut_index[f]
                rsna_event('a5_study_failed', study=str(block[i]), error=f'{type(e).__name__}: {str(e)[:500]}')
                print(f'[a5] {block[i]}: build failed, treated as A5-absent (flagged): {type(e).__name__}: {str(e)[:200]}', flush=True)
                imgs[i] = 0; msks[i] = 0
        _a5_eligible[c0:c0 + len(block)] = (msks > 0).any(axis=1)
        preds[:, c0:c0 + len(block)] = predict(imgs, msks)
        _a5_validate_applicability(preds[:, c0:c0 + len(block)], _a5_eligible[c0:c0 + len(block)])
        done += len(block)
        el = time.time() - t0
        print(f'  {done:,}/{len(studies):,}  {el / 60:.1f}m  eta {el / done * (len(studies) - done) / 60:.1f}m', flush=True)
        del imgs, msks
        gc.collect()
print(f'\ninference done in {(time.time() - t0) / 60:.1f} min')
np.savez_compressed('/kaggle/working/speed_a5_raw.npz',
    study_uids=np.asarray(studies), raw_probabilities=preds, applicable=_a5_eligible)
_speed_a5_pool.shutdown(wait=True)
del _speed_a5_models, models, m, enc, z
gc.collect()
for _speed_device in range(torch.cuda.device_count()):
    with torch.cuda.device(_speed_device):
        torch.cuda.empty_cache()
A5_W = 0.52
A5_LABELS = list(LABELS)
_a5_ok = _a5_validate_applicability(preds, _a5_eligible)
_a5_rank_mean = _a5_rank_available(preds, _a5_eligible)
A5_PREDS = dict(zip(sub_df['StudyInstanceUID'].astype(str), _a5_rank_mean.astype(np.float32)))
A5_AVAILABLE = dict(zip(sub_df['StudyInstanceUID'].astype(str), _a5_eligible.tolist()))
rsna_save_predictions('a5_rank_mean', studies, _a5_rank_mean, LABELS)
rsna_json('/kaggle/working/diagnostics/a5_applicability.json', {
    'unavailable_uids':[u for u, ok in A5_AVAILABLE.items() if not ok],
    'available_count':int(_a5_eligible.sum()), 'model_failure_fallback_allowed':False,
    'rank_placeholder_for_unavailable':0.5, 'placeholder_used_in_blend':False})
rsna_phase('a5', 'COMPLETE', applicable=int(_a5_eligible.sum()), absent=int((~_a5_eligible).sum()))
for _a5k, _a5v in _A5_SAVED.items():
    globals()[_a5k] = _a5v
del _A5_SAVED, _a5k, _a5v


In [ ]:
_MA_DINO_STAGE = pd.read_csv('/kaggle/working/_pipeline_stage.csv', dtype={'StudyInstanceUID':str})[MA_LABELS].to_numpy().copy()
_a5_sub = pd.read_csv('/kaggle/working/_pipeline_stage.csv', dtype={'StudyInstanceUID':str})
assert _a5_sub.columns.tolist()[1:] == A5_LABELS, 'submission schema drift'
if A5_W > 0:
    _a5_uid_order = _a5_sub.StudyInstanceUID.astype(str).tolist()
    _a5_ours = np.stack([A5_PREDS[u] for u in _a5_uid_order])
    _a5_presence = np.asarray([A5_AVAILABLE[u] for u in _a5_uid_order], dtype=bool)
    _a5_sub[A5_LABELS] = _a5_blend_available(
        _a5_sub[A5_LABELS].to_numpy(), _a5_ours, _a5_presence, A5_W)
    assert np.isfinite(_a5_sub[A5_LABELS].to_numpy()).all()
    _a5_sub.to_csv('/kaggle/working/_pipeline_stage.csv', index=False)

_MA_A5_VALUES = _a5_ours.copy()
_MA_A5_MASK = _a5_presence.copy()
_MA_A5_BLEND_FN = _a5_blend_available
_MA_A5_STAGE = pd.read_csv('/kaggle/working/_pipeline_stage.csv', dtype={'StudyInstanceUID':str})[MA_LABELS].to_numpy().copy()


## 7. Original RadImageNet readers and fitted calibrator

In [ ]:
from __future__ import annotations
import base64 as _rad_b64
import zlib as _rad_zlib
_RAD_CAL_PAYLOAD = 'eNrtmk1vI8cRhv9KsJdcKKE/q6tzc4z4ZCMBcjQWhrCRDSG2ZEjaIEGQ/57n7RlRQ3KG4jqLJAcDS4o709NdXR9vvVU9/3z30+3N/bvffRuuawgxlm7eq2ePeffrpV8v/V9euvLraD3k6ilW6zn126vYd+U6eKmx9RiaF7OSx+X1weE67K7SdSgpVSauuaecUxr3rtp1LK0HyyV7y9Gmy/E6pBhTL61Zt2gWx+WtSew6JmOoM5AHutXp+ro8+ZrlsnvJOfDdfRL+Kl4zd2bqllNmga7rvrva2OzGNFuynGxpmnxrS/069hCtpt6T1dLLOQVsTbJ1fWunG2qXAX8NiU+7VK8rdqvNU89uwQwjpWyeLdTCnVy84ua9pthSjznHXLjDpZxbLe4WPRAQCT+LrSVcGGdLzNX0XKhesC2eV5uZ67lQPDlmSzUhS2+61ELtoTOyWGjdPudc73fvnj7c/Hg7ElpyzYXjdwIZ32m7X3INZ0crtvtc833ua6fyoUYUGf4H13rJpX+GK5aa5VbeWO9ye/03bLi97i8SpaSCl49nc4gdxI2p1dZyVmQnfL56jBH/j70GqSq6dyMROLHQGvCpcSnkYsyeohNErnVjbamVjs47wOpBa7BAsa61SXulDfSIwAbAkQqDmSK38WwImKK1kFJ3d11CpFqR1mPHlj1pWVAHeDfyU2hk0vGogz0GqCBbKmFMx9xIWEAbQ8I4PUvmAplQCeuij7GNXGMk3yhBsFIfEnvVE6HFYMHDtFuLzKwpya9gisaZduAkcyAvmw1ROjmd7OnBYytpOBqJjTyK4KzT0Pi0tQJYslc0nGqcNZV7dEaRvfcSewg9h4p41iwO+4CWMkNG1326VBmHFUoB7VoakzE+JAtYN/NknS9lLJ+9S5qR56Q2kLA4qgTuplGNmUJAkbXiGbrU0HCIsgtYaGOUVXbayNeykLfpGv8lulBfgiFEmx61xm3LTlrOPoa51Ng7IyJiT4tWxrE0ZmnJRnhawZsyLgjXIC9MRu0tRAgImkJ7bUyHKk1eUgIewtRjXGDJqO1mNJrHfNUty1HRW2SSoV3FAVIA//hrUXKANxhbRbEkgqAFKiCC43qOEXsPTaKtKCdkqx3/koMYsuP+mh4HrHoQtqSIw/h4DM7EpRL5zRirAcHiUDj2SUlr9fFAsElHBX/zWgJCQlw+83Qksw8Pt9+Ty0hmOBQBh9XQAr7fxjRQ2IkGTT/w8cAiBKwRc1jwcMh+WKtMgFShNaDGJWRAechNSOMUldXTynNIUPAaEKKkCv1cLFzxaowBOmX8vU8Pj1voWa5JTPFcGN4QXm+PpZPjA8QQYYp9Qab9rZiIAuLX8AA8f/jX4kHgI+BXCtBEeOTFvNPapIyC92YAEOmHypJcCSiFOpQi712M73IL8KiBX8Hz62pDNsAH8MK/rMR+tNSRIRJwAkIIY8Rn/fD2kB2V9I7BMX+KSJ/XJtNAbwFglmQZbu/90CZcMhAeZKudKAvlSBLwNwP1aA88BYpPdJRkbADGeBzWN2IOvySVELxyU0Lx0JHQHsFpEQRsCB9iO1qT3IOHGUjMDqeczX4BiIbvCjkiLD6t6G239p+gAML1KQuAdaLLdidDV6Y6uPR+9+3LT8KrlYgzAsjg7ok+VJakimvgAjn5qUFmtTGJKXGxSadg2UuLkWok4EvGJ0Hdsr+DmpVlGsUMZkxtuTI8vDAVaIsnSIDbK0DhVSpAFlu4iRtgrLYWRaArADOcFDAfErEdwBS8dWpNqHupW3o7nIukTmxlmASq4L/51ESrznqJTSkgCzslVSMncekb8659ljeyYttxK0oX9k50n3h2kDqoapToWt8S9/yO3tTXyteLu+19rkPViKkIblLnzJhJUhYENUJCtdfWKkEFe9WTsWin6azpqJFIfEwNyLNes+PZXFV3h+tIOXjb5mzIRnCLaWKuhnPtjsCVPAOwdkhQhBSMfWLVruTgwEM/WXt1FYgv5AN4IsyBiHrOSscCBjIomksgZCPFn9grqDrE9UXDQCSPfs5RXyeuQl3gwcEIdn5JzADhpG34s4uF5NQ24eh1GWISkEmYA6iFiNezYAbkKNGJhIk+l9XNXK3r7AgLT4YnRUuHicJS0NRLLG0KDrF1IbkaMzj2MoeKzAH/kTo9KsnWJe8gDZUuxs1iPi0CayABMdLIT4qQCbfENfCiAsujIsj9KMUQ1kXwXUBFZspHZm9Zj/dB/SrVy1qOJg9ApKAlmSQNpr6SjibnNgVoqZQCr8gllgsTiFAOCFSeDcYWuKOChZQXi5dPxouF5Oo6ogVTaxDwnVE8KfQFI6Zomwgb/oI4VG2Ae1MdsTQCwQuZp5ZROZPcdudWHrZRfwe4cPE1cv9L/FDuwRwGNYOttGllMRItS1K3sFDQDAwQMvgpfIzyIeQj2yTFOhomGLsBVPtAASGLqiXKnChASW/4ILUTGlftQlUVl8FDjifbKYIzLKXmco4anNPK6ZVl8MhVBs+D5YgTo8HXuIEDQDJF4wHEYL5PvEExn4PIt7x0JumDeUgjxHeFIZDt+6xN5mALCc6pMjTZ7oy8EomCUA09MTRvvuCLxyFC+sEWCGiqjE+HIAJ4g39Bi5sadSN9MJZaFWLbpubBNBvlmwrPAONJWS1DpaIgbyKuZSbKg7qZzgNKsqnwIX8RAOvINdgfRg0jPNJBWE+5xAQ9qDaAPh4nKInagjAL9otj26U+sAnbUERTF0QYjHV8j8OEA+mohtH9SLLuoUIh6uAWHI6yAw54uEsqL6zufMAAt3yW+6xSFLmKwEmYgIPlXHYbPAo3AvKovCk/0KR5m3fmKkpFz2W3ng58h7KzqVVHqVYUpfGlAJHvQ2dN7QKmXoQITggHUz9HGZhSty5smYHeSL4pFdvbXlXZcSJV7GCHKPpsePEmVAOm41WxbuW3Y+qkrhYhisbrSBqbGiGdAkBwd5gzNehxGUVhzu5JCKa2VAyL2pfYgYgUtcTwu3RKZzO2JublhfCDbO1Qqypu8SdTWO0501Z6iAAH68g21mECvjX8bZZ7yvpViqq9BlHCnhj7DFuiXm8qEwgrAkK9hpfbuU/nN/i4l7I/qQEnpR4qVHUxgDjblWuM2UZFhOpzp+apb8i4ua8gnyHrqPMHs839gK7gaaYDDtILqNTWOF8kxVYdMxlVDwE68/F8DR2CgpCSkkzkEvKY3x9GoRqjpn8ZLj62TyEoRVV1dxrV2GtNOMosPAjqSC6pm6yRKRJFM5wnixbhh6uLl9FdjElsUvX7WxR61T1gGohOkQaej27MxiR+DQjAndQ5SgL4Yb+VMKT2UncGK4se5hfeR7Jr6nygbTJcPGK7QSpXOoBJkuMXlFTYQX0aRsObYEQn22BNrDQdKwmZ1DDaZNcANxumdkGs3pZIJcaCcxBuPSlGziTf+UNZIgqmLMTWS1/XYNBxFowH5FchZb7uT5EQqervUK+Rc15pP55mBpKra6Wju5MCbZwiKOSS/HFuZyVRBFeeIoVrc35oNnVt1ARpKDaHF2BO1z3DLILOrOVZdhIG52ojyFxsVa1k6Bq+sOS7AA0upIqpiCt9Om8+1SsIoI4/ZYFKZq9tdwnhCzrVRpoCxpqSo+0ua1DNLCOMc3X19vGSZZ9znfEAMTUpymSGqYW/kpXU6FWV6+pah9OGLveaTkopOV1d3U9oWfAhMiyK46fRgPeL9LTSDAtUjj5cFN4D8S5zmaCjmaTjJ/Kvxb3MaFrHVI27QS2vRfcMCmeqcUXY+NX6652okyj19OEC3SaFrdWyeyJMxo7gPrANn17GjQ5RgtqvregNjzr1hYOOxATeos4b/IItGRxEFZC4aNruVL1ZbGx8ojpE5moLiGNavazB+baxPoXr/gK56zjI1VGbjtG8nA3Xg3SpFl2gtqzqsM/oGgb5Axj4w+XD1g48MBFrMAnR+TRxMci1+8h+uCC1e1nmC7XEWhf2zEdIw5SsCe3Tcc18XibsU/PKhJhd7a380s67RLEvQQWUw9KKjmps7qefVeub/IZQoLbK0qxnHRFP7qlOy8BURC7Fy2LDPk7N1F1VEm1lrZbSmSWVSoNk63DQXgvIohAjYcjXU7b/zI8u7RVvPiRChSqVjkiU6YQjrT1ImVAAN/WoKEp62V2aQGAdAuQKhSW5qouyIdQ4jNe5NS6I8v10hLIeqIIMqplDn71o0dYl4fEFkRZ4LmhFJNUSQqZCNlBT2fIWnKzJ9XWwG48bOzGZWEIffcVx/q23xAziBV83gf1cE4/+mvI8/EFV3QVmPW0a6rB7ZD314ploTllOSlFYqX3X4u7TJg6jmxLRWxit1FaPtArKoHfHeb3jwPmNGDqfvS+Iv9Ns14Vv6p9rfyft+HGiFqmJIQSU++rlbegPBqDumli9zoOGSptec8O6GAWqtaAgkOgqZ3P1WhQQ08aj2KGONuEIqdZetzuLFB6qQniid8HkZTnlyGsPGnA4lY5Qu1456WnbokG9IupwU0NoPhHTwRRUqynZU0HObV8QUyZuo/lb6tFhsdxb7bCoU9qWe8vLxBwj2laVzgS+bIZGMfee1Qaldl87gBZ5jjr3Y4rq2Xaf3LatohHwxzbqBKtv4d8bHnh1eUqfVRx07jc4zfzySrj8IGV9NMFX4mBKrq6FXc4Jz154/3737u7++fbxw+3Pz9N7eg0X0mHCeHfHbHrNhrCIpdXRA/LRRC4WR9sU/ZqJB46XJsJouwU1rPT+il6uKHqNAdbJ2InUJp0wVWBV7w8NuldU7uMyajZqh2N6UfeoM6ym1zLGizF6T6AnNQZiHO/KZMijZMOV2/yKQFKv0TSXq0Hb5teE9F4HLp50QKJ3OX64edZ7ie+++PLrd7t339z+5e7mx9/88Qt+f82dx5f//Omr6e8fvv/+49Pdwz0/f3/z19vH3z7x68uH++fpKhP+/Pjw/PDh4cfv+Hz86f5Jk99/93T7eHersfff/fnmh/H3y4fH8feLv9/x96ub5+l7vq9f0wj9msf8+HH6fhnDr3kMvzRGG3p8+PizVt3vaXy/ysjox5sPzx8fbxn+7cuWv7m9v3v68PFpsfH9pcWwLc1oyEI3f/7H/cPf7p7vnhZ6ev/+X/8GYIe3xg=='
# BTKD Rad graph: preserve its E13/E11 calibrated feature contract; remove unused twin work.
#
# The pinned reference Rad family is fused with correct-contract E13, then
# the same E13 heads run on the E11 layout at 0.15. No twin/legacy wrapper
# follows it, matching the branch that produced V48's visible submission.

import contextlib as _rad_contextlib
import gc as _rad_gc
import hashlib as _rad_hashlib
import json as _rad_json
import os as _rad_os
import re as _rad_re
import time as _rad_time
from concurrent.futures import ThreadPoolExecutor as _RadThreadPool
from pathlib import Path as _RadPath

import numpy as _rad_np
import pandas as _rad_pd
import pydicom as _rad_pydicom
import torch as _rad_torch
import torch.nn as _rad_nn
import torch.nn.functional as _rad_F
from torchvision.models import resnet50 as _rad_resnet50

_RAD_LABELS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
    'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's",
    'Contusion', 'Fracture',
]
_RAD_ALPHA = 0.55
_RAD_EXCLUDE = ("Baker's", 'Fracture')
_RAD_HEADS_SHA256 = '54f657826b3458a7ba3d462e198ba380732f2b136246182312704929874a9a2c'
_RAD_REFERENCE_HEADS_SHA256 = '0f465649799ecfbccaac1767844639e7ced44e1bc9babde6e4bac7c5d9b89eaa'
_RAD_ENCODER_SHA256 = '08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734'
_RAD_E13_HEADS_SHA256 = 'ad9f19af73bfdf4e49263c0e45060dc3cb239e1195039b26dc8c0a3a6bcd1a8a'
_RAD_E13_MEMBER_WEIGHT = 0.50
_RAD_V48_SECOND_ALPHA = 0.20
_RAD_TWIN_ALT_WEIGHT = 0.500001
_RAD_TOKEN_DIM, _RAD_HEAD_DIM = 2048, 512

_RAD_E11_SLOTS = [
    ('SAG_NOFS', 'Sagittal', None, False),
    ('COR_NOFS', 'Coronal', None, False),
    ('AX_NOFS', 'Axial', None, False),
    ('SAG_FS', 'Sagittal', None, True),
]
_RAD_E11_CROP_MM = 130.0
_RAD_E11_CACHE_SLICES = 8
_RAD_E11_IMG = 224

_RAD_E13_SLOTS = [
    ('SAG_FS', 'Sagittal', None, True),
    ('COR_FS', 'Coronal', None, True),
    ('AX_FS', 'Axial', None, True),
    ('SAG_NOFS', 'Sagittal', None, False),
]
_RAD_E13_CROP_MM = 130.0
_RAD_E13_CACHE_SLICES = 8
_RAD_E13_IMG = 224

# Our independently trained five-fold family.  Its preprocessing and estimator
# are preserved from V35: native DICOM geometry/fat-sat handling and a mean of
# per-fold percentile ranks (rather than v15's rank of the probability mean).
_OUR_N_SLOT, _OUR_N_SLICE, _OUR_IMG = 3, 8, 224

# Exact V40/E10 test representation: three fat-suppressed planes, eight
# acquired slices per plane, full frame, legacy ordering/laterality/fill.
SLOTS = [
    ('SAG_FS', 'Sagittal', None, True),
    ('COR_FS', 'Coronal', None, True),
    ('AX_FS', 'Axial', None, True),
]
N_SLOT = len(SLOTS)
CACHE_SLICES = 8
IMG = CACHE_IMG = 224
CROP_MM = 10_000.0
SLICE_BAND = (0.2, 0.8)
RULES = dict(RULES_LEGACY)
TIME_BUDGET = 8.0 * 3600


def _rad_log(message):
    print(f'[Rad-dual5] {message}', flush=True)


def _rad_sha256(path, chunk=8 << 20):
    digest = _rad_hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(chunk), b''):
            digest.update(block)
    return digest.hexdigest()


def _rad_find_file(name, expected_sha=None, explicit_env=None):
    if explicit_env and _rad_os.environ.get(explicit_env):
        candidates = [_RadPath(_rad_os.environ[explicit_env])]
    else:
        by_name = {
            'ResNet50.pt': [
                '/kaggle/input/datasets/marwanmath/resnet-50-radimagenet-marwan/ResNet50.pt',
                '/kaggle/input/resnet-50-radimagenet-marwan/ResNet50.pt',
            ],
            'v52_radimagenet_heads.pt': [
                '/kaggle/input/datasets/prvsiyan/rsna-knee-v52-radimagenet-heads-20260812/v52_radimagenet_heads.pt',
                '/kaggle/input/rsna-knee-v52-radimagenet-heads-20260812/v52_radimagenet_heads.pt',
                '/kaggle/input/datasets/antoinegg1/rsna-knee-e9-radimagenet-heads-v15/v52_radimagenet_heads.pt',
                '/kaggle/input/rsna-knee-e9-radimagenet-heads-v15/v52_radimagenet_heads.pt',
            ],
            'v52_e11_heads.pt': [
                '/kaggle/input/notebooks/sofiaanjenje/rsna-knee-e13-train/rsna_rad_e11/v52_e11_heads.pt',
                '/kaggle/input/rsna-knee-e13-train/rsna_rad_e11/v52_e11_heads.pt',
                '/kaggle/input/notebooks/sofiaanjenje/rsna-knee-e11-train/rsna_rad_e11/v52_e11_heads.pt',
                '/kaggle/input/rsna-knee-e11-train/rsna_rad_e11/v52_e11_heads.pt',
                '/kaggle/input/datasets/antoinegg1/rsna-knee-e11-diverse-heads-v20/v52_e11_heads.pt',
                '/kaggle/input/rsna-knee-e11-diverse-heads-v20/v52_e11_heads.pt',
            ],
        }
        if name not in by_name:
            raise FileNotFoundError(f'unpinned Rad artifact name: {name}')
        candidates = [_RadPath(path) for path in by_name[name]]
    existing = [path for path in candidates if path.is_file()]
    valid = [
        path for path in existing
        if expected_sha is None or _rad_sha256(path) == expected_sha
    ]
    if len(valid) != 1:
        raise RuntimeError(
            f'expected one verified Rad artifact {name}, found {valid}; existing={existing}'
        )
    return valid[0]


class _RadEncoder(_rad_nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = _rad_nn.Sequential(
            *list(_rad_resnet50(weights=None).children())[:-2]
        )

    def forward(self, image):
        return self.backbone(image).mean(dim=(2, 3))


class _RadHead(_rad_nn.Module):
    def __init__(self):
        super().__init__()
        self.project = _rad_nn.Sequential(
            _rad_nn.LayerNorm(_RAD_TOKEN_DIM),
            _rad_nn.Linear(_RAD_TOKEN_DIM, _RAD_HEAD_DIM),
            _rad_nn.GELU(),
        )
        self.plane = _rad_nn.Parameter(_rad_torch.randn(N_SLOT, _RAD_HEAD_DIM) * .01)
        self.position = _rad_nn.Parameter(_rad_torch.randn(CACHE_SLICES, _RAD_HEAD_DIM) * .01)
        self.query = _rad_nn.Parameter(_rad_torch.randn(len(_RAD_LABELS), _RAD_HEAD_DIM) * .02)
        self.attn = _rad_nn.MultiheadAttention(
            _RAD_HEAD_DIM, 8, dropout=.10, batch_first=True
        )
        self.fuse = _rad_nn.Sequential(
            _rad_nn.LayerNorm(_RAD_HEAD_DIM * 4),
            _rad_nn.Linear(_RAD_HEAD_DIM * 4, _RAD_HEAD_DIM),
            _rad_nn.GELU(),
            _rad_nn.Dropout(.15),
        )
        self.weight = _rad_nn.Parameter(
            _rad_torch.randn(len(_RAD_LABELS), _RAD_HEAD_DIM) * .02
        )
        self.bias = _rad_nn.Parameter(_rad_torch.zeros(len(_RAD_LABELS)))

    def forward(self, feature, mask):
        token = self.project(feature.float())
        token = token.view(len(token), self.plane.shape[0], self.position.shape[0], self.plane.shape[-1])
        token = token + self.plane[None, :, None] + self.position[None, None]
        token = token.flatten(1, 2)
        key_padding = mask <= 0
        all_empty = key_padding.all(1)
        if all_empty.any():
            key_padding = key_padding.clone()
            key_padding[all_empty, 0] = False
        query = self.query.unsqueeze(0).expand(len(token), -1, -1)
        attended = query + self.attn(
            query, token, token, key_padding_mask=key_padding, need_weights=False
        )[0]
        denominator = mask.sum(1, keepdim=True).clamp_min(1).unsqueeze(-1)
        mean = (token * mask.unsqueeze(-1)).sum(1, keepdim=True) / denominator
        mean = mean.expand(-1, len(_RAD_LABELS), -1)
        fused = self.fuse(_rad_torch.cat(
            [attended, mean, _rad_torch.abs(attended - mean), attended * mean], dim=-1
        ))
        return (fused * self.weight.unsqueeze(0)).sum(-1) + self.bias


def _rad_load_public_heads(device, expected_sha):
    heads_path = _rad_find_file('v52_radimagenet_heads.pt', expected_sha)
    payload = _rad_torch.load(heads_path, map_location='cpu', weights_only=True)
    expected = {
        'version': 'v52-radimagenet-resnet50-official-1',
        'targets': _RAD_LABELS,
        'encoder_sha256': _RAD_ENCODER_SHA256,
        'encoder_source_commit': '0ce16f7375db4236e646829d1eca61cdb4282133',
        'img': 224,
        'slices_per_plane': 8,
        'feature': 'global_average_pool',
    }
    for key, value in expected.items():
        if payload.get(key) != value:
            raise RuntimeError(f'public-v15 head contract drift for {key}')
    folds = payload.get('folds')
    if not isinstance(folds, list) or len(folds) != 5:
        raise RuntimeError('public-v15 bundle requires exactly five heads')
    if sorted(int(record.get('fold', -1)) for record in folds) != list(range(5)):
        raise RuntimeError('public-v15 fold identity drift')
    heads = []
    for record in folds:
        head = _RadHead().to(device).eval()
        head.load_state_dict(record['state_dict'], strict=True)
        heads.append(head)
    return heads, str(heads_path)


def _rad_load_e13_heads(device):
    # V48 used an unqualified filename shared by E11 and E13. Resolve the
    # intended E13 bundle by content and validate its complete pixel contract.
    heads_path = _rad_find_file('v52_e11_heads.pt', _RAD_E13_HEADS_SHA256)
    payload = _rad_torch.load(heads_path, map_location='cpu', weights_only=False)
    expected = {
        'version': 'e11-radimagenet-resnet50-diverse-1',
        'targets': _RAD_LABELS,
        'encoder_sha256': _RAD_ENCODER_SHA256,
        'slots': [list(slot) for slot in _RAD_E13_SLOTS],
        'crop_mm': _RAD_E13_CROP_MM,
        'img': _RAD_E13_IMG,
        'slices_per_plane': _RAD_E13_CACHE_SLICES,
        'feature': 'global_average_pool',
    }
    for key, value in expected.items():
        if payload.get(key) != value:
            raise RuntimeError(f'E13 head contract drift for {key}')
    folds = payload.get('folds')
    if not isinstance(folds, list) or len(folds) != 5:
        raise RuntimeError('E13 bundle requires exactly five heads')
    if sorted(int(record.get('fold', -1)) for record in folds) != list(range(5)):
        raise RuntimeError('E13 fold identity drift')
    heads = []
    for record in folds:
        head = _RadHead().to(device).eval()
        head.load_state_dict(record['state_dict'], strict=True)
        heads.append(head)
    return heads, str(heads_path)


def _rad_load_models(device):
    """Two persistent encoder replicas; unused alternative heads are not loaded."""
    import copy
    path = _rad_find_file('ResNet50.pt', _RAD_ENCODER_SHA256, explicit_env='RSNA_RAD_WEIGHT_PATH')
    encoder = _RadEncoder().eval()
    rsna_strict_load(encoder, _rad_torch.load(path,map_location='cpu',weights_only=True),'Rad encoder')
    if sum(p.numel() for p in encoder.parameters())!=23508032:
        raise RuntimeError('Rad encoder parameter count changed')
    replicas=[encoder.to('cuda:0'), copy.deepcopy(encoder).to('cuda:1')] if torch.cuda.device_count() >= 2 else [encoder.to('cuda:0')]
    for m in replicas:
        m.requires_grad_(False)
    heads,hpath=_rad_load_public_heads(device,_RAD_REFERENCE_HEADS_SHA256)
    return replicas,heads,str(path),hpath


@_rad_torch.inference_mode()
def _rad_encode(encoder, pixels, slot_mask, device):
    """Preserve parent's <=96 images/device chunks without DataParallel replication."""
    n,slots,slices,height,width=pixels.shape
    token_mask=_rad_np.repeat(slot_mask[:,:,None],slices,axis=2).reshape(n,-1)
    _rad_empty=int((token_mask.sum(1)==0).sum())
    if _rad_empty:
        print(f'[rad] {_rad_empty} study(ies) with no slot evidence encoded as all-zero rows (parent policy)', flush=True)
    valid=_rad_np.flatnonzero(token_mask.reshape(-1)>0)
    flat=pixels.reshape(-1,height,width)
    features=_rad_np.zeros((n,slots*slices,_RAD_TOKEN_DIM),_rad_np.float16)
    def run(indices,d):
        dev=_rad_torch.device(f'cuda:{d}')
        def _rad_forward(_amp):
            with _rad_torch.cuda.device(dev), _rad_torch.inference_mode(), _rad_torch.autocast('cuda',dtype=_rad_torch.float16,enabled=_amp):
                image=_rad_torch.from_numpy(flat[indices]).to(dev).float().div_(127.5).sub_(1.)
                image=image.unsqueeze(1).expand(-1,3,-1,-1).contiguous()
                return encoder[d](image).float().cpu().numpy()
        value=_rad_forward(True)
        half=value.astype(_rad_np.float16)
        if not (_rad_np.isfinite(value).all() and _rad_np.isfinite(half).all()):
            rsna_event('rad_fp16_nonfinite_retry_fp32',tokens=int(len(indices)))
            value=_rad_forward(False); half=_rad_np.clip(value,-65000,65000).astype(_rad_np.float16)
        if not _rad_np.isfinite(half).all():
            rsna_event('rad_nonfinite_zeroed',tokens=int((~_rad_np.isfinite(half).reshape(len(half),-1).all(axis=1)).sum()))
            half=_rad_np.where(_rad_np.isfinite(half),half,0).astype(_rad_np.float16)
        return indices,half
    _dev_count = len(encoder)
    with _RadThreadPool(max_workers=_dev_count) as pool:
        for start in range(0, len(valid), 96 * _dev_count):
            rsna_deadline('Rad full-cohort inference')
            block = valid[start:start + 96 * _dev_count]
            if _dev_count == 1:
                splits = (block,)
            else:
                cut = (len(block) + 1) // 2
                splits = (block[:cut], block[cut:])
            jobs = [pool.submit(run, ii, d) for d, ii in enumerate(splits) if len(ii)]
            for job in jobs:
                indices, value = job.result()
                features.reshape(-1, _RAD_TOKEN_DIM)[indices] = value
    return features,token_mask.astype(_rad_np.float32)


@_rad_torch.inference_mode()
def _rad_predict_head(head, features, masks, device, batch=64):
    predictions = []
    for start in range(0, len(features), batch):
        image = _rad_torch.from_numpy(features[start:start + batch]).to(device)
        mask = _rad_torch.from_numpy(masks[start:start + batch]).to(device)
        def _head_forward(_amp):
            amp = (_rad_torch.autocast('cuda', enabled=_amp)
                   if device.type == 'cuda' else _rad_contextlib.nullcontext())
            with amp:
                return _rad_torch.sigmoid(head(image if _amp else image.float(), mask)).float().cpu()
        block = _head_forward(True)
        if not bool(_rad_torch.isfinite(block).all()):
            rsna_event('rad_head_fp16_nonfinite_retry_fp32', rows=int(len(image)))
            block = _head_forward(False)
        if not bool(_rad_torch.isfinite(block).all()):
            rsna_event('rad_head_nonfinite_neutral', rows=int((~_rad_torch.isfinite(block)).any(dim=1).sum()))
            block = _rad_torch.where(_rad_torch.isfinite(block), block, _rad_torch.full_like(block, 0.5))
        predictions.append(block)
    return _rad_torch.cat(predictions).numpy()


def _rad_rank_columns(values):
    return rsna_rankpct(values)


def _rad_validate(frame, expected_ids):
    if frame.columns.tolist() != ['StudyInstanceUID', *_RAD_LABELS]:
        raise RuntimeError('V36 submission schema drift')
    ids = frame['StudyInstanceUID'].astype(str).tolist()
    if ids != list(map(str, expected_ids)) or len(ids) != len(set(ids)):
        raise RuntimeError('V36 submission study identity/order drift')
    values = frame[_RAD_LABELS].to_numpy(_rad_np.float64)
    if not _rad_np.isfinite(values).all() or values.min() < 0 or values.max() > 1:
        rsna_event('rad_invalid_values_repaired', nonfinite=int((~_rad_np.isfinite(values)).sum()), out_of_range=int(((values < 0) | (values > 1)).sum()))
        frame[_RAD_LABELS] = _rad_np.clip(_rad_np.where(_rad_np.isfinite(values), values, 0.5), 0, 1)


class _RadLayoutPrefetch:
    """One CPU layout builder overlaps GPU work; GPU head shapes are model-owned."""
    def __init__(self, prepare, configurations, release):
        from concurrent.futures import ThreadPoolExecutor
        self.prepare = prepare
        self.configurations = list(configurations)
        self.release = release
        self.executor = ThreadPoolExecutor(max_workers=1)
        self.pending = None
        self.next_index = 0
        self.acquired = None

    def _submit(self):
        if self.next_index < len(self.configurations):
            config = self.configurations[self.next_index]
            self.next_index += 1
            self.pending = self.executor.submit(self.prepare, *config)

    def __enter__(self):
        self._submit()
        return self

    def take(self):
        if self.acquired is not None:
            raise RuntimeError('release the current Rad layout before taking another')
        if self.pending is None:
            raise StopIteration('all Rad layouts consumed')
        future, self.pending = self.pending, None
        try:
            self.acquired = future.result()
        except BaseException:
            self.executor.shutdown(wait=True, cancel_futures=True)
            raise
        self._submit()
        return self.acquired

    def release_current(self):
        if self.acquired is not None:
            self.release(self.acquired[1])  # (study IDs, pixels, masks)
            self.acquired = None

    def __exit__(self, exc_type, exc, tb):
        self.release_current()
        pending, self.pending = self.pending, None
        self.executor.shutdown(wait=True, cancel_futures=True)
        if pending is not None and not pending.cancelled():
            try:
                extra = pending.result()
                self.release(extra[1])
            except BaseException:
                if exc_type is None:
                    raise
        return False


def _rad_main():
    """BTKD Rad/calibrator graph unchanged; diagnostic-only twin work removed.

    BTKD's E13-on-E11 second input is intentional in its fitted calibrator.
    Keep that pair together; do not claim standalone E13 train/serve parity.
    """
    started=_rad_time.time();work=_RadPath('/kaggle/working');primary=work/'_pipeline_stage.csv'
    test=_rad_pd.read_csv(ROOT/'test.csv',dtype={'StudyInstanceUID':str})
    ids=test.StudyInstanceUID.tolist()
    baseline=rsna_frame(_rad_pd.read_csv(primary,dtype={'StudyInstanceUID':str}),ids,_RAD_LABELS,'Rad parent')
    device=_rad_torch.device('cuda:0')
    encoder,reference_heads,encoder_path,reference_heads_path=_rad_load_models(device)
    tser=_rad_pd.read_csv(ROOT/'test_series.csv',dtype={'StudyInstanceUID':str,'SeriesInstanceUID':str})
    plane=dict(zip(tser.SeriesInstanceUID,tser.Anatomical_Plane))
    # One annotation pass. All three layouts use the identical legacy annotation rules.
    headers=annotate(walk('test_series'))
    side_map=lat_of(headers,'rad common')
    def prepare_layout(layout,crop,tag):
        globals().update(SLOTS=list(layout),N_SLOT=len(layout),CACHE_SLICES=8,IMG=224,CACHE_IMG=224,CROP_MM=float(crop),RULES=dict(RULES_LEGACY))
        value=build_cache(pick_slots(headers,plane),plane,side_map,tag)
        if value[0]!=ids:
            rsna_release_pixels(value[1])
            raise RuntimeError('Rad cache order changed')
        return value
    def encode_layout(prefetch):
        studies,pixels,mask=prefetch.take()
        try:
            f,tm=_rad_encode(encoder,pixels,mask,device)
            return f,tm,int(tm.sum())
        finally:
            prefetch.release_current()
    def predict_heads(heads,f,mask,tag):
        if len(heads)!=5: raise RuntimeError(tag+': five heads required')
        probs=_rad_np.stack([_rad_predict_head(h,f,mask,device) for h in heads])
        rsna_finite(probs,tag,probability=True);rsna_save_predictions(tag,ids,probs,_RAD_LABELS)
        return probs.mean(0)
    first=[('SAG_FS','Sagittal',None,True),('COR_FS','Coronal',None,True),('AX_FS','Axial',None,True)]
    # E13 has four slots; construct its model before the producer mutates globals.
    globals().update(SLOTS=list(_RAD_E13_SLOTS),N_SLOT=4,CACHE_SLICES=8)
    e13_heads,e13_path=_rad_load_e13_heads(device)
    configurations=[(first,10000.,'rad_e10'),(_RAD_E13_SLOTS,130.,'rad_e13'),(_RAD_E11_SLOTS,130.,'rad_legacy_second')]
    with _RadLayoutPrefetch(prepare_layout,configurations,rsna_release_pixels) as prefetch:
        f,m,tokens0=encode_layout(prefetch)
        reference_rank=_rad_rank_columns(predict_heads(reference_heads,f,m,'rad_e10_raw'))
        del f,m,reference_heads
        f,m,tokens1=encode_layout(prefetch)
        e13_rank=_rad_rank_columns(predict_heads(e13_heads,f,m,'rad_e13_raw'))
        reference_rank=_rad_rank_columns((1-_RAD_E13_MEMBER_WEIGHT)*reference_rank+_RAD_E13_MEMBER_WEIGHT*e13_rank)
        del f,m,e13_rank
        baseline_rank=_rad_rank_columns(baseline[_RAD_LABELS].to_numpy())
        candidate=baseline.copy()
        for j,label in enumerate(_RAD_LABELS):
            if label not in _RAD_EXCLUDE:
                candidate[label]=(1-_RAD_ALPHA)*baseline_rank[:,j]+_RAD_ALPHA*reference_rank[:,j]
        _rad_validate(candidate,ids)
        f,m,tokens2=encode_layout(prefetch)
        second_rank=_rad_rank_columns(predict_heads(e13_heads,f,m,'rad_legacy_second_raw'))
        del f,m,e13_heads
    branch=candidate.copy()
    branch[_RAD_LABELS]=_rad_rank_columns((1-_RAD_V48_SECOND_ALPHA)*_rad_rank_columns(candidate[_RAD_LABELS].to_numpy())+_RAD_V48_SECOND_ALPHA*second_rank)
    cal=_rad_json.loads(_rad_zlib.decompress(_rad_b64.b64decode(_RAD_CAL_PAYLOAD)).decode())
    protocol=_rad_pd.DataFrame(index=_rad_pd.Index(ids,name='StudyInstanceUID'))
    protocol['n_series']=tser.groupby('StudyInstanceUID').size().reindex(ids).fillna(0)
    for pl in ('Sagittal','Coronal','Axial'):
        part=tser[tser.Anatomical_Plane.astype(str)==pl]
        protocol[f'n_{pl[:3]}']=part.groupby('StudyInstanceUID').size().reindex(ids).fillna(0)
    for flag in ('Fat_Suppression','Fluid_Sensitive'):
        marked=tser[_rad_pd.to_numeric(tser[flag],errors='coerce').fillna(0)>0]
        protocol[flag[:3]]=marked.groupby('StudyInstanceUID').size().reindex(ids).fillna(0)
        for pl in ('Sagittal','Coronal','Axial'):
            part=marked[marked.Anatomical_Plane.astype(str)==pl]
            protocol[f'{flag[:3]}_{pl[:3]}']=part.groupby('StudyInstanceUID').size().reindex(ids).fillna(0)
    if list(protocol.columns)!=list(cal['protocol_columns']): raise RuntimeError('calibrator protocol drift')
    mean=(baseline_rank+reference_rank+second_rank)/3.
    blocks=[baseline_rank,reference_rank,second_rank,reference_rank-baseline_rank,second_rank-baseline_rank,mean]
    for group in cal['groups']:
        blocks.append(mean[:,[_RAD_LABELS.index(t) for t in group]].mean(axis=1,keepdims=True))
    blocks.append(protocol.to_numpy(_rad_np.float64))
    x=_rad_np.concatenate(blocks,axis=1); center=_rad_np.asarray(cal['mean']);spread=_rad_np.asarray(cal['scale']);coef=_rad_np.asarray(cal['coef']);bias=_rad_np.asarray(cal['intercept'])
    if x.shape[1]!=coef.shape[1] or not bool((spread>0).all()): raise RuntimeError('calibrator shape/scale drift')
    adjusted=_rad_rank_columns(((x-center)/spread)@coef.T+bias)
    values=branch[_RAD_LABELS].to_numpy(_rad_np.float64).copy()
    for j,label in enumerate(_RAD_LABELS):
        if label in set(cal['gate']): values[:,j]=.6*values[:,j]+.4*adjusted[:,j]
    final=branch.copy();final[_RAD_LABELS]=_rad_rank_columns(values)
    _rad_validate(final,ids);rsna_save_predictions('btkd_parent_after_rad',ids,final[_RAD_LABELS].to_numpy(),_RAD_LABELS)
    tmp=primary.with_suffix('.tmp');final.to_csv(tmp,index=False);_rad_os.replace(tmp,primary)
    globals()['V18_CALIBRATOR_APPLIED']=True
    rsna_json(work/'v558_rad_receipt.json',{'status':'COMPLETE','graph':'BTKD legacy Rad plus its matched calibrator preserved','reference_heads_sha256':_RAD_REFERENCE_HEADS_SHA256,'e13_sha256':_RAD_E13_HEADS_SHA256,'encoder_sha256':_RAD_ENCODER_SHA256,'legacy_cross_layout_pass_preserved':True,'standalone_cross_layout_train_parity_claimed':False,'unused_alternate_family_removed':True,'encoder_replicas':2,'layout_cpu_prefetch':True,'model_owned_head_shape':True,'tokens':[tokens0,tokens1,tokens2],'seconds':_rad_time.time()-started})
    del encoder;_rad_gc.collect()
    for d in range(2):
        with _rad_torch.cuda.device(d): _rad_torch.cuda.empty_cache()
    _rad_log('BTKD reference Rad/calibrator complete; unused alternate-head branch removed')


rsna_phase('radimagenet', 'START')
_rad_main()
rsna_phase('radimagenet', 'COMPLETE')

_MA_RAD_STAGE = pd.read_csv('/kaggle/working/_pipeline_stage.csv', dtype={'StudyInstanceUID':str})[MA_LABELS].to_numpy().copy()


## 8. Four Raptor views, residual CoAtNet, and D4

In [ ]:
"""Two attention-pooling evaluations from ONE unchanged backbone feature bank.

The reference prediction retains every original window. The alternate head
uses the same trained head on same-series windows only, when enough remain.
Memory retries never reduce the selected windows or omit a model.
"""
def v3_raptor_pair(model, xwins, device, same_series, settings, event=None):
    import gc
    import numpy as np
    import torch
    dev=torch.device(device)
    if model.training:
        raise RuntimeError('Raptor must be in evaluation mode')
    mask=np.asarray(same_series)
    policy=v3_mask_policy(mask,settings)
    if len(xwins)!=len(mask) or xwins.ndim!=4 or xwins.shape[1]!=3:
        raise ValueError('Raptor window/mask shape mismatch')
    if not torch.isfinite(xwins).all():
        raise RuntimeError('Raptor input has nonfinite values')
    chunk=int(settings['raptor_micro_images'])
    amp=dev.type=='cuda'
    attempted_chunks=[]
    backbone_calls=0
    def emit(kind,**info):
        if event is not None: event(kind,**info)
    def forward(enabled, micro):
        nonlocal backbone_calls
        with torch.inference_mode(), torch.autocast(
                'cuda' if dev.type=='cuda' else 'cpu',dtype=torch.float16,
                enabled=enabled):
            pieces=[]
            for start in range(0,len(xwins),micro):
                backbone_calls+=1
                pieces.append(model.backbone(xwins[start:start+micro].to(dev)))
            feats=torch.cat(pieces,dim=0).unsqueeze(0)
            # This is the source's original order of operations. No change to
            # the encoder, original windows, normalization or original head.
            anchor=torch.sigmoid(model.head(feats).float())[0]
            if policy['applied']:
                keep=torch.as_tensor(mask,dtype=torch.bool,device=dev)
                coherent=torch.sigmoid(model.head(feats[:,keep]).float())[0]
            else:
                coherent=anchor
            return anchor.cpu().numpy(),coherent.cpu().numpy()
    while True:
        attempted_chunks.append(chunk)
        oom=False
        try:
            original,alternate=forward(amp,chunk)
        except torch.cuda.OutOfMemoryError:
            if chunk==1:
                raise
            oom=True
        if oom:
            # Outside the exception block: the failed forward's traceback and
            # tensor references have been released before empty_cache().
            before=chunk; chunk=max(1,chunk//2)
            gc.collect()
            if dev.type=='cuda':
                with torch.cuda.device(dev): torch.cuda.empty_cache()
            emit('raptor_v3_oom_retry',from_micro=before,to_micro=chunk,
                 windows=len(xwins),device=str(dev))
            continue
        if not np.isfinite(original).all() or not np.isfinite(alternate).all():
            if amp:
                amp=False
                emit('raptor_v3_nonfinite_retry_fp32',windows=len(xwins),device=str(dev))
                continue
            raise RuntimeError('Nonfinite Raptor prediction after FP32 retry')
        if original.shape!=(12,) or alternate.shape!=(12,):
            raise RuntimeError('Raptor head must return twelve findings')
        if ((original<0)|(original>1)).any() or ((alternate<0)|(alternate>1)).any():
            raise RuntimeError('Raptor head returned values outside [0,1]')
        report={**policy,'micro_images':chunk,'attempted_micro_images':attempted_chunks,
                'backbone_calls':backbone_calls,'executed_amp':bool(amp),
                'head_evaluations_per_successful_pass':2 if policy['applied'] else 1,
                'max_probability_change':float(np.max(np.abs(alternate-original)))}
        return original,alternate,report


"""Encode every reference window unchanged, then only the new gap windows.

The feature bank is reused for expanded all-window and series-coherent heads.
An added-window OOM does not force the successful original encoding to rerun.
No model member or selected reference window is dropped to recover memory.
"""
def v4_raptor_extended(model, xwins, device, same_series, gap_windows, settings, event=None):
    import gc
    import numpy as np
    import torch
    dev=torch.device(device)
    if model.training:
        raise RuntimeError('Raptor must be in evaluation mode')
    mask=np.asarray(same_series)
    policy=v3_mask_policy(mask,settings)
    if xwins.ndim!=4 or xwins.shape[1]!=3 or len(xwins)!=len(mask):
        raise ValueError('Original Raptor window/mask shape mismatch')
    if gap_windows is None:
        gap_windows=xwins.new_empty((0,*xwins.shape[1:]))
    if gap_windows.ndim!=4 or tuple(gap_windows.shape[1:])!=tuple(xwins.shape[1:]):
        raise ValueError('Gap window shape mismatch')
    if len(gap_windows)>settings['gap_max_windows']:
        raise ValueError('Too many gap windows')
    if settings['gap_view_alpha']==0. and len(gap_windows):
        raise ValueError('Disabled gap branch must not encode extra windows')
    if not torch.isfinite(xwins).all() or not torch.isfinite(gap_windows).all():
        raise RuntimeError('Nonfinite Raptor image input')
    def emit(kind,**info):
        if event is not None:
            event(kind,**info)
    def cleanup():
        gc.collect()
        if dev.type=='cuda':
            with torch.cuda.device(dev):torch.cuda.empty_cache()
    def check_predictions(*arrays):
        for arr in arrays:
            if arr.shape!=(12,) or not np.isfinite(arr).all() or ((arr<0)|(arr>1)).any():
                return False
        return True
    micro=int(settings['raptor_micro_images']); amp=dev.type=='cuda'
    attempts=[]; calls=0
    def reference_forward(enabled,chunk):
        nonlocal calls
        with torch.inference_mode(),torch.autocast('cuda' if dev.type=='cuda' else 'cpu',dtype=torch.float16,enabled=enabled):
            pieces=[]
            for start in range(0,len(xwins),chunk):
                calls+=1
                pieces.append(model.backbone(xwins[start:start+chunk].to(dev)))
            feats=torch.cat(pieces,dim=0).unsqueeze(0)
            original=torch.sigmoid(model.head(feats).float())[0]
            if policy['applied']:
                keep=torch.as_tensor(mask,dtype=torch.bool,device=dev)
                coherent=torch.sigmoid(model.head(feats[:,keep]).float())[0]
            else:
                coherent=original
            return feats,original.cpu().numpy(),coherent.cpu().numpy()
    while True:
        attempts.append(micro);oom=False
        try:
            features,original,coherent=reference_forward(amp,micro)
        except torch.cuda.OutOfMemoryError:
            if micro==1:raise
            oom=True
        if oom:
            old=micro;micro=max(1,micro//2);cleanup()
            emit('raptor_v4_reference_oom_retry',from_micro=old,to_micro=micro,windows=len(xwins))
            continue
        if not check_predictions(original,coherent):
            if amp:
                del features; amp=False;cleanup()
                emit('raptor_v4_reference_retry_fp32',windows=len(xwins));continue
            raise RuntimeError('Invalid reference Raptor predictions after FP32 retry')
        break
    base_report={**policy,'micro_images':micro,'attempted_micro_images':attempts,
                 'backbone_calls':calls,'executed_amp':bool(amp),
                 'head_evaluations_per_successful_pass':2 if policy['applied'] else 1,
                 'max_probability_change':float(np.max(np.abs(coherent-original)))}
    if not len(gap_windows):
        del features
        return original,coherent,original.copy(),coherent.copy(),{
            **base_report,'extra_windows':0,'extra_micro_images':0,
            'extra_attempted_micro_images':[],'extra_backbone_calls':0,
            'extra_executed_amp':False,'coverage_max_probability_change':0.}
    extra_micro=micro;extra_amp=amp;extra_attempts=[];extra_calls=0
    def extended_forward(enabled,chunk):
        nonlocal extra_calls
        with torch.inference_mode(),torch.autocast('cuda' if dev.type=='cuda' else 'cpu',dtype=torch.float16,enabled=enabled):
            pieces=[]
            for start in range(0,len(gap_windows),chunk):
                extra_calls+=1
                pieces.append(model.backbone(gap_windows[start:start+chunk].to(dev)))
            extra=torch.cat(pieces,dim=0).unsqueeze(0)
            old=features if enabled else features.float()
            if not enabled:extra=extra.float()
            expanded=torch.cat([old,extra],dim=1)
            p=torch.sigmoid(model.head(expanded).float())[0]
            if policy['applied']:
                keep=torch.as_tensor(mask,dtype=torch.bool,device=dev)
                clean=torch.cat([old[:,keep],extra],dim=1)
                q=torch.sigmoid(model.head(clean).float())[0]
            else:q=p
            return p.cpu().numpy(),q.cpu().numpy()
    while True:
        extra_attempts.append(extra_micro);oom=False
        try:
            extended,extended_coherent=extended_forward(extra_amp,extra_micro)
        except torch.cuda.OutOfMemoryError:
            if extra_micro==1:raise
            oom=True
        if oom:
            old=extra_micro;extra_micro=max(1,extra_micro//2);cleanup()
            emit('raptor_v4_extra_oom_retry',from_micro=old,to_micro=extra_micro,
                 extra_windows=len(gap_windows),original_features_reused=True)
            continue
        if not check_predictions(extended,extended_coherent):
            if extra_amp:
                extra_amp=False;cleanup()
                emit('raptor_v4_extra_retry_fp32',extra_windows=len(gap_windows),
                     original_features_reused=True);continue
            raise RuntimeError('Invalid gap-view prediction after full-precision head retry')
        break
    del features
    return original,coherent,extended,extended_coherent,{
        **base_report,'extra_windows':len(gap_windows),'extra_micro_images':extra_micro,
        'extra_attempted_micro_images':extra_attempts,'extra_backbone_calls':extra_calls,
        'extra_executed_amp':bool(extra_amp),
        'extra_head_evaluations_per_successful_pass':2 if policy['applied'] else 1,
        'coverage_max_probability_change':float(np.max(np.abs(extended-original)))}


def v5_mass_probability(model, features, log_weights, reference_probability):
    """Same trained head plus a sampling prior; no backbone invocation.

    The new softmax/pooling reduction is FP32. A constant log prior cancels,
    so those cases reuse the actual reference tensor rather than relying on
    floating-point cancellation. The ordinary v4 head is not rewritten.
    """
    import torch
    import numpy as np
    weights = np.asarray(log_weights, dtype=np.float64)
    if model.training or features.ndim != 3 or features.shape[0] != 1:
        raise ValueError('Expected one evaluation-mode feature sequence')
    if weights.shape != (features.shape[1],) or not len(weights) or not np.isfinite(weights).all() or (weights > 0).any():
        raise ValueError('Invalid acquisition log-mass vector')
    if reference_probability.shape != (12,):
        raise ValueError('Expected twelve reference probabilities')
    if np.ptp(weights) == 0:
        return reference_probability
    if not torch.isfinite(features).all():
        raise RuntimeError('Nonfinite cached features')
    # Learnable layers retain the caller's precision convention.
    h = model.norm(features)
    logits = model.att(h)
    # Explicitly disable autocast: merely calling .float() is insufficient for
    # operations (such as einsum/matmul) that autocast can downcast again.
    with torch.autocast(device_type=features.device.type, enabled=False):
        prior = torch.as_tensor(weights, device=features.device, dtype=torch.float32).view(1, -1, 1)
        attention = torch.softmax(logits.float() + prior, dim=1, dtype=torch.float32)
        pooled = torch.einsum('bkn,bkf->bnf', attention, h.float())
        result = (pooled * model.clsW.float()).sum(-1) + model.clsb.float()
        return torch.sigmoid(result)[0]


def v5_raptor_extended(model, xwins, device, same_series, gap_windows, settings,
                       mass_plan, event=None):
    import gc
    import numpy as np
    import torch
    dev=torch.device(device)
    if model.training:
        raise RuntimeError('Raptor must be in evaluation mode')
    mask=np.asarray(same_series)
    policy=v3_mask_policy(mask,settings)
    if xwins.ndim!=4 or xwins.shape[1]!=3 or len(xwins)!=len(mask):
        raise ValueError('Original Raptor window/mask shape mismatch')
    if gap_windows is None:
        gap_windows=xwins.new_empty((0,*xwins.shape[1:]))
    if gap_windows.ndim!=4 or tuple(gap_windows.shape[1:])!=tuple(xwins.shape[1:]):
        raise ValueError('Gap window shape mismatch')
    mass_plan=v5_validate_plan(mass_plan, mask, len(gap_windows),
                              settings['sampling_mass_enabled'])
    log_mass=np.asarray(mass_plan['log_weights'],np.float64)
    if len(gap_windows)>settings['gap_max_windows']:
        raise ValueError('Too many gap windows')
    if settings['gap_view_alpha']==0. and len(gap_windows):
        raise ValueError('Disabled gap branch must not encode extra windows')
    if not torch.isfinite(xwins).all() or not torch.isfinite(gap_windows).all():
        raise RuntimeError('Nonfinite Raptor image input')
    def emit(kind,**info):
        if event is not None:
            event(kind,**info)
    def cleanup():
        gc.collect()
        if dev.type=='cuda':
            with torch.cuda.device(dev):torch.cuda.empty_cache()
    def check_predictions(*arrays):
        for arr in arrays:
            if arr.shape!=(12,) or not np.isfinite(arr).all() or ((arr<0)|(arr>1)).any():
                return False
        return True
    micro=int(settings['raptor_micro_images']); amp=dev.type=='cuda'
    attempts=[]; calls=0
    def reference_forward(enabled,chunk):
        nonlocal calls
        with torch.inference_mode(),torch.autocast('cuda' if dev.type=='cuda' else 'cpu',dtype=torch.float16,enabled=enabled):
            pieces=[]
            for start in range(0,len(xwins),chunk):
                calls+=1
                pieces.append(model.backbone(xwins[start:start+chunk].to(dev)))
            feats=torch.cat(pieces,dim=0).unsqueeze(0)
            original=torch.sigmoid(model.head(feats).float())[0]
            if policy['applied']:
                keep=torch.as_tensor(mask,dtype=torch.bool,device=dev)
                coherent=torch.sigmoid(model.head(feats[:,keep]).float())[0]
            else:
                coherent=original
            return feats,original.cpu().numpy(),coherent.cpu().numpy()
    while True:
        attempts.append(micro);oom=False
        try:
            features,original,coherent=reference_forward(amp,micro)
        except torch.cuda.OutOfMemoryError:
            if micro==1:raise
            oom=True
        if oom:
            old=micro;micro=max(1,micro//2);cleanup()
            emit('raptor_v5_reference_oom_retry',from_micro=old,to_micro=micro,windows=len(xwins))
            continue
        if not check_predictions(original,coherent):
            if amp:
                del features; amp=False;cleanup()
                emit('raptor_v5_reference_retry_fp32',windows=len(xwins));continue
            raise RuntimeError('Invalid reference Raptor predictions after FP32 retry')
        break
    base_report={**policy,'micro_images':micro,'attempted_micro_images':attempts,
                 'backbone_calls':calls,'executed_amp':bool(amp),
                 'head_evaluations_per_successful_pass':2 if policy['applied'] else 1,
                 'max_probability_change':float(np.max(np.abs(coherent-original))),
                 'sampling_mass_applied':mass_plan['applied'],
                 'sampling_mass_plan_sha256':mass_plan['sha256'],
                 'sampling_mass_extra_encoder_images':0}
    if not len(gap_windows):
        del features
        return original,coherent,original.copy(),coherent.copy(),original.copy(),coherent.copy(),{
            **base_report,'extra_windows':0,'extra_micro_images':0,
            'extra_attempted_micro_images':[],'extra_backbone_calls':0,
            'extra_executed_amp':False,'coverage_max_probability_change':0.,
            'sampling_mass_head_evaluations':0,'mass_max_probability_change':0.}
    extra_micro=micro;extra_amp=amp;extra_attempts=[];extra_calls=0
    def extended_forward(enabled,chunk):
        nonlocal extra_calls
        with torch.inference_mode(),torch.autocast('cuda' if dev.type=='cuda' else 'cpu',dtype=torch.float16,enabled=enabled):
            pieces=[]
            for start in range(0,len(gap_windows),chunk):
                extra_calls+=1
                pieces.append(model.backbone(gap_windows[start:start+chunk].to(dev)))
            extra=torch.cat(pieces,dim=0).unsqueeze(0)
            old=features if enabled else features.float()
            if not enabled:extra=extra.float()
            expanded=torch.cat([old,extra],dim=1)
            p=torch.sigmoid(model.head(expanded).float())[0]
            if policy['applied']:
                keep=torch.as_tensor(mask,dtype=torch.bool,device=dev)
                clean=torch.cat([old[:,keep],extra],dim=1)
                q=torch.sigmoid(model.head(clean).float())[0]
            else:q=p
            if mass_plan['applied']:
                balanced=v5_mass_probability(model,expanded,log_mass,p)
                if policy['applied']:
                    selected=np.concatenate([mask,np.ones(len(gap_windows),bool)])
                    balanced_coherent=v5_mass_probability(model,clean,log_mass[selected],q)
                else:
                    balanced_coherent=balanced
            else:
                balanced,balanced_coherent=p,q
            return (p.cpu().numpy(),q.cpu().numpy(),balanced.cpu().numpy(),
                    balanced_coherent.cpu().numpy())
    while True:
        extra_attempts.append(extra_micro);oom=False
        try:
            extended,extended_coherent,balanced,balanced_coherent=extended_forward(extra_amp,extra_micro)
        except torch.cuda.OutOfMemoryError:
            if extra_micro==1:raise
            oom=True
        if oom:
            old=extra_micro;extra_micro=max(1,extra_micro//2);cleanup()
            emit('raptor_v5_extra_oom_retry',from_micro=old,to_micro=extra_micro,
                 extra_windows=len(gap_windows),original_features_reused=True)
            continue
        if not check_predictions(extended,extended_coherent,balanced,balanced_coherent):
            if extra_amp:
                extra_amp=False;cleanup()
                emit('raptor_v5_extra_retry_fp32',extra_windows=len(gap_windows),
                     original_features_reused=True);continue
            raise RuntimeError('Invalid gap-view prediction after full-precision head retry')
        break
    del features
    return original,coherent,extended,extended_coherent,balanced,balanced_coherent,{
        **base_report,'sampling_mass_head_evaluations':
            ((1 + int(policy['applied'] and np.ptp(log_mass[np.concatenate([mask,np.ones(len(gap_windows),bool)])]) > 0)) if mass_plan['applied'] else 0),
        'mass_max_probability_change':float(np.max(np.abs(balanced-extended))),
        'extra_windows':len(gap_windows),'extra_micro_images':extra_micro,
        'extra_attempted_micro_images':extra_attempts,'extra_backbone_calls':extra_calls,
        'extra_executed_amp':bool(extra_amp),
        'extra_head_evaluations_per_successful_pass':2 if policy['applied'] else 1,
        'coverage_max_probability_change':float(np.max(np.abs(extended-original)))}


In [ ]:
if globals().get('_DINOV2_MATCHED_MEMBERS') != 20:
    raise RuntimeError('DINOv2 20/20 fingerprint gate did not pass')
import gc as _ke_gc
import os as _ke_os
import time as _ke_time
from concurrent.futures import ThreadPoolExecutor as _KeThreadPool
import numpy as _ke_np
import pandas as _ke_pd
from pathlib import Path as _KePath

_ke_primary = _KePath('/kaggle/working/_pipeline_stage.csv')
_ke_ours = _ke_pd.read_csv(_ke_primary, dtype={'StudyInstanceUID': str})
_KE_LAB = [c for c in _ke_ours.columns if c != 'StudyInstanceUID']

_KE_SRC = r'''
import os, glob, time, gc, hashlib
os.environ.setdefault('HF_HUB_OFFLINE', '1')
os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')
os.environ.setdefault('HF_HUB_DISABLE_TELEMETRY', '1')
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import timm
torch.backends.cudnn.benchmark = False  # ragged 94-window chunks: no repeated autotuning
torch.backends.cuda.matmul.allow_tf32 = True
IMG = 336
CROP_MM = 140.0
SPAN_LO, SPAN_HI = 0.02, 0.98
SLOTS = [("Sagittal", 1, 18), ("Sagittal", 0, 14),
         ("Coronal", 1, 12), ("Coronal", 0, 8), ("Axial", -1, 12)]
MAXS = sum(slot[2] for slot in SLOTS)
K_EVAL = 62
NORM = "imagenet"
LAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
       "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
       "Contusion", "Fracture"]
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
_SLOTS64 = [("Sagittal", 1, 18), ("Sagittal", 0, 14),
            ("Coronal", 1, 12), ("Coronal", 0, 8), ("Axial", -1, 12)]
_SLOTS44 = [("Sagittal", 1, 12), ("Sagittal", 0, 10),
            ("Coronal", 1, 8), ("Coronal", 0, 6), ("Axial", -1, 8)]
ARMS = [
    {"name": "maxspan-v5", "file": "raptor_ft_coatnet_v5_full_swa.pt",
     "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "res": 384,
     "img": 336, "slots": _SLOTS64, "span": (0.02, 0.98), "k_eval": 62,
     "reverse": False, "w": 0.60},
    {"name": "native384dense-v10", "file": "raptor_ft_coatnet_v10_full.pt",
     "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "res": 384,
     "img": 384, "slots": _SLOTS64, "span": (0.02, 0.98), "k_eval": 62,
     "reverse": False, "w": 0.10},
    {"name": "maxspan-v5-reverse", "file": "raptor_ft_coatnet_v5_full_swa.pt",
     "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "res": 384,
     "img": 336, "slots": _SLOTS64, "span": (0.02, 0.98), "k_eval": 62,
     "reverse": True, "w": 0.10},
    {"name": "native384-v8", "file": "raptor_ft_coatnet_v8_full_swa.pt",
     "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "res": 384,
     "img": 384, "slots": _SLOTS44, "span": (0.06, 0.94), "k_eval": 42,
     "reverse": False, "w": 0.20},
]

def build_backbone(arch, pretrained=False):
    hybrid = arch.startswith(('maxvit', 'maxxvit', 'coatnet', 'coat_', 'convnext'))
    is_vit = not hybrid and any((k in arch for k in ('vit', 'deit', 'dinov2', 'eva', 'beit')))
    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3)
    if is_vit:
        kw.update(global_pool='token', dynamic_img_size=True)
    else:
        kw.update(global_pool='avg')
    return timm.create_model(arch, **kw)

class RaptorClassifier(nn.Module):

    def __init__(self, backbone, F_dim=768, n=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.n = n

    def encode(self, x):
        B, K = x.shape[:2]
        f = self.backbone(x.flatten(0, 1))
        return f.view(B, K, -1)

    def head(self, feats):
        h = self.norm(feats)
        a = self.att(h)
        a = torch.softmax(a, dim=1)
        pooled = torch.einsum('bkn,bkf->bnf', a, h)
        logits = (pooled * self.clsW).sum(-1) + self.clsb
        return logits

    def forward(self, x):
        return self.head(self.encode(x))

def load_model(pt_path, arch_default, res_default, device, ngpu=1):
    ck = torch.load(pt_path, map_location='cpu', weights_only=False)
    arch = ck.get('arch', arch_default)
    if arch != arch_default:
        raise RuntimeError(f'Raptor architecture drift: {arch} != {arch_default}')
    ck_res = int(ck.get('res', res_default))
    bb = build_backbone(arch, pretrained=False)
    model = RaptorClassifier(bb, F_dim=bb.num_features)
    rsna_strict_load(model,ck['model'],str(pt_path))
    rsna_event('raptor_checkpoint',file=str(pt_path),sha256=rsna_sha(pt_path),arch=arch,res=ck_res)
    model.eval().to(device)
    del ck
    gc.collect()
    return (model, ck_res)



def _eval_centers(mask, D, k):
    valid = np.where(mask > 0)[0]
    if len(valid) < 3:
        valid = np.arange(min(3, D))
    lo, hi = (int(valid.min()), int(valid.max()))
    cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]
    if not cs:
        cs = [max(1, min((lo + hi) // 2, D - 2))]
    idx = np.linspace(0, len(cs) - 1, k).round().astype(int)
    return [cs[i] for i in idx]

def eval_windows(vol, mask, k, res, norm=NORM):
    """Resize each selected source plane once, then gather the unchanged RGB triplets."""
    volume = np.asarray(vol)
    depth = int(volume.shape[0])
    centers = np.asarray(_eval_centers(mask, depth, k), dtype=np.int64)
    centers = np.clip(centers, 1, depth - 2)
    if tuple(volume.shape[-2:]) == (res, res):
        # Native384 needs no interpolation; the existing small per-triplet copy
        # avoids a slower large advanced-index gather and preserves its exact math.
        wins = np.empty((len(centers), 3, res, res), np.float32)
        for j, center in enumerate(centers):
            wins[j] = np.stack([volume[center - 1], volume[center],
                                volume[center + 1]], axis=0).astype(np.float32) / 255.0
        x = torch.from_numpy(wins)
        if norm == 'imagenet':
            x = (x - _MEAN) / _STD
        return x
    triplets = centers[:, None] + np.asarray([-1, 0, 1], dtype=np.int64)
    # Use only planes referenced by the original eval-center recipe.
    unique, inverse = np.unique(triplets.reshape(-1), return_inverse=True)
    source = volume[unique].astype(np.float32) / 255.0
    planes = torch.from_numpy(source)
    if tuple(planes.shape[-2:]) != (res, res):
        resized = torch.empty((len(planes), res, res), dtype=torch.float32)
        for start in range(0, len(planes), 16):
            resized[start:start+16] = F.interpolate(
                planes[start:start+16, None], size=(res, res),
                mode='bilinear', align_corners=False)[:, 0]
        planes = resized
    x = planes[torch.from_numpy(inverse.reshape(-1, 3))]
    if norm == 'imagenet':
        x = (x - _MEAN) / _STD
    return x





def rankpct(x):
    return rsna_rank01(x)

def _make_reader():
    import pydicom, cv2
    from pydicom.pixel_data_handlers.util import apply_modality_lut

    def order_and_meta(sdir):
        fs = glob.glob(sdir + '/*.dcm')
        recs = []
        ps_list = []
        for f in fs:
            try:
                h = pydicom.dcmread(f, stop_before_pixels=True)
                iop = getattr(h, 'ImageOrientationPatient', None)
                ipp = getattr(h, 'ImagePositionPatient', None)
                if iop is not None and ipp is not None and (len(iop) == 6):
                    r = np.array(iop[:3], float)
                    c = np.array(iop[3:], float)
                    n = np.cross(r, c)
                    pos = float(np.dot(np.array(ipp, float), n))
                else:
                    instance = getattr(h, 'InstanceNumber', None)
                    if instance is None:
                        rsna_event('raptor_order_index_fallback', file=str(f)); instance = len(recs)
                    pos = float(instance)
                if not np.isfinite(pos):
                    rsna_event('raptor_order_nonfinite_fallback', file=str(f)); pos = float(len(recs))
                ps = getattr(h, 'PixelSpacing', None)
                ps = float(ps[0]) if ps is not None else 0.5
                ps_list.append(ps)
                recs.append((pos, f, ps))
            except Exception as exc:
                rsna_event('raptor_order_unreadable_skipped', file=str(f), error=f'{type(exc).__name__}: {exc}'); continue
        recs.sort(key=lambda x: (x[0], x[1]))
        med_ps = float(np.median(ps_list)) if ps_list else 0.5
        return ([(f, ps) for _, f, ps in recs], med_ps)

    def read_px(f):
        d = pydicom.dcmread(f)
        a = apply_modality_lut(d.pixel_array, d).astype(np.float32)
        if str(getattr(d, 'PhotometricInterpretation', '')) == 'MONOCHROME1':
            a = a.max() - a
        return a

    def mm_crop_resize(a, ps):
        h, w = a.shape
        cpx = int(round(CROP_MM / max(ps, 0.001)))
        cpx = min(cpx, min(h, w))
        y0 = (h - cpx) // 2
        x0 = (w - cpx) // 2
        a = a[y0:y0 + cpx, x0:x0 + cpx]
        return cv2.resize(a, (IMG, IMG), interpolation=cv2.INTER_AREA)
    return (order_and_meta, read_px, mm_crop_resize)

def _pick_series_for_slot(rows, plane, fluid, used):
    """An unknown protocol flag is not false and must not reach int(NaN)."""
    def flag(value):
        if value is None:
            return None
        if isinstance(value, str) and value.strip().lower() in ('', 'nan', 'none', '<na>', 'unknown'):
            return None
        try:
            number = float(value)
        except (TypeError, ValueError):
            text = str(value).strip().lower()
            if text in ('true', 'yes', 'y', 't'):
                return 1
            if text in ('false', 'no', 'n', 'f'):
                return 0
            return None  # unknown code: neither false nor fatal; the plane-level fallback applies
        if not np.isfinite(number):
            return None
        if number not in (0.0, 1.0):
            return None
        return int(number)
    candidates = [r for r in rows if r['Anatomical_Plane'] == plane
                  and r['SeriesInstanceUID'] not in used]
    if fluid in (0, 1):
        preferred = [r for r in candidates if flag(r.get('Fluid_Sensitive')) == fluid]
        if preferred:
            return preferred[0]
    return candidates[0] if candidates else None



def find_test_root():
    from pathlib import Path
    for value in [os.environ.get('RSNA_COMP_ROOT', ''),
                  '/kaggle/input/competitions/rsna-knee-abnormality-detection',
                  '/kaggle/input/rsna-knee-abnormality-detection']:
        if value and (Path(value)/'test.csv').is_file():
            return value
    raise FileNotFoundError('explicit competition root absent')

def find_weight_file(fname):
    return str(_asset_find_asset(fname))
'''
_KE_NS = {'__name__': '_ke_raptor', '_asset_find_asset': _asset_find_asset, 'rsna_rank01':rsna_rank01, 'rsna_strict_load':rsna_strict_load, 'rsna_event':rsna_event, 'rsna_sha':rsna_sha}
exec(compile(_KE_SRC, '<raptor>', 'exec'), _KE_NS)
from concurrent.futures import Future as _KeFuture
import threading as _ke_threading

_ke_base_make_reader = _KE_NS['_make_reader']
_ke_order_cache = {}

# DINO already read these immutable tags. Pre-resolved Futures plug directly
# into Raptor's existing single-flight cache; ambiguous series are absent.
for _ke_series_dir, _ke_header_value in globals().get(
    'RAPTOR_HEADER_CACHE', {}
).items():
    _ke_future = _KeFuture()
    _ke_future.set_result(_ke_header_value)
    _ke_order_cache[str(_ke_series_dir)] = _ke_future
print(
    f"[raptor-fast] seeded {len(_ke_order_cache)} DINO header records",
    flush=True,
)

from collections import OrderedDict as _KeOrderedDict
_ke_pixel_cache = _KeOrderedDict()
_ke_order_cache_lock = _ke_threading.Lock()


def _ke_make_cached_reader():
    """Share immutable DICOM ordering metadata across Raptor views."""
    base_order, read_pixel, crop_resize = _ke_base_make_reader()

    def cached_order(series_dir):
        key = str(series_dir)
        owner = False
        with _ke_order_cache_lock:
            future = _ke_order_cache.get(key)
            if future is None:
                future = _KeFuture()
                _ke_order_cache[key] = future
                owner = True
        if owner:
            try:
                future.set_result(base_order(series_dir))
            except BaseException as error:
                future.set_exception(error)
                with _ke_order_cache_lock:
                    _ke_order_cache.pop(key, None)
                raise
        return future.result()

    def cached_pixel(path):
        global _ke_pixel_cache_bytes
        from collections import OrderedDict
        key=str(path)
        with _ke_order_cache_lock:
            got=_ke_pixel_cache.get(key)
            if got is not None:
                _ke_pixel_cache.move_to_end(key);return got
        value=read_pixel(path)
        if value.ndim!=2 or not _ke_np.isfinite(value).all():
            raise RuntimeError(f'Raptor invalid pixels {path}')
        value.setflags(write=False)
        with _ke_order_cache_lock:
            previous = _ke_pixel_cache.pop(key, None)
            if previous is not None:
                _ke_pixel_cache_bytes -= previous.nbytes
            _ke_pixel_cache[key] = value
            _ke_pixel_cache_bytes += value.nbytes
            while _ke_pixel_cache_bytes > (192 << 20):
                _, removed = _ke_pixel_cache.popitem(last=False)
                _ke_pixel_cache_bytes -= removed.nbytes
        return value
    return cached_order, cached_pixel, crop_resize


_KE_NS['_make_reader'] = _ke_make_cached_reader
if _ke_os.environ.get('RSNA_COMP_ROOT'):
    _KE_NS['find_test_root'] = lambda: _ke_os.environ['RSNA_COMP_ROOT']


def _ke_prepare_windows(arm, study_uid, series, series_root, reader):







    order_and_meta, read_px, _ = reader
    image_size = int(arm['img'])
    slots = list(arm['slots'])
    span_lo, span_hi = map(float, arm['span'])
    used, pools = set(), []
    rows = series.get(study_uid, [])
    for plane, fluid, count in slots:
        record = _KE_NS['_pick_series_for_slot'](rows, plane, fluid, used)
        if record is None:
            pools.append(None)
            continue
        used.add(record['SeriesInstanceUID'])
        files, median_spacing = order_and_meta(
            f"{series_root}/{study_uid}/{record['SeriesInstanceUID']}")
        if not files:
            rsna_event('raptor_acquisition_empty', study=str(study_uid), series=str(record['SeriesInstanceUID'])); pools.append(None); continue
        lo = int(len(files) * span_lo)
        hi = max(int(len(files) * span_hi) - 1, lo)
        original = _ke_np.linspace(lo, hi, count).round().astype(int)
        pools.append((files, median_spacing, lo, hi, original))
    capacities = [0 if p is None else p[3] - p[2] + 1 for p in pools]
    quotas = _dense_allocate(capacities, [s[2] for s in slots], 96)
    volume, sources = [], []
    _v4_records = []
    import cv2
    for pool, quota in zip(pools, quotas):
        if pool is None or not quota:
            continue
        files, median_spacing, lo, hi, original = pool
        picks = lo + _dense_unique_linspace(hi - lo + 1, int(quota))
        arrays = {}
        # Decode union once. Failure is explicit, not an unreported 0.5 score.
        for i in sorted(set(original.tolist()) | set(picks.tolist())):
            arrays[i] = read_px(files[i][0])
        all_pixels = _ke_np.concatenate([arrays[int(i)].ravel() for i in original])
        low, high = _ke_np.percentile(all_pixels, [2.0, 98.0])
        _v4_records.append({'files':files,'lo':lo,'hi':hi,'picks':picks.copy(),
                            'normalization':(float(low),float(high))})
        for i in picks:
            path, spacing = files[int(i)]
            a = arrays[int(i)]
            a = _ke_np.clip((a-low)/(high-low+1e-6), 0, 1)
            spacing = spacing if spacing > 0 else median_spacing
            h, w = a.shape
            c = min(int(round(140.0/max(spacing, .001))), min(h, w))
            y, x = (h-c)//2, (w-c)//2
            a = cv2.resize(a[y:y+c, x:x+c], (image_size,image_size),
                           interpolation=cv2.INTER_AREA)
            volume.append((a*255).astype(_ke_np.uint8))
            sources.append(str(path))
    if len(sources) != len(set(sources)):
        raise RuntimeError('capacity-aware sampling selected duplicate source files')
    if len(volume) < 3:
        raise RuntimeError(f'{study_uid}: fewer than three unique source slices')
    volume = _ke_np.stack(volume)
    # Presence means an acquired source, not an intensity test. A genuinely
    # black acquired slice is not interchangeable with padding.
    mask = _ke_np.ones(len(volume), dtype=_ke_np.uint8)
    windows = _KE_NS['eval_windows'](
        volume, mask, k=len(volume)-2, res=int(arm['res']), norm=_KE_NS['NORM'])
    if len(windows) != min(96, sum(capacities))-2:
        raise RuntimeError('capacity-aware sampling Raptor cardinality drift')
    import hashlib, json
    signature = hashlib.sha256(json.dumps(sources, ensure_ascii=False,
                                separators=(',', ':')).encode()).hexdigest()
    audit = {'arm':arm['name'], 'uid':str(study_uid),
             'capacities':capacities, 'quotas':quotas.tolist(),
             'source_count':len(sources), 'source_list_sha256':signature,
             'windows':len(windows), 'duplicate_sources':0}
    # Track acquisition identity for the ACTUAL eval centers. No pixels,
    # quotas, normalization or reference windows are changed by this metadata.
    _v3_sources = [_KePath(path).parent.name for path in sources]
    _v3_centers = _ke_np.asarray(_KE_NS['_eval_centers'](mask, len(volume), len(volume)-2), dtype=_ke_np.int64)
    _v3_same = v3_series_mask(_v3_sources, _v3_centers)
    if len(_v3_same) != len(windows):
        raise RuntimeError('Series mask no longer describes the actual Raptor windows')
    audit.update(same_series_windows=int(_v3_same.sum()),
                 cross_series_window_indices=_ke_np.flatnonzero(~_v3_same).tolist(),
                 series_coherence_policy=v3_mask_policy(_v3_same, MASTER))
    _v4_extra, _v4_coverage = v4_prepare_gap_windows(
        arm, windows, _v4_records, read_px, MASTER)
    audit['gap_coverage'] = _v4_coverage
    _v5_plan = v5_build_plan(_v3_sources, _v3_centers, _v4_coverage['accepted'],
                             enabled=MASTER['sampling_mass_enabled'])
    v5_validate_plan(_v5_plan, _v3_same, len(_v4_extra), MASTER['sampling_mass_enabled'])
    audit['sampling_mass'] = _v5_plan
    identity = (arm['name'], str(study_uid))
    with _ke_input_lock:
        if identity in _ke_input_ids:
            raise RuntimeError(f'Duplicate Raptor preparation: {identity}')
        _ke_input_ids.add(identity)
        with _ke_input_audit_path.open('a') as handle:
            handle.write(json.dumps(audit, sort_keys=True, allow_nan=False) + '\n')
    return {'windows':windows, 'same_series':_v3_same, 'gap_windows':_v4_extra, 'mass_plan':_v5_plan}


def _ke_infer_input(model, xwins, device):
    """Bound backbone workspace; keep all window features for one head call."""
    torch = _KE_NS['torch']
    def _forward(_amp):
        with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16, enabled=_amp):
            features = [model.backbone(xwins[i:i+8].to(device))
                        for i in range(0,len(xwins),8)]
            feats = torch.cat(features,dim=0).unsqueeze(0)
            return torch.sigmoid(model.head(feats).float())[0].cpu().numpy()
    probabilities = _forward(True)
    if not _ke_np.isfinite(probabilities).all():
        rsna_event('raptor_fp16_nonfinite_retry_fp32')
        probabilities = _forward(False)
    if not _ke_np.isfinite(probabilities).all():
        raise RuntimeError('nonfinite Raptor prediction')
    return probabilities


def _ke_prefetched_windows(arm, test_ids, series, series_root, reader):
    """Bound host memory while decoding one study ahead of its GPU forward."""
    depth = int(_ke_os.environ.get('RSNA_RAPTOR_PREFETCH', '2'))
    if not 1 <= depth <= 4: raise ValueError('Raptor prefetch must be 1..4')
    with _KeThreadPool(max_workers=1) as executor:
        pending = {}
        submit_at = 0
        while submit_at < min(depth, len(test_ids)):
            pending[submit_at] = executor.submit(
                _ke_prepare_windows,
                arm,
                test_ids[submit_at],
                series,
                series_root,
                reader,
            )
            submit_at += 1
        for study_index, study_uid in enumerate(test_ids):
            future = pending.pop(study_index)
            if submit_at < len(test_ids):
                pending[submit_at] = executor.submit(
                    _ke_prepare_windows,
                    arm,
                    test_ids[submit_at],
                    series,
                    series_root,
                    reader,
                )
                submit_at += 1
            yield study_index, study_uid, future


def _ke_run_raptor_arms():
    """Run the four unchanged Raptor arms across exactly two GPUs."""
    torch = _KE_NS['torch']
    if not torch.cuda.is_available() or torch.cuda.device_count() < 1:
        raise RuntimeError(
            f"optimized Raptor requires at least one GPU, got {torch.cuda.device_count()}"
        )
    started = _ke_time.time()
    root = _KE_NS['find_test_root']()
    series_root = root + '/test_series'
    if not _ke_os.path.isdir(series_root):
        series_root = root + '/test_images'
    test = _ke_pd.read_csv(root + '/test.csv')
    test['StudyInstanceUID'] = test['StudyInstanceUID'].astype(str)
    test_ids = test['StudyInstanceUID'].tolist()
    test_series = _ke_pd.read_csv(root + '/test_series.csv')
    test_series['StudyInstanceUID'] = test_series['StudyInstanceUID'].astype(str)
    test_series['SeriesInstanceUID'] = test_series['SeriesInstanceUID'].astype(str)
    series = {
        key: frame.to_dict('records')
        for key, frame in test_series.groupby('StudyInstanceUID')
    }
    sample = root + '/sample_submission.csv'
    columns = ['StudyInstanceUID', *_KE_NS['LAB']]
    if _ke_os.path.exists(sample):
        columns = list(_ke_pd.read_csv(sample, nrows=1).columns)
    arms = list(_KE_NS['ARMS'])
    if tuple(a['name'] for a in arms) != V3_ARM_NAMES:
        raise RuntimeError('Raptor view order changed; paired fusion must be re-audited')
    if not _ke_np.array_equal(_ke_np.asarray([a['w'] for a in arms]), V3_ARM_WEIGHTS):
        raise RuntimeError('Raptor view weights changed')
    _v3_stop = _ke_threading.Event()
    outputs = [
        _ke_np.full((len(test_ids), len(_KE_NS['LAB'])), np.nan, _ke_np.float32)
        for _ in arms
    ]
    print(
        f"[raptor-fast] {len(test_ids)} studies; balanced arm groups "
        f"cuda:0=[0,2], cuda:1=[1,3]",
        flush=True,
    )

    expanded_outputs = [_ke_np.full_like(x, _ke_np.nan) for x in outputs]
    expanded_coherent_outputs = [_ke_np.full_like(x, _ke_np.nan) for x in outputs]
    balanced_outputs = [_ke_np.full_like(x, _ke_np.nan) for x in outputs]
    balanced_coherent_outputs = [_ke_np.full_like(x, _ke_np.nan) for x in outputs]
    mass_applied = _ke_np.zeros((4,len(test_ids)),dtype=bool)
    mass_plan_hashes = _ke_np.full((4,len(test_ids)), '', dtype='<U64')
    extra_counts = _ke_np.zeros((4,len(test_ids)),dtype=_ke_np.int16)
    extra_micro = _ke_np.zeros((4,len(test_ids)),dtype=_ke_np.int8)
    coherent_outputs = [_ke_np.full_like(x, _ke_np.nan) for x in outputs]
    coherent_applied = _ke_np.zeros((4,len(test_ids)),dtype=bool)
    coherent_counts = _ke_np.zeros((4,len(test_ids),2),dtype=_ke_np.int16)
    coherent_micro = _ke_np.zeros((4,len(test_ids)),dtype=_ke_np.int8)
    infer_audit = _KePath('/kaggle/working/diagnostics/raptor_series_head.jsonl')
    infer_audit.unlink(missing_ok=True)

    def predict_pair(arm_index, study_index, study_uid, model, windows, mask, device, gap_windows=None, mass_plan=None):
        ref, alternate, extended, extended_coherent, balanced, balanced_coherent, report = v5_raptor_extended(
            model,windows,device,mask,gap_windows,MASTER,mass_plan,event=rsna_event)
        balanced_outputs[arm_index][study_index] = balanced
        balanced_coherent_outputs[arm_index][study_index] = balanced_coherent
        mass_applied[arm_index,study_index] = report['sampling_mass_applied']
        mass_plan_hashes[arm_index,study_index] = report['sampling_mass_plan_sha256']
        expanded_outputs[arm_index][study_index] = extended
        expanded_coherent_outputs[arm_index][study_index] = extended_coherent
        extra_counts[arm_index,study_index] = report['extra_windows']
        extra_micro[arm_index,study_index] = report['extra_micro_images']
        outputs[arm_index][study_index] = ref
        coherent_outputs[arm_index][study_index] = alternate
        coherent_applied[arm_index,study_index] = report['applied']
        coherent_counts[arm_index,study_index] = [report['windows'],report['same_series_windows']]
        coherent_micro[arm_index,study_index] = report['micro_images']
        import json
        with _ke_input_lock:
            with infer_audit.open('a') as handle:
                handle.write(json.dumps({'arm':arms[arm_index]['name'],
                    'study_uid':str(study_uid),**report},sort_keys=True,allow_nan=False)+'\n')

    def run_single(arm_index, device):
        arm = arms[arm_index]
        reader = _KE_NS['_make_reader']()
        weight_path = _KE_NS['find_weight_file'](arm['file'])
        model, resolution = _KE_NS['load_model'](
            weight_path, arm['arch'], arm['res'], device
        )
        if int(resolution) != int(arm['res']):
            raise RuntimeError(
                f"{arm['name']} checkpoint resolution {resolution} != {arm['res']}"
            )
        for study_index, study_uid, future in _ke_prefetched_windows(
            arm, test_ids, series, series_root, reader
        ):
            if _v3_stop.is_set():
                raise RuntimeError('Another required Raptor worker failed; stopping the model graph')
            rsna_deadline('Raptor full-cohort inference')
            try:
                pack = future.result()
                predict_pair(arm_index,study_index,study_uid,model,
                             pack['windows'],pack['same_series'],device,pack['gap_windows'],pack['mass_plan'])
                del pack
            except Exception as error:
                rsna_event('raptor_study_failed', arm=str(arm['name']), study=str(study_uid), error=f'{type(error).__name__}: {str(error)[:500]}')
                _v3_stop.set()
                raise RuntimeError(f"Required Raptor view {arm['name']} failed on {study_uid}") from error
        del model
        _ke_gc.collect()
        with torch.cuda.device(device):
            torch.cuda.empty_cache()

    def run_shared_maxspan(device):
        first, reverse = arms[0], arms[2]
        comparable = ('file', 'arch', 'res', 'img', 'slots', 'span', 'k_eval')
        if any(first[key] != reverse[key] for key in comparable):
            raise RuntimeError('MaxSpan forward/reverse arms no longer share preprocessing')
        reader = _KE_NS['_make_reader']()
        weight_path = _KE_NS['find_weight_file'](first['file'])
        model, resolution = _KE_NS['load_model'](
            weight_path, first['arch'], first['res'], device
        )
        if int(resolution) != int(first['res']):
            raise RuntimeError(
                f"MaxSpan checkpoint resolution {resolution} != {first['res']}"
            )
        for study_index, study_uid, future in _ke_prefetched_windows(
            first, test_ids, series, series_root, reader
        ):
            if _v3_stop.is_set():
                raise RuntimeError('Another required Raptor worker failed; stopping the model graph')
            rsna_deadline('Raptor full-cohort inference')
            try:
                pack = future.result()
                predict_pair(0,study_index,study_uid,model,
                             pack['windows'],pack['same_series'],device,pack['gap_windows'],pack['mass_plan'])
                predict_pair(2,study_index,study_uid,model,
                             pack['windows'].flip(1).contiguous(),pack['same_series'],device,
                             mass_plan=v5_reverse_plan(pack['mass_plan']))
                del pack
            except Exception as error:
                rsna_event('raptor_study_failed', arm=str(first['name']), study=str(study_uid), error=f'{type(error).__name__}: {str(error)[:500]}')
                _v3_stop.set()
                raise RuntimeError(f"Required shared MaxSpan view failed on {study_uid}") from error
        del model
        _ke_gc.collect()
        with torch.cuda.device(device):
            torch.cuda.empty_cache()

    if torch.cuda.device_count() >= 2:
        def gpu_zero():
            with torch.cuda.device(0):
                run_shared_maxspan(torch.device('cuda:0'))

        def gpu_one():
            with torch.cuda.device(1):
                run_single(1, torch.device('cuda:1'))
                run_single(3, torch.device('cuda:1'))

        with _KeThreadPool(max_workers=2) as executor:
            def guarded_owner(fn):
                try:
                    return fn()
                except BaseException:
                    _v3_stop.set()
                    raise
            workers = [executor.submit(guarded_owner,gpu_zero), executor.submit(guarded_owner,gpu_one)]
            for w in workers:
                w.result()
    else:
        with torch.cuda.device(0):
            run_shared_maxspan(torch.device('cuda:0'))
            run_single(1, torch.device('cuda:0'))
            run_single(3, torch.device('cuda:0'))

    _ke_np.savez_compressed('/kaggle/working/raptor_raw.npz',
        study_uids=_ke_np.asarray(test_ids), raw_probabilities=_ke_np.stack(outputs))
    rsna_finite(_ke_np.stack(outputs),'all four Raptor views',probability=True)
    rsna_finite(_ke_np.stack(coherent_outputs),'all four coherent Raptor views',probability=True)
    v3_save_bank('/kaggle/working/diagnostics',test_ids,_ke_np.stack(outputs),
                 _ke_np.stack(coherent_outputs),coherent_applied,coherent_counts,coherent_micro)
    _v4_parent_bank = v3_load_bank('/kaggle/working/diagnostics',test_ids,
                                 '/kaggle/working/raptor_raw.npz',MASTER)
    v4_validate_source_audit(_ke_input_audit_path,test_ids,extra_counts,coherent_counts,MASTER)
    v4_save_bank('/kaggle/working/diagnostics',test_ids,_v4_parent_bank,
                 _ke_np.stack(expanded_outputs),_ke_np.stack(expanded_coherent_outputs),
                 extra_counts,extra_micro,MASTER,_ke_input_audit_path)
    _v5_gap_bank = v4_load_bank('/kaggle/working/diagnostics',test_ids,
                               _v4_parent_bank,MASTER,_ke_input_audit_path)
    v5_save_bank('/kaggle/working/diagnostics',test_ids,_v4_parent_bank,_v5_gap_bank,
                 _ke_np.stack(balanced_outputs),_ke_np.stack(balanced_coherent_outputs),
                 mass_applied,mass_plan_hashes,MASTER,_ke_input_audit_path)
    weights = _ke_np.asarray([float(arm['w']) for arm in arms], _ke_np.float64)
    weights /= weights.sum()
    probability_blend = _ke_np.tensordot(
        weights,
        _ke_np.stack([_ke_np.clip(value, 0, 1) for value in outputs]),
        axes=(0, 0),
    )
    ranks = _KE_NS['rankpct'](probability_blend)
    rsna_finite(ranks,'Raptor ranked ensemble',probability=True)
    submission = _ke_pd.DataFrame(ranks.astype(_ke_np.float32), columns=_KE_NS['LAB'])
    submission.insert(0, 'StudyInstanceUID', test_ids)
    submission = submission[columns]
    if submission['StudyInstanceUID'].tolist() != test_ids:
        raise RuntimeError('Raptor study order drift')
    if not _ke_np.isfinite(submission[_KE_NS['LAB']].to_numpy()).all():
        raise RuntimeError('Raptor produced non-finite predictions')
    output = _KePath('/kaggle/working/_raptor.csv')
    submission.to_csv(output, index=False)
    print(
        f"[raptor-fast] wrote {output}; elapsed {_ke_time.time() - started:.1f}s",
        flush=True,
    )


_ke_input_lock = _ke_threading.Lock()
_ke_input_ids = set()
_ke_pixel_cache_bytes = 0
_ke_input_audit_path = _KePath('/kaggle/working/diagnostics/raptor_inputs.jsonl')
_ke_input_audit_path.unlink(missing_ok=True)
_KE_NS['infer_probs'] = _ke_infer_input
for _arm in _KE_NS['ARMS']:
    _arm['parent_k_eval'] = _arm['k_eval']
    _arm['k_eval'] = 94
rsna_phase('public_raptor', 'START')
_ke_run_raptor_arms()
rsna_phase('public_raptor', 'COMPLETE')
_ke_pixel_cache.clear()
_ke_order_cache.clear()
_ke_gc.collect()
_KePath('/kaggle/working/raptor_input_before_coat.csv').write_bytes(_KePath('/kaggle/working/_raptor.csv').read_bytes())
import json as _asset_json
_KePath('/kaggle/working/raptor_input_summary.json').write_text(
    _asset_json.dumps({'studies':len(_RSNA_TEST_IDS), 'preparations':len(_ke_input_ids),
                    'view_names':[a['name'] for a in _KE_NS['ARMS']],
                    'shared_preparation_views':['maxspan-v5','maxspan-v5-reverse'],
                    'records':str(_ke_input_audit_path)},indent=2))



def _coat_substitute():
    import hashlib as _h, os as _o, subprocess as _sp, sys as _sy
    from pathlib import Path as _P
    import pandas as _pd

    import numpy as _np
    _P('/kaggle/working/_raptor_public.csv').write_bytes(
        _P('/kaggle/working/_raptor.csv').read_bytes())
    _np.savez_compressed('/kaggle/working/btk_v32_rest_input.npz',
        study_uids=_ke_ours.StudyInstanceUID.to_numpy().astype(str),
        labels=_np.asarray(_KE_LAB), values=_ke_ours[_KE_LAB].to_numpy())
    MAN_SHA = '98511a8fdeb9da0e6e70c78d013ff636e1476f31c80b5dc134d294b18c3f284e'
    WHL_SHA = '236c8df54a90f4d02076e6f9c1cc763d794542e886c576a6fee46ec8ff75a7a9'
    raptor = _P('/kaggle/working/_raptor.csv')

    def sha(p):
        d = _h.sha256()
        with _P(p).open('rb') as f:
            for b in iter(lambda: f.read(8 << 20), b''):
                d.update(b)
        return d.hexdigest()

    def find(name, want):
        return _asset_find_asset(name, want)

    man = find('coat_resgated_ep10_top3_manifest.json', MAN_SHA)
    if man is None:
        raise RuntimeError('coat manifest absent or hash mismatch')

    art = man.parent

    whl = find('opencv_python_headless-4.12.0.88-*.whl', WHL_SHA)
    envd = _P('/kaggle/working/_coat_env')
    _sp.run(
        [
            _sy.executable,
            '-m',
            'pip',
            'install',
            '--no-deps',
            '--quiet',
            '--target',
            str(envd),
            str(whl),
        ],
        check=True,
    )

    out = _P('/kaggle/working/_coat_arm.csv')

    import inspect
    runtime_path = _P('/kaggle/working/input_resgated_runtime.py')
    helper_source = 'import numpy as np\n' + '\n'.join(inspect.getsource(f) for f in
        (_dense_allocate, _dense_unique_linspace, _dense_coat_specs))
    runtime_sha = _asset_write_coat_runtime(art,runtime_path,helper_source)

    child = (
        "import sys, json, os\n"
        f"sys.path.insert(0, {str(envd)!r})\n"
        f"sys.path.insert(0, {str(art)!r})\n"
        "import cv2; assert cv2.__version__ == '4.12.0', cv2.__version__\n"
        "import torch; assert torch.cuda.device_count() >= 1\n"
        "sys.path.insert(0, '/kaggle/working')\n"
        "import input_resgated_runtime as rt\n"
        "assert rt.base.cv2.__version__ == '4.12.0'\n"
        "from pathlib import Path\n"
        "r = rt.run_submission("
        "competition_root=Path(os.environ['RSNA_COMP_ROOT']) "
        "if os.environ.get('RSNA_COMP_ROOT') "
        "else rt.base.find_competition_root(),\n"
        f"    artifact_root=Path({str(art)!r}), "
        f"output_path=Path({str(out)!r}),\n"
        "    gpu_batch_studies=2, backbone_micro_images=8)\n"
        "assert r['status'] == rt.SUBMISSION_STATUS\n"
        "assert r['models'] == 3\n"
        "assert [i['epoch'] for i in r['checkpoints']] == [4, 6, 8]\n"
        "print('[coat-child] fallback_studies', r['fallback_studies'], flush=True)\n"
        "Path('/kaggle/working/_coat_arm_receipt.json').write_text("
        "json.dumps(r, indent=2))\n"
    )

    env = dict(_o.environ)
    env['CUDNN_CONV_WSCAP_DBG'] = '1024'
    env['RSNA_COMP_ROOT'] = _KE_NS['find_test_root']()
    env['PYTHONPATH'] = f"{envd}:{art}:" + env.get('PYTHONPATH', '')

    import json as _j
    d4_out = _P('/kaggle/working/d4_input')/'d4.csv'

    def _run_residual_reader():
        _run_required_child([_sy.executable, '-c', child], env,
                            _P('/kaggle/working/resgated_runtime.log'))
        return _j.loads(_P('/kaggle/working/_coat_arm_receipt.json').read_text())

    def _run_d4_reader():
        d4_manifest = _asset_find_asset('coatnet_pairfilm_manifest.json',
            '7ada0605bca6b0530569c6454e988ace479606a3328ed591d090e5764fea661d')
        timm_whl = d4_manifest.parent/'timm-1.0.22-py3-none-any.whl'
        if sha(timm_whl) != '888981753e65cbaacfc07494370138b1700a27b1f0af587f4f9b47bc024161d0':
            raise RuntimeError('D4 timm wheel changed')
        d4_env = _P('/kaggle/working/_d4_env')
        _sp.run([_sy.executable,'-m','pip','install','--no-deps','--quiet',
            '--target',str(d4_env),str(timm_whl)],check=True)
        d4_dir = _P('/kaggle/working/d4_input')
        d4_dir.mkdir(exist_ok=True)
        return _asset_run_d4(
            d4_manifest.parent, d4_env, _KE_NS['find_test_root'](), d4_out
        )

    _parallel_coat_readers = _o.environ.get(
        'RSNA_PARALLEL_COAT_READERS', '0'
    ) == '1'
    _coat_receipt = None
    _d4_receipt = None
    _coat_exc = None
    _d4_exc = None
    if _parallel_coat_readers:
        rsna_phase('coat_readers', 'START', parallel=True)
        with _KeThreadPool(max_workers=2) as _coat_pool:
            _coat_future = _coat_pool.submit(_run_residual_reader)
            _d4_future = _coat_pool.submit(_run_d4_reader)
            try:
                _coat_receipt = _coat_future.result()
            except Exception as _error:
                _coat_exc = _error
            try:
                _d4_receipt = _d4_future.result()
            except Exception as _error:
                _d4_exc = _error
        rsna_phase(
            'coat_readers', 'COMPLETE', parallel=True,
            residual_ok=_coat_exc is None, d4_ok=_d4_exc is None,
        )
    else:
        rsna_phase('residual_coat', 'START')
        try:
            _coat_receipt = _run_residual_reader()
        except Exception as _error:
            _coat_exc = _error
        rsna_phase('residual_coat', 'COMPLETE', ok=_coat_exc is None)
        if _coat_exc is not None:
            raise RuntimeError('Required residual CoAt reader failed; stopping before D4') from _coat_exc
        rsna_phase('d4', 'START')
        try:
            _d4_receipt = _run_d4_reader()
        except Exception as _error:
            _d4_exc = _error
        rsna_phase('d4', 'COMPLETE', ok=_d4_exc is None)
        if _d4_exc is not None:
            raise RuntimeError('Required D4 reader failed; stopping before final fusion') from _d4_exc

    _coat_resgated_ok = _coat_exc is None
    if _coat_resgated_ok:
        if int(_coat_receipt.get('fallback_studies', 0) or 0):
            rsna_event('coat_resgated_fallback_studies', count=int(_coat_receipt['fallback_studies']), failures=list(_coat_receipt.get('failures', []))[:20])
    else:
        rsna_event('coat_resgated_child_failed', error=f'{type(_coat_exc).__name__}: {str(_coat_exc)[:1500]}')
        print(f'[coat-arm] residual CoAt child FAILED; continuing without it (flagged): {type(_coat_exc).__name__}', flush=True)

    _coat_d4_ok = _d4_exc is None
    if not _coat_d4_ok:
        rsna_event('coat_d4_child_failed', error=f'{type(_d4_exc).__name__}: {str(_d4_exc)[:1500]}')
        print(f'[coat-arm] D4 CoAt child FAILED; continuing without it (flagged): {type(_d4_exc).__name__}', flush=True)

    pub = _pd.read_csv(raptor, dtype={'StudyInstanceUID': str})
    pub = rsna_frame(pub,_RSNA_TEST_IDS,_KE_LAB,'public Raptor')
    lab = [c for c in pub.columns if c != 'StudyInstanceUID']
    import numpy as _np
    # Retain BTKD inner blend coefficients; input/new family scores are unvalidated.
    private_alpha = 0.4
    public_rank = pub[lab].rank(method='average', pct=True)
    _members = []
    if _coat_resgated_ok:
        try:
            ours = _pd.read_csv(out, dtype={'StudyInstanceUID': str})
            ours = rsna_frame(ours,_RSNA_TEST_IDS,_KE_LAB,'resgated CoAt')
            if list(ours.columns) != list(pub.columns):
                raise RuntimeError('coat arm column drift')
            ours = ours.set_index('StudyInstanceUID').reindex(pub.StudyInstanceUID.astype(str).tolist()).reset_index()
            if ours[lab].isna().any().any():
                raise RuntimeError('coat arm does not cover every study')
            rsna_save_predictions('coat_resgated_rank',_RSNA_TEST_IDS,ours[lab].to_numpy(),lab)
            _members.append(('resgated_top3', ours[lab].rank(method='average', pct=True)))
        except Exception as _exc:
            rsna_event('coat_resgated_output_rejected', error=f'{type(_exc).__name__}: {str(_exc)[:500]}')
    if _coat_d4_ok:
        try:
            d4 = _pd.read_csv(d4_out,dtype={'StudyInstanceUID':str})
            d4 = rsna_frame(d4,_RSNA_TEST_IDS,_KE_LAB,'D4 CoAt')
            rsna_save_predictions('coat_d4_rank',_RSNA_TEST_IDS,d4[lab].to_numpy(),lab)
            if d4.StudyInstanceUID.duplicated().any() or list(d4.columns)!=list(pub.columns):
                raise RuntimeError('D4 schema/UID drift')
            d4 = d4.set_index('StudyInstanceUID').reindex(pub.StudyInstanceUID.astype(str))
            if d4[lab].isna().any().any():
                raise RuntimeError('D4 missing predictions')
            _members.append(('d4_swa3', d4[lab].reset_index(drop=True).rank(method='average',pct=True)))
        except Exception as _exc:
            rsna_event('coat_d4_output_rejected', error=f'{type(_exc).__name__}: {str(_exc)[:500]}')
    if not _members:
        rsna_event('coat_family_unavailable', note='public Raptor kept as the CoAt/Raptor input (flagged)')
        print('[coat-arm] no CoAt family member available; public Raptor retained (flagged)', flush=True)
        return 0
    if len(_members) == 2:
        private_rank = _asset_half_rank_mix(_members[0][1], _members[1][1])
    else:
        rsna_event('coat_family_partial', member=_members[0][0])
        private_rank = _members[0][1]
    family = pub.copy()
    family[lab] = private_rank
    family.to_csv('/kaggle/working/_coat_family_rank.csv', index=False)
    hybrid = pub.copy()
    hybrid[lab] = (1.0 - private_alpha) * public_rank + private_alpha * private_rank
    if not _np.isfinite(hybrid[lab].to_numpy(_np.float64)).all():
        raise RuntimeError('CoAt/Raptor hybrid contains non-finite values')
    tmp = raptor.with_name('.raptor_coat_hybrid.csv')
    hybrid.to_csv(tmp, index=False)
    _o.replace(tmp, raptor)
    raptor.with_name('_coat_raptor_blend_receipt.json').write_text(_j.dumps({
        'contract': 'public_raptor_private_residual_coat_v32_no_inner_rerank_v1', 'inner_rerank': False, 'private_alpha': private_alpha,
        'within_coat': {m: 0.5 for m, _ in _members} if len(_members) == 2 else {_members[0][0]: 1.0}, 'public_raptor_alpha': 1.0 - private_alpha,
        'study_count': len(pub), 'finding_specific_weights': False, 'members': [m for m, _ in _members]}, indent=2, sort_keys=True) + '\n')
    return len(pub)

_coat_n = _coat_substitute()
print(f'[coat-arm] residual top3 / D4 SWA3 equal rank family; BTKD32 no-inner-rerank; {_coat_n} studies',flush=True)

_ke_theirs = _ke_pd.read_csv('/kaggle/working/_raptor.csv',
                             dtype={'StudyInstanceUID': str})
assert list(_ke_theirs.columns) == list(_ke_ours.columns), 'column drift'
_ke_theirs = _ke_theirs.set_index('StudyInstanceUID').reindex(
    _ke_ours['StudyInstanceUID']).reset_index()
assert _ke_theirs[_KE_LAB].notna().all().all(), 'study identity drift'


_ke_tr = _ke_ours[_KE_LAB].rank(method='average', pct=True)
_ke_cr = _ke_theirs[_KE_LAB].copy()
_blend_transformer = _ke_ours.copy()
_blend_coatnet = _ke_theirs.copy()
_blend_labels = list(_KE_LAB)
_blend_tr = _ke_tr.copy()
_blend_cr = _ke_cr.copy()
_coatnet_weight = {label: 0.60 for label in _blend_labels}
_coatnet_weight.update({
    'ACL': 0.75,
    'Medial Meniscus': 0.80,
    'Lateral Meniscus': 1.00,
    'Lateral OA': 0.75,
    'Fracture': 0.75,
})
_blend_output = _blend_transformer.copy()
for _blend_label in _blend_labels:
    _blend_w = float(_coatnet_weight[_blend_label])
    _blend_output[_blend_label] = (
        (1.0 - _blend_w) * _blend_tr[_blend_label]
        + _blend_w * _blend_cr[_blend_label]
    )
_blend_output[_blend_labels] = _blend_output[_blend_labels].rank(
    method='average', pct=True
)
assert _ke_np.isfinite(
    _blend_output[_blend_labels].to_numpy(_ke_np.float64)
).all()
_blend_output.to_csv(_ke_primary, index=False)


## 9. Verify the complete source graph on the current test cohort

In [ ]:
# Exact anchor replay is a mandatory gate before promoting new acquisition evidence.
rsna_phase('mast_fusion', 'START')
_ma_ids = list(_RSNA_TEST_IDS)
_ma_work = Path('/kaggle/working')
_ma_anchor_frame = ma_frame(pd.read_csv(_ma_work/'_pipeline_stage.csv',
                                      dtype={'StudyInstanceUID':str}), _ma_ids)
if _DINOV2_MATCHED_MEMBERS != 20 or not globals().get('V18_CALIBRATOR_APPLIED'):
    raise RuntimeError('The complete parent graph did not finish')
_ma_receipt_path = _ma_work/'_coat_raptor_blend_receipt.json'
if not _ma_receipt_path.is_file(): raise RuntimeError('Missing complete CoAt-family receipt')
_ma_coat_receipt = json.loads(_ma_receipt_path.read_text())
if set(_ma_coat_receipt.get('members', [])) != {'resgated_top3','d4_swa3'}:
    raise RuntimeError('A required CoAt reader failed; no reduced ensemble will be published')
_ma_required_preps = {(name,uid) for name in ('maxspan-v5','native384dense-v10','native384-v8') for uid in _ma_ids}
if _ke_input_ids != _ma_required_preps:
    raise RuntimeError('Raptor input-preparation coverage was not complete')
_ma_fatal = [e for e in _RSNA_AUDIT['events'] if e.get('kind') in MASTER_FATAL_EVENTS]
if _ma_fatal:
    rsna_json(_ma_work/'diagnostics/rejected_events.json', _ma_fatal)
    raise RuntimeError(f'{len(_ma_fatal)} required-model failure(s); refusing a repaired/partial submission')

def _ma_read_bank(tag):
    with np.load(_ma_work/'diagnostics'/(tag+'.npz'), allow_pickle=False) as data:
        if data['study_uids'].astype(str).tolist() != _ma_ids:
            raise RuntimeError(tag + ': prediction UID order mismatch')
        if data['labels'].astype(str).tolist() != MA_LABELS:
            raise RuntimeError(tag + ': prediction labels mismatch')
        return data['values'].copy()

_ma_e10 = _ma_read_bank('rad_e10_raw')
_ma_e13 = _ma_read_bank('rad_e13_raw')
_ma_second = _ma_read_bank('rad_legacy_second_raw')
_ma_cal = json.loads(_rad_zlib.decompress(_rad_b64.b64decode(_RAD_CAL_PAYLOAD)).decode())
_ma_meta = pd.read_csv(ROOT/'test_series.csv', dtype={'StudyInstanceUID':str,'SeriesInstanceUID':str})
_ma_proto = ma_protocol(_ma_meta, _ma_ids)
_ma_hybrid = ma_frame(pd.read_csv(_ma_work/'_raptor.csv', dtype={'StudyInstanceUID':str}), _ma_ids)[MA_LABELS].to_numpy()

def _ma_dino_values(members):
    order, raw = _combine(members)
    frame = pd.DataFrame(ma_rank(raw), columns=MA_LABELS)
    frame.insert(0, 'StudyInstanceUID', list(map(str, order)))
    return ma_csv_values(ma_frame(frame, _ma_ids)[MA_LABELS].to_numpy())

def _ma_replay_complete(dino_values):
    a5 = _MA_A5_BLEND_FN(dino_values, _MA_A5_VALUES, _MA_A5_MASK, A5_W)
    a5 = ma_csv_values(a5)
    rad = ma_replay_rad(a5, _ma_e10, _ma_e13, _ma_second, _ma_proto, _ma_cal,
        primary_alpha=_RAD_ALPHA, e13_weight=_RAD_E13_MEMBER_WEIGHT,
        second_alpha=_RAD_V48_SECOND_ALPHA, calibration_alpha=.40)
    rad = ma_csv_values(rad)
    final = ma_outer_fuse(rad, _ma_hybrid)
    return a5, rad, ma_csv_values(final)

_ma_replay_dino = _ma_dino_values(_MA_ANCHOR_MEMBERS)
_ma_replay_a5, _ma_replay_rad_values, _ma_replay_final = _ma_replay_complete(_ma_replay_dino)
_ma_gates = {}
for _name, _got, _want in [
    ('dino', _ma_replay_dino, _MA_DINO_STAGE),
    ('a5', _ma_replay_a5, _MA_A5_STAGE),
    ('rad', _ma_replay_rad_values, _MA_RAD_STAGE),
    ('final', _ma_replay_final, _ma_anchor_frame[MA_LABELS].to_numpy()),
]:
    _ma_gates[_name] = dict(exact=bool(np.array_equal(_got, _want)),
                           max_abs_difference=float(np.max(np.abs(_got-_want))))
rsna_json(_ma_work/'diagnostics/mast_replay_gates.json', _ma_gates)
if not all(v['exact'] for v in _ma_gates.values()):
    raise RuntimeError('Anchor replay mismatch: inspect diagnostics/mast_replay_gates.json; '
                       'the candidate is NOT permitted to bypass a fusion mismatch')

# Save the full same-run anchor before any challenger transformations.
_ma_anchor_file = _ma_work/'submission_anchor.csv'
_ma_anchor_receipt = ma_publish(_ma_anchor_frame, _ma_ids, _ma_anchor_file, {
    'status':'COMPLETE_ANCHOR', 'source':'Speedy Raptors / fast two-T4 execution donor',
    'inherited_score':'user-reported 0.942; not independently reproduced here',
    'new_run_score':None, 'replay_gates':_ma_gates})

_ma_candidate_members = []
_ma_changed_studies = set()
for _m in _MA_ANCHOR_MEMBERS:
    _copy = {**_m, 'pred':_m['pred'].copy()}
    _record = _MA_ALTERNATES.get(str(_m['id']))
    if _record is not None:
        if _record['ids'] != list(map(str,_m['ids'])):
            raise RuntimeError('Alternate prediction UID order drift')
        _hash = hashlib.sha256(np.ascontiguousarray(_m['pred']).tobytes()).hexdigest()
        if _hash != _record['anchor_sha256']:
            raise RuntimeError('The anchor used for an extra view has changed')
        _copy['pred'] = ma_mix_raw(_m['pred'], _record['alternate'], _record['indices'],
                                   MASTER['acquisition_alpha'])
        _ma_changed_studies.update(_record['ids'][int(i)] for i in _record['indices'])
    _ma_candidate_members.append(_copy)
_ma_candidate_dino = _ma_dino_values(_ma_candidate_members)
_, _v2_candidate_rad, _ma_candidate_final = _ma_replay_complete(_ma_candidate_dino)

# Save the no-Global96 candidate for diagnosis, without making it the selected output.
_v2_before_g96 = _ma_anchor_frame.copy()
_v2_before_g96[MA_LABELS] = _ma_candidate_final
ma_publish(_v2_before_g96,_ma_ids,_ma_work/'diagnostics/submission_before_global96.csv',
           {'status':'COMPLETE_REPAIR_VIEW_CANDIDATE','score_improvement_verified':False})
# Use full-precision reads for the rank-valued FAMILY written before its CSV round trip.
_v2_public_frame = ma_frame(pd.read_csv(_ma_work/'_raptor_public.csv',dtype={'StudyInstanceUID':str}),_ma_ids)
_v2_family_frame = ma_frame(pd.read_csv(_ma_work/'_coat_family_rank.csv',dtype={'StudyInstanceUID':str},
                                      float_precision='round_trip'),_ma_ids)
_v2_public_values = _v2_public_frame[MA_LABELS].to_numpy()
_v2_family_values = _v2_family_frame[MA_LABELS].to_numpy()
_v2_replayed_hybrid = ma_csv_values(v2_coat_hybrid(_v2_public_values,_v2_family_values,share=0.))
if not np.array_equal(_v2_replayed_hybrid,_ma_hybrid):
    raise RuntimeError('CoAt-family anchor reconstruction mismatch; refusing the Global96 experiment')
rsna_json(_ma_work/'diagnostics/v2_family_gate.json',{'exact':True,'global96_share_zero_replays_parent':True})
print('Original graph replayed exactly. Immutable submission_anchor.csv saved.')

_v3_bank = v3_load_bank(_ma_work/'diagnostics', _ma_ids,
                        _ma_work/'raptor_raw.npz', MASTER)
_v3_replayed_public = v3_public_scores(_v3_bank['anchor'])
if not np.array_equal(_v3_replayed_public,_v2_public_values):
    raise RuntimeError('Raw Raptor reconstruction differs from its original CSV stage')
_v3_raw_candidate = v3_blend_raw(_v3_bank['anchor'],_v3_bank['coherent'],
                                _v3_bank['applied'],MASTER['series_coherence_alpha'])
_v3_public_values = v3_public_scores(_v3_raw_candidate)
rsna_save_predictions('v3_public_raptor',_ma_ids,_v3_public_values,MA_LABELS)
rsna_json(_ma_work/'diagnostics/v3_public_replay_gate.json', {
    'original_raw_bank_exact':True,'uid_order_exact':True,
    'paired_bank_sha256':_v3_bank['receipt']['sha256'],
    'applied_view_studies':int(_v3_bank['applied'].sum()),
    'series_coherence_alpha':MASTER['series_coherence_alpha']})
print('Paired Raptor bank validated; original raw-to-CSV replay is exact.')

_v4_bank = v4_load_bank(_ma_work/'diagnostics',_ma_ids,_v3_bank,MASTER,_ke_input_audit_path)
v4_validate_source_audit(_ke_input_audit_path,_ma_ids,_v4_bank['counts'],_v3_bank['counts'],MASTER)
rsna_json(_ma_work/'diagnostics/v4_coverage_gate.json',{
    'source_bank_link_exact':True,'extra_windows':int(_v4_bank['counts'].sum()),
    'gap_bank_sha256':_v4_bank['receipt']['sha256'],'score_improvement_verified':False})
print('Gap coverage bank linked to original sources/settings; all budgets validated.')

_v5_bank = v5_load_bank(_ma_work/'diagnostics', _ma_ids, _v3_bank, _v4_bank,
                        MASTER, _ke_input_audit_path)
rsna_json(_ma_work/'diagnostics/v5_sampling_mass_gate.json', {
    'source_plan_hashes_exact':True, 'applied_view_studies':int(_v5_bank['applied'].sum()),
    'raw_bank_sha256':_v5_bank['receipt']['sha256'], 'additional_encoder_images_vs_v4':0,
    'score_improvement_verified':False})
print('Sampling-mass bank linked to executed window groups; no extra encoder images.')


## 10. Three-checkpoint Global96 reader

In [ ]:
def v2_make_global96_child(root, timm_env, cv_env, competition, output, receipt_path):
    """Original Global96 loader/94-window contract, with strict completion checks."""
    args = {k:str(v) for k,v in dict(root=root,timm_env=timm_env,cv_env=cv_env,
           competition=competition,output=output,receipt_path=receipt_path).items()}
    return '''import os, sys, json, hashlib
from pathlib import Path
ARGS = ''' + repr(args) + '''
for directory in (ARGS['root'], ARGS['cv_env'], ARGS['timm_env']):
    sys.path.insert(0, directory)
import cv2, timm, torch
if cv2.__version__ != '4.12.0' or timm.__version__ != '1.0.22':
    raise RuntimeError('Global96 dependency versions changed')
if torch.cuda.device_count() != 2 or not all('T4' in torch.cuda.get_device_name(i).upper() for i in range(2)):
    raise RuntimeError('Global96 requires actual T4 x2, without device-name spoofing')
import coatnet_global96_baseline_top3_inference as rt
if rt.base.cv2.__version__ != '4.12.0' or rt.MAX_EVAL_WINDOWS != 94:
    raise RuntimeError('Global96 reader contract changed')
# The supplied reader owns checkpoint loading and multi-GPU worker execution.
result = rt.run_submission(
    competition_root=Path(ARGS['competition']), artifact_root=Path(ARGS['root']),
    output_path=Path(ARGS['output']), gpu_batch_studies=2, backbone_micro_images=8)
if result['status'] != rt.SUBMISSION_STATUS or result['models'] != 3:
    raise RuntimeError('Global96 did not finish its complete three-model graph')
if [item['epoch'] for item in result['checkpoints']] != [16,23,18]:
    raise RuntimeError('Global96 checkpoint identities changed')
if int(result['fallback_studies']) != 0:
    raise RuntimeError('Global96 contains failed studies; no partial ensemble allowed')
result['v2_wrapper'] = {
    'expected_status':rt.SUBMISSION_STATUS, 'max_eval_windows':94,
    'resolution_policy':'original Global96 reader, not modified',
    'gpu_batch_studies':2, 'backbone_micro_images':8,
    'manifest_sha256':hashlib.sha256((Path(ARGS['root'])/'coatnet_global96_top3_manifest.json').read_bytes()).hexdigest(),
    'output_sha256':hashlib.sha256(Path(ARGS['output']).read_bytes()).hexdigest()}
path = Path(ARGS['receipt_path'])
temporary = path.with_name('.'+path.name+'.tmp')
with temporary.open('w') as handle:
    json.dump(result,handle,indent=2,sort_keys=True,allow_nan=False)
    handle.write('\\n'); handle.flush(); os.fsync(handle.fileno())
os.replace(temporary,path)
print('Global96 complete:',result['v2_wrapper'],flush=True)
'''


def v2_run_global96():
    """Run AFTER the original ensemble has completed and its anchor is saved.

    The default v2 recipe REQUIRES this branch. Missing assets fail early;
    a timeout or reader error fails explicitly rather than changing the recipe.
    """
    if not MASTER['global96_enabled']:
        return None, {'status':'DISABLED_EXPLICITLY'}
    if _V2_GLOBAL96_ASSET is None:
        raise RuntimeError('Missing Global96 preflight receipt')
    asset = Path(_V2_GLOBAL96_ASSET['root'])
    if ma_sha(asset/V2_GLOBAL96_MANIFEST) != V2_GLOBAL96_MANIFEST_SHA:
        raise RuntimeError('Global96 artifact changed since preflight')
    # Reuse the already isolated, pinned timm installation from D4.
    timm_env = Path('/kaggle/working/_d4_env')
    cv_env = Path('/kaggle/working/_coat_env')
    if not (timm_env/'timm').is_dir() or not cv_env.is_dir():
        raise RuntimeError('D4/CoAt did not create the expected isolated dependency directories')
    folder = Path('/kaggle/working/global96'); folder.mkdir(exist_ok=True)
    output, receipt = folder/'reader.csv', folder/'reader.receipt.json'
    for path in (output, receipt): path.unlink(missing_ok=True)
    source = v2_make_global96_child(asset,timm_env,cv_env,ROOT,output,receipt)
    script = folder/'run_global96.py'
    compile(source,str(script),'exec'); script.write_text(source)
    environment = dict(os.environ)
    environment['RSNA_COMP_ROOT'] = str(ROOT)
    environment['CUDNN_CONV_WSCAP_DBG'] = '1024'
    environment['PYTHONPATH'] = f'{timm_env}:{cv_env}:{asset}:' + environment.get('PYTHONPATH','')
    environment.update(HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', PIP_NO_INDEX='1')
    gc.collect()
    for gpu in range(torch.cuda.device_count()):
        with torch.cuda.device(gpu): torch.cuda.empty_cache()
    remaining = v2_remaining_budget(MASTER_STARTED_AT,MASTER)
    rsna_phase('global96_v2','START',remaining_budget_seconds=remaining,complete_cohort=len(_RSNA_TEST_IDS))
    try:
        process = v2_run_checked_child([sys.executable,str(script)], environment,
                                      folder/'runtime.log', remaining)
        if not receipt.is_file(): raise RuntimeError('Global96 did not publish its completion receipt')
        report = json.loads(receipt.read_text())
        wrapper = report.get('v2_wrapper',{})
        if wrapper.get('manifest_sha256') != V2_GLOBAL96_MANIFEST_SHA:
            raise RuntimeError('Global96 child manifest receipt mismatch')
        if wrapper.get('output_sha256') != ma_sha(output):
            raise RuntimeError('Global96 child output changed')
        if not wrapper.get('expected_status'):
            raise RuntimeError('Missing Global96 child runtime-status contract')
        frame = v2_validate_global96_receipt(report,_RSNA_TEST_IDS,output,
                                           expected_status=wrapper['expected_status'])
        report['v2_process'] = process
        rsna_save_predictions('v2_global96_reader',_RSNA_TEST_IDS,frame[MA_LABELS].to_numpy(),MA_LABELS)
        rsna_json(folder/'validated_receipt.json',report)
    except BaseException as exc:
        rsna_phase('global96_v2','FAILED',error_type=type(exc).__name__,error=str(exc))
        # The immutable anchor remains available, but no default-recipe CSV is published.
        raise
    rsna_phase('global96_v2','COMPLETE',models=3,study_count=len(frame),fallback_studies=0)
    return frame,report


In [ ]:
_V2_GLOBAL96_FRAME, _V2_GLOBAL96_REPORT = v2_run_global96()


## 11. Probability-pooling candidate, reference export, and final publication

In [ ]:
# Only reached after ALL original readers, replay gates, and Global96 succeed.
rsna_phase('complete_v2_fusion','START')
if MASTER['acquisition_alpha'] or MASTER['series_coherence_alpha'] or MASTER['gap_view_alpha'] or MASTER['sampling_mass_enabled']:
    raise RuntimeError('This build isolates one fusion change; sampling experiments must stay disabled')
if not MASTER['global96_enabled'] or _V2_GLOBAL96_FRAME is None:
    raise RuntimeError('The configured full pretrained graph requires Global96')
_fatal=[e for e in _RSNA_AUDIT['events'] if e.get('kind') in MASTER_FATAL_EVENTS]
if _fatal:raise RuntimeError('A required model/reader failed: '+str(_fatal[:5]))
if ma_sha(_ma_work/'submission_anchor.csv')!=_ma_anchor_receipt['submission_sha256']:
    raise RuntimeError('Source anchor changed after its replay gate')
_residual,_res_evidence=s2.load_probability_mean(
    _ma_work/'coat_resgated_ep10_top3_predictions.npz',_ma_work/'_coat_arm.csv',_ma_ids,3)
_d4,_d4_evidence=s2.load_probability_mean(
    _ma_work/'d4_input/coatnet_d4_depthzone_swa3_predictions.npz',_ma_work/'d4_input/d4.csv',_ma_ids,1)
_g96=ma_frame(_V2_GLOBAL96_FRAME,_ma_ids)[MA_LABELS].to_numpy()
_family_probability=s2.pooled_family(_residual,_d4)
_reference=s2.fuse(_v2_candidate_rad,_v2_public_values,_v2_family_values,_g96,MASTER['global96_family_share'])
_candidate=s2.fuse(_v2_candidate_rad,_v2_public_values,_family_probability,_g96,MASTER['global96_family_share'])
# Independently replay the inherited function ordering; no extra hybrid ranking.
_reference_old=ma_outer_fuse(_v2_candidate_rad,v2_coat_hybrid(
    _v2_public_values,_v2_family_values,_g96,share=MASTER['global96_family_share']))
if not np.array_equal(_reference,_reference_old):raise RuntimeError('Full-reference fusion parity failed')
rsna_json(_ma_work/'diagnostics/complete_v2_raw_evidence.json',{'residual':_res_evidence,'d4':_d4_evidence})
_frames={}
for _name,_values in [('rank_reference',_reference),('probability_pool',_candidate)]:
    _frame=_ma_anchor_frame.copy();_frame[MA_LABELS]=_values
    _frames[_name]=s2.align(_frame,_ma_ids)
_reference_receipt=ma_publish(_frames['rank_reference'],_ma_ids,_ma_work/'submission_reference.csv',{
    'status':'COMPLETE_REFERENCE_WITH_GLOBAL96','measured_kaggle_score':None,
    'source_not_your_unrecovered_0956_anchor':True})
# Reusable small predictions, not precomputed hidden-test substitutes. They are
# produced only by actual current-run model inference on this exact test cohort.
np.savez_compressed(_ma_work/'complete_v2_prediction_bank.npz',
    study_uids=np.asarray(_ma_ids),labels=np.asarray(MA_LABELS),
    transformer=_v2_candidate_rad,raptor=_v2_public_values,
    reference_family=_v2_family_values,global96=_g96,
    residual_mean=_residual,d4_mean=_d4,reference=_reference,candidate=_candidate)
_delta=_candidate-_reference
pd.DataFrame({'finding':MA_LABELS,'changed_rows':np.count_nonzero(_delta,axis=0),
    'mean_absolute_rank_change':np.abs(_delta).mean(0),
    'max_absolute_rank_change':np.abs(_delta).max(0)}).to_csv(
        _ma_work/'diagnostics/complete_v2_prediction_changes.csv',index=False)
_selected=_frames[SINGLE_CONFIG['recipe']]
rsna_deadline('complete v2 final publication')
_record={'status':'COMPLETE_PRETRAINED_INFERENCE','method':'Complete Offline v2',
    'selected_recipe':SINGLE_CONFIG['recipe'],'config':SINGLE_CONFIG,'source_controls':MASTER,
    'research_target':0.966,'measured_kaggle_score':None,
    'models':{'dino_members':20,'a5_folds':5,'raptor_views':4,
      'residual_coat_checkpoints':3,'d4_parent_plus_adapter':1,'global96_checkpoints':3,
      'rad':'inherited required E10/E13/second-Rad and fitted calibration'},
    'added_model_forward_passes_for_probability_pooling':0,
    'full_gpu_runtime_benchmarked_before_delivery':False,
    'elapsed_seconds':time.time()-MASTER_STARTED_AT,
    'source_reference_csv_sha256':_reference_receipt['submission_sha256'],
    'test_contract':_SINGLE_TEST_CONTRACT,'decoded_pixel_cache':_SINGLE_DECODE_CACHE.stats(),
    'prediction_bank_sha256':s2.sha256(_ma_work/'complete_v2_prediction_bank.npz'),
    'clinical_or_leaderboard_gain_estimated_from_test_predictions':False}
_final_receipt=ma_publish(_selected,_ma_ids,_ma_work/'submission.csv',_record)
_SINGLE_DECODE_CACHE.clear()
for _cache_file in list(_RSNA_CACHE_FILES):Path(_cache_file).unlink(missing_ok=True)
rsna_json(_ma_work/'complete_v2_receipt.json',_final_receipt)
rsna_json(_ma_work/'diagnostics/complete_v2_full_runtime_audit.json',_RSNA_AUDIT)
print('COMPLETE — submit /kaggle/working/submission.csv')
print('Recipe:',SINGLE_CONFIG['recipe'])
print('Actual final CSV SHA256:',_final_receipt['submission_sha256'])
print('Rows:',len(_selected),'Columns:',list(_selected.columns))
print('Same-run reference: /kaggle/working/submission_reference.csv')
print('No 0.966 score claim. Only Kaggle scoring measures the competition result.')


## What completion means

A successful run means all required branches covered the **current** test cohort and the actual final CSV passed integrity checks. It does not mean the target AUC was achieved. `submission_reference.csv` is the same graph with rank-family fusion, not your separately reported 0.956 anchor. The original source anchor without Global96 remains `submission_anchor.csv`.

The previous kit had no trained new-reader bundle, no actual anchor integration, and no Kaggle score. This notebook removes those missing training prerequisites by using existing pretrained artifacts. Its probability-pooling change remains an unscored hypothesis; hardware and full hidden-test runtime must still be checked by a Kaggle run.

Sources: [competition](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection), [official metadata fields](https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels_metadata.md), [public source family](https://www.kaggle.com/code/mattiaangeli/bend-the-knee-to-speedy-raptors-the-original), [fast two-T4 source](https://www.kaggle.com/code/jiweiliu/rsna-knee-fast-2xt4-inference). Source model/data licenses remain applicable. Credits include Mattia Angeli, Jiwei Liu, Dread Development, Pilkwang Kim, Sofia Anjenje, Antoine G., prvsiyan, Marwan Mahmoud, and the authors named in the inherited notebook.
